# v0-05d1 Alakazam UNKNOWN_0 Policy Hit-Check Notebook

This notebook continues from v0-05d and sets `FINAL_SUBMISSION_VARIANT = "unknown0_policy_table"` by default.

Goals:

- Train/evaluate the UNKNOWN_0 submission-safe MLP and distilled policy table.
- Build both rule-only and UNKNOWN_0 policy-table submission archives.
- Select `unknown0_policy_table` as the final `submission.tar.gz` unless safety fallback is needed.
- Instrument the injected runtime policy table so local variant eval reports hit/miss/fallback counts.
- Keep `torch` out of `main.py`; the final agent uses only a distilled policy table and rule fallback.

Safety principles:

- `main.py` never reads `deck.csv` at import time.
- `submission.tar.gz` always contains `main.py`, `deck.csv`, and `cg/`.
- UNKNOWN_0 policy assist only applies when the context, min/max count, signature, and option type match.
- If the policy table misses or cannot find a candidate, the original rule selection path is used.


In [1]:
from pathlib import Path
import csv
import glob
import hashlib
import importlib.util
import json
import math
import os
import pickle
import random
import re
import shutil
import sys
import tarfile
import time
import traceback
import unicodedata
from collections import Counter, defaultdict
from typing import Any

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError('pandas is required for this notebook') from exc

EXPERIMENT_NAME = 'v0_05d25_lgbm_winner_wt4'
RANDOM_SEED = 1515
random.seed(RANDOM_SEED)
if np is not None:
    np.random.seed(RANDOM_SEED)

MC_DISCOUNT_GAMMA = float(os.environ.get('V05_MC_GAMMA', '0.98'))  # γ^(T-t) weight
USE_MC_RETURN_WEIGHTS = os.environ.get('V05_USE_MC_WEIGHTS', '1') != '0'
# v0-05d10: use abstract option signature instead of concrete card-ID set
USE_ABSTRACT_OPTION_SIGNATURE = os.environ.get('V05_ABSTRACT_SIG', '1') != '0'
EARLY_GAME_ONLY_POLICY_FIT = os.environ.get('V05_EARLY_GAME_FILTER', '1') != '0'

# -----------------------------
# Run flags
# -----------------------------
RUN_MODE = os.environ.get('V05_RUN_MODE', 'full')  # smoke | medium | full
TRAIN_MODE = True
RUN_REPLAY_MINING = True
RUN_DECK_SELECTION = True
RUN_ALAKAZAM_BC = True
RUN_ALAKAZAM_VALUE = True
RUN_RULE_GAP_DIAGNOSTICS = True
RUN_LOCAL_EVAL = True
RUN_CONFIRM_EVAL = True
RUN_META_EVAL = True
RUN_UNKNOWN_CONTEXT_DEEP_DIVE = True
RUN_COMPLETE_BASELINE_DIAGNOSTICS = True
RUN_B2_DIAGNOSTICS = True
RUN_UNKNOWN0_DEEP_DIVE = True
RUN_CANDIDATE_DECK_REVIEW = True
RUN_MLP_RERANKER = True
RUN_RULE_VS_MLP_REPORT = True
RUN_UNKNOWN0_SUBMISSION_SAFE_MLP = True
RUN_UNKNOWN0_MLP_CALIBRATION = True
RUN_UNKNOWN0_POLICY_DISTILLATION = True
BUILD_RULE_ONLY_SUBMISSION = True
BUILD_UNKNOWN0_POLICY_SUBMISSION = True
BUILD_SUBMISSION = True

# Neural models are diagnostic by default. The MLP reranker is trained offline;
# submission assist is safety-gated and OFF unless explicitly enabled after reviewing reports.
USE_NEURAL_FOR_SUBMISSION = False
USE_MLP_RERANKER_FOR_SUBMISSION = False
MLP_SUBMISSION_CONTEXTS = []
MLP_ALPHA = float(os.environ.get('V05_MLP_ALPHA', '1.00'))
# v05d16: training data quality improvements
TOP200_EPISODE_FILTER = os.environ.get('V05_TOP200_FILTER', '1') != '0'
USE_PHASE_AWARE_WEIGHT = os.environ.get('V05_PHASE_WEIGHT', '1') != '0'
N_STEP_LOOKAHEAD = int(os.environ.get('V05_N_STEP', '3'))
# v0-05d1: select UNKNOWN_0 policy-table artifact by default.
FINAL_SUBMISSION_VARIANT = os.environ.get('V05_FINAL_SUBMISSION_VARIANT', 'unknown0_mlp_direct').strip()
UNKNOWN0_MLP_MARGIN_THRESHOLD = float(os.environ.get('V05_UNKNOWN0_MLP_MARGIN_THRESHOLD', '0.20'))
UNKNOWN0_POLICY_MIN_SIGNATURE_DECISIONS = int(os.environ.get('V05_UNKNOWN0_POLICY_MIN_SIGNATURE_DECISIONS', '300'))
UNKNOWN0_POLICY_MIN_BUCKET_DECISIONS = int(os.environ.get('V05_UNKNOWN0_POLICY_MIN_BUCKET_DECISIONS', '40'))
UNKNOWN0_POLICY_MIN_GAIN_OVER_RULE = float(os.environ.get('V05_UNKNOWN0_POLICY_MIN_GAIN_OVER_RULE', '0.03'))
UNKNOWN0_POLICY_MIN_HIGH_CONF_ACC = float(os.environ.get('V05_UNKNOWN0_POLICY_MIN_HIGH_CONF_ACC', '0.55'))
UNKNOWN0_POLICY_MIN_TABLE_ACC = float(os.environ.get('V05_UNKNOWN0_POLICY_MIN_TABLE_ACC', '0.75'))
UNKNOWN0_POLICY_MIN_HIGH_CONF_COVERAGE = float(os.environ.get('V05_UNKNOWN0_POLICY_MIN_HIGH_CONF_COVERAGE', '0.08'))
AUTO_SELECT_UNKNOWN0_POLICY_SIGNATURES = os.environ.get('V05_AUTO_SELECT_UNKNOWN0_SIGNATURES', '1') != '0'
SUBMISSION_POLICY_MODE = 'alakazam_rule_base'
AUTO_SELECT_SUBMISSION_DECK = False  # keep False until candidate-deck comparison is manually reviewed
SUBMISSION_DECK_HASH_OVERRIDE = os.environ.get('V05_DECK_HASH_OVERRIDE', '').strip()
CANDIDATE_DECK_HASHES = [
    'e702674c1864',  # Kadoraba-style fallback / observed Top10-heavy candidate
    '2420d2c8ce4b',  # high sample count candidate from v0-05 logs
    '5430416cbea4',  # high win-rate candidate from v0-05 logs
    '5aae58bb890e',  # high win-rate candidate from v0-05 logs
    '2d8ce01487f9',  # high avg-rank candidate from v0-05 logs
]

# -----------------------------
# Paths
# -----------------------------
TOP200_CSV_PATH = Path('/kaggle/input/competitions/top200-20260624-ranking/top200-20260624-ranking.csv')
EPISODE_ROOT_CANDIDATES = [
    Path('/kaggle/input/competitions/pokemon-tcg-ai-battle-episodes-2026-06-24'),
]
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
RUN_PREFIX = 'pokemon-20260625-v0-05d25'
OUTPUT_DIR = WORKING_DIR / RUN_PREFIX
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Training limits
# -----------------------------
if RUN_MODE == 'smoke':
    MAX_EPISODES = 200
    MAX_CONTEXT_TRAIN_ROWS = 50_000
    MAX_GLOBAL_TRAIN_ROWS = 100_000
elif RUN_MODE == 'medium':
    MAX_EPISODES = 1_500
    MAX_CONTEXT_TRAIN_ROWS = 200_000
    MAX_GLOBAL_TRAIN_ROWS = 400_000
else:
    MAX_EPISODES = None
    MAX_CONTEXT_TRAIN_ROWS = 350_000
    MAX_GLOBAL_TRAIN_ROWS = 700_000


# -----------------------------
# MLP reranker limits
# -----------------------------
if RUN_MODE == 'smoke':
    MLP_MAX_ROWS = int(os.environ.get('V05_MLP_MAX_ROWS', '100000'))
    MLP_EPOCHS = int(os.environ.get('V05_MLP_EPOCHS', '2'))
elif RUN_MODE == 'medium':
    MLP_MAX_ROWS = int(os.environ.get('V05_MLP_MAX_ROWS', '300000'))
    MLP_EPOCHS = int(os.environ.get('V05_MLP_EPOCHS', '3'))
else:
    MLP_MAX_ROWS = int(os.environ.get('V05_MLP_MAX_ROWS', '700000'))
    MLP_EPOCHS = int(os.environ.get('V05_MLP_EPOCHS', '4'))
MLP_BATCH_SIZE = int(os.environ.get('V05_MLP_BATCH_SIZE', '4096'))
MLP_VALID_SIZE = float(os.environ.get('V05_MLP_VALID_SIZE', '0.15'))
MLP_LEARNING_RATE = float(os.environ.get('V05_MLP_LR', '0.002'))
MLP_WEIGHT_DECAY = float(os.environ.get('V05_MLP_WEIGHT_DECAY', '0.0001'))

# -----------------------------
# Evaluation limits
# -----------------------------
if RUN_MODE == 'smoke':
    SMOKE_EVAL_GAMES = int(os.environ.get('V05_SMOKE_EVAL_GAMES', '8'))
    CONFIRM_EVAL_GAMES = int(os.environ.get('V05_CONFIRM_EVAL_GAMES', '12'))
    META_EVAL_GAMES_PER_OPPONENT = int(os.environ.get('V05_META_EVAL_GAMES', '4'))
elif RUN_MODE == 'medium':
    SMOKE_EVAL_GAMES = int(os.environ.get('V05_SMOKE_EVAL_GAMES', '12'))
    CONFIRM_EVAL_GAMES = int(os.environ.get('V05_CONFIRM_EVAL_GAMES', '50'))
    META_EVAL_GAMES_PER_OPPONENT = int(os.environ.get('V05_META_EVAL_GAMES', '12'))
else:
    SMOKE_EVAL_GAMES = int(os.environ.get('V05_SMOKE_EVAL_GAMES', '16'))
    CONFIRM_EVAL_GAMES = int(os.environ.get('V05_CONFIRM_EVAL_GAMES', '80'))
    META_EVAL_GAMES_PER_OPPONENT = int(os.environ.get('V05_META_EVAL_GAMES', '20'))

# Keep these conservative by default. Meta eval opponents are mostly deck/engine sanity checks,
# not faithful top-agent replicas unless a real opponent source is supplied later.
LOCAL_EVAL_MAX_STEPS = int(os.environ.get('V05_LOCAL_EVAL_MAX_STEPS', '1000'))

TOP_RANK_CUTOFF = 200
PRIMARY_MATCHUPS = {'Alakazam_vs_Hop_Trevenant', 'Alakazam_mirror'}

# -----------------------------
# Card IDs: Alakazam / Kadoraba-style fallback deck
# -----------------------------
Abra = 741
Kadabra = 742
Alakazam = 743
Dunsparce = 305
Dudunsparce = 66
Fezandipiti_ex = 140
Dawn = 1231
Hilda = 1225
Boss_Orders = 1182
Lana_Aid = 1184
Buddy_Buddy_Poffin = 1086
Poke_Pad = 1152
Rare_Candy = 1079
Enhanced_Hammer = 1081
Sacred_Ash = 1129
Night_Stretcher = 1097
Lucky_Helmet = 1156
Air_Balloon = 1174
Nighttime_Mine = 1266
Telepath_Psychic_Energy = 19
Enriching_Energy = 13
Basic_Psychic_Energy = 5

Hop_Phantump = 878
Hop_Trevenant = 879
Mega_Lucario_ex = 678
Riolu = 677

DEFAULT_ALAKAZAM_DECK = (
    [Abra] * 4 + [Kadabra] * 4 + [Alakazam] * 3 +
    [Dunsparce] * 3 + [Dudunsparce] * 3 + [Fezandipiti_ex] * 1 +
    [Dawn] * 4 + [Hilda] * 4 + [Boss_Orders] * 3 + [Lana_Aid] * 2 +
    [Buddy_Buddy_Poffin] * 4 + [Poke_Pad] * 4 + [Rare_Candy] * 3 +
    [Enhanced_Hammer] * 4 + [Sacred_Ash] * 1 + [Night_Stretcher] * 1 +
    [Lucky_Helmet] * 1 + [Air_Balloon] * 1 + [Nighttime_Mine] * 3 +
    [Telepath_Psychic_Energy] * 4 + [Enriching_Energy] * 1 + [Basic_Psychic_Energy] * 2
)
assert len(DEFAULT_ALAKAZAM_DECK) == 60
assert all(isinstance(x, int) for x in DEFAULT_ALAKAZAM_DECK)

V05_ERROR_ROWS = []

def log_error(stage: str, **kwargs):
    row = {'stage': stage, **kwargs}
    V05_ERROR_ROWS.append(row)
    return row

print('experiment:', EXPERIMENT_NAME)
print('run_mode:', RUN_MODE)
print('output_dir:', OUTPUT_DIR)
print('eval games:', {'smoke': SMOKE_EVAL_GAMES, 'confirm': CONFIRM_EVAL_GAMES, 'meta_per_opponent': META_EVAL_GAMES_PER_OPPONENT})
print('default deck size:', len(DEFAULT_ALAKAZAM_DECK), 'unique:', len(set(DEFAULT_ALAKAZAM_DECK)))


experiment: v0_05d25_lgbm_winner_wt4
run_mode: full
output_dir: /kaggle/working/pokemon-20260625-v0-05d25
eval games: {'smoke': 16, 'confirm': 80, 'meta_per_opponent': 20}
default deck size: 60 unique: 22


In [2]:
# Complete Alakazam rule-agent source.
# v0-05b: complete baseline rule agent with matchup suspicion and safer control heuristics.
ALAKAZAM_RULE_AGENT_SOURCE = r'''import os
import sys
from collections import defaultdict

from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

"""
Alakazam Deck
This deck uses Alakazam's Powerful Hand attack (20 damage per card in hand)
with a draw engine built around Kadabra/Alakazam Psychic Draw, Dudunsparce's
Run Away Draw, and Fezandipiti ex's Flip the Script.
"""

# Hardcoded deck for notebook training/evaluation/submission import safety.
# This avoids FileNotFoundError when confirm/eval imports this module outside
# /kaggle_simulations/agent, where deck.csv does not exist.
my_deck = [741, 741, 741, 741, 742, 742, 742, 742, 743, 743, 743, 305, 305, 305, 66, 66, 66, 140, 1231, 1231, 1231, 1231, 1225, 1225, 1225, 1225, 1182, 1182, 1182, 1184, 1184, 1086, 1086, 1086, 1086, 1152, 1152, 1152, 1152, 1079, 1079, 1079, 1081, 1081, 1081, 1081, 1129, 1097, 1156, 1174, 1266, 1266, 1266, 19, 19, 19, 19, 13, 5, 5]
assert len(my_deck) == 60, "Alakazam rule: hardcoded deck must contain 60 cards"

# Fetch card metadata database and create an ID-to-Card lookup table
all_card = all_card_data()
card_table = {c.cardId: c for c in all_card}

# Decklist
Abra = 741              # x4
Kadabra = 742            # x4
Alakazam = 743           # x3
Dunsparce = 305          # x3
Dudunsparce = 66         # x3
Fezandipiti_ex = 140     # x1
Genesect = 142           # x1
Psyduck = 858            # x1
Shaymin = 343            # x1
Rare_Candy = 1079        # x3
Enhanced_Hammer = 1081   # x4
Buddy_Buddy_Poffin = 1086  # x4
Night_Stretcher = 1097   # x1
Sacred_Ash = 1129        # x1
Poke_Pad = 1152          # x4
Lucky_Helmet = 1156      # x1
Boss_Orders = 1182       # x3
Hilda = 1225             # x4
Dawn = 1231              # x4
Lana_Aid = 1184          # x2
Air_Balloon = 1174       # x1
Nighttime_Mine = 1266    # x3
Battle_Cage = 1264       # not in default deck; retained for compatibility
Basic_Psychic_Energy = 5   # x2
Telepath_Psychic_Energy = 19  # x4
Enriching_Energy = 13    # x1  (ACE SPEC)

# Opponent card IDs to watch for
Duskull = 131
Slowpoke_IDs = (162, 327)
Froakie_IDs = (33, 945)
Wellspring_Mask_Ogerpon_ex = 108
N_Darumaka = 257
Dreepy = 119
Drakloak = 120
Dragapult_ex = 121
Mist_Energy = 11
Rock_Fighting_Energy = 20

# Publicly observed archetype anchors for lightweight opponent-plan suspicion
Hop_Phantump = 878
Hop_Trevenant = 879
Hop_Cramorant = 311
Hop_Snorlax = 304
Postwick = 1255
Riolu = 677
Mega_Lucario_ex = 678
Fighting_Gong = 1142
Basic_Fighting_Energy = 6

# Attack IDs
ATTACK_TELEPORTATION = 1070   # Abra: 10 dmg, cost {P}
ATTACK_SUPER_PSY_BOLT = 1071  # Kadabra: 30 dmg, cost {P}
ATTACK_POWERFUL_HAND = 1072   # Alakazam: 20 per card in hand, cost {P}

# Card ID sets
ABRA_LINE = {Abra, Kadabra, Alakazam}
DUNSPARCE_LINE = {Dunsparce, Dudunsparce}
PSYCHIC_ENERGY_IDS = {Basic_Psychic_Energy, Telepath_Psychic_Energy}

pre_turn = 0
ability_used_dudunsparce = False
ability_used_fezandipiti = False


def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None


def prize_count(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == 12:  # Legacy Energy
            count -= 1
    for card in pokemon.tools:
        if card.id == 1172 and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def count_special_defense_energies(pokemon: Pokemon) -> int:
    cnt = 0
    for ec in pokemon.energyCards:
        if ec.id == Mist_Energy or ec.id == Rock_Fighting_Energy:
            cnt += 1
    return cnt


def safe_unique_action(indices: list[int], option_count: int, min_count: int, max_count: int) -> list[int]:
    """Clamp and deduplicate action indices before returning to the game engine."""
    out = []
    seen = set()
    for x in indices:
        try:
            i = int(x)
        except Exception:
            continue
        if 0 <= i < option_count and i not in seen:
            out.append(i)
            seen.add(i)
        if len(out) >= max_count:
            break
    if len(out) < min_count:
        for i in range(option_count):
            if i not in seen:
                out.append(i)
                seen.add(i)
            if len(out) >= min_count:
                break
    return out[:max_count]


def context_threshold(context: SelectContext, safe_draws: int, can_win_this_turn: bool,
                      opponent_archetype: str = 'unknown', has_alakazam: bool = False) -> int:
    """Decision threshold learned from Top Alakazam reports: choose fewer than maxCount.

    Conservative contexts are intentionally harder to select from. This is the rule-level
    translation of the BC/full-rate diagnostics; the model itself is not used at inference.
    """
    if can_win_this_turn:
        return 1
    if context == SelectContext.TO_HAND:
        if safe_draws <= 1:
            return 460
        if safe_draws <= 3:
            return 340
        return 150 if not has_alakazam else 210
    if context == SelectContext.TO_ACTIVE or context == SelectContext.SWITCH:
        return 70
    if context == SelectContext.TO_BENCH:
        return 45 if not has_alakazam else 75
    if context == SelectContext.TO_DECK:
        return 70
    if context == SelectContext.MAIN:
        return 1800
    if context == SelectContext.ATTACH_FROM:
        return 60
    # Numeric/unknown contexts are safer when conservative.
    return 5


def detect_opponent_archetype(op_all_pokemon, stadium_id: int = 0) -> tuple[str, int]:
    """Lightweight public-information archetype suspicion.

    Uses only observed public board cards/stadium. It is intentionally conservative;
    low confidence falls back to generic_control.
    """
    ids = {p.id for p in op_all_pokemon if p is not None}
    scores = defaultdict(int)
    if ids & {Hop_Phantump, Hop_Trevenant, Hop_Cramorant, Hop_Snorlax} or stadium_id == Postwick:
        scores['hop_control'] += 2 + len(ids & {Hop_Phantump, Hop_Trevenant, Hop_Cramorant, Hop_Snorlax})
    if ids & {Abra, Kadabra, Alakazam, Dunsparce, Dudunsparce} or stadium_id == Nighttime_Mine:
        scores['alakazam_mirror'] += 2 + len(ids & {Abra, Kadabra, Alakazam, Dunsparce, Dudunsparce})
    if ids & {Riolu, Mega_Lucario_ex}:
        scores['lucario'] += 2 + len(ids & {Riolu, Mega_Lucario_ex})
    if not scores:
        return 'generic_control', 0
    best, score = max(scores.items(), key=lambda kv: kv[1])
    if score < 2:
        return 'generic_control', score
    return best, int(score)


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return my_deck

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
    my_prize_count = len(my_state.prize)

    global pre_turn, ability_used_dudunsparce, ability_used_fezandipiti
    if pre_turn != state.turn:
        pre_turn = state.turn
        ability_used_dudunsparce = False
        ability_used_fezandipiti = False

    # ---- Count cards on field / hand / discard ----
    field_counts = defaultdict(int)
    hand_counts = defaultdict(int)
    discard_counts = defaultdict(int)

    my_field = []  # (field_index, pokemon) where 0=active, 1..=bench
    for card in my_state.active:
        if card is not None:
            field_counts[card.id] += 1
            my_field.append((0, card))
    for idx, card in enumerate(my_state.bench):
        if card is not None:
            field_counts[card.id] += 1
            my_field.append((idx + 1, card))

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    abra_line_on_field = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]
    dunsparce_line_on_field = field_counts[Dunsparce] + field_counts[Dudunsparce]

    # ---- Opponent field analysis ----
    op_all_pokemon = []
    for card in op_state.active:
        if card is not None:
            op_all_pokemon.append(card)
    for card in op_state.bench:
        if card is not None:
            op_all_pokemon.append(card)

    op_has_duskull = any(p.id == Duskull for p in op_all_pokemon)
    op_has_water_threat = any(
        p.id in Slowpoke_IDs or p.id in Froakie_IDs
        or p.id == Wellspring_Mask_Ogerpon_ex or p.id == N_Darumaka
        for p in op_all_pokemon
    )
    op_has_dragapult_line = any(
        p.id in (Dreepy, Drakloak, Dragapult_ex) for p in op_all_pokemon
    )

    # Detect if opponent has used ACE SPEC
    op_used_ace_spec = False
    for log in obs.logs:
        if hasattr(log, 'cardId') and log.cardId is not None:
            cd = card_table.get(log.cardId)
            if cd and cd.aceSpec and hasattr(log, 'playerIndex') and log.playerIndex == (1 - my_index):
                op_used_ace_spec = True

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    opponent_archetype, opponent_confidence = detect_opponent_archetype(op_all_pokemon, stadium_id)

    bench_count = len(my_state.bench)
    bench_max = my_state.benchMax
    bench_free = bench_max - bench_count

    # ---- Active pokemon info ----
    active_pokemon = my_state.active[0] if my_state.active else None
    active_id = active_pokemon.id if active_pokemon else -1
    active_has_psychic = False
    if active_pokemon:
        for ec in active_pokemon.energyCards:
            if ec.id in PSYCHIC_ENERGY_IDS:
                active_has_psychic = True
                break

    # ---- Opponent active info ----
    op_active = op_state.active[0] if op_state.active else None
    op_active_hp = op_active.hp if op_active else 9999

    # ---- Estimate Powerful Hand damage range ----
    hand_size = len(my_state.hand) if my_state.hand else my_state.handCount

    def estimate_hand_increase():
        """Returns (min_increase, max_increase) of hand size this turn from draw effects."""
        min_inc = 0
        max_inc = 0
        for _, p in my_field:
            if p.id == Abra and hand_counts[Kadabra] > 0:
                max_inc += 1  # evolve Kadabra: hand -1, draw +2 = net +1
            elif p.id == Abra and hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:
                max_inc += 1  # Rare Candy + Alakazam: hand -2, draw +3 = net +1
            elif p.id == Kadabra and hand_counts[Alakazam] > 0:
                max_inc += 2  # evolve Alakazam: hand -1, draw +3 = net +2
            elif p.id == Dunsparce and hand_counts[Dudunsparce] > 0:
                max_inc += 1  # evolve: hand -1, ability draw +2 = net +1
            elif p.id == Dudunsparce:
                if not ability_used_dudunsparce:
                    max_inc += 3  # Run Away Draw
            elif p.id == Fezandipiti_ex:
                if not ability_used_fezandipiti:
                    max_inc += 3  # Flip the Script
        if hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:
            max_inc += 2  # play -1, ability +3 = net +2

        # Supporter (only 1 can be used)
        supporter_options = []
        if not state.supporterPlayed:
            if hand_counts[Hilda] > 0:
                supporter_options.append(1)   # play -1, search +2 = net +1
            if hand_counts[Dawn] > 0:
                supporter_options.append(2)   # play -1, search +3 = net +2
            if hand_counts[Boss_Orders] > 0:
                supporter_options.append(-1)  # play -1 = net -1
        if supporter_options:
            max_inc += max(supporter_options)

        # Enriching Energy attach: hand -1, draw +4 = net +3
        if hand_counts[Enriching_Energy] > 0 and not state.energyAttached:
            if active_id == Alakazam and active_has_psychic:
                max_inc += 3
        return min_inc, max_inc

    min_hand_inc, max_hand_inc = estimate_hand_increase()
    max_hand_size = hand_size + max_hand_inc
    min_hand_size = hand_size + min_hand_inc
    max_damage = max_hand_size * 20
    min_damage = min_hand_size * 20

    # ---- Target selection for attack ----
    target_idx = -1       # 0 = active, 1.. = bench
    target_pokemon = None
    target_use_boss = False
    target_can_kill = False
    target_prize_gain = 0
    target_hammer_needed = 0
    use_kadabra_finish = False

    if state.turn >= 2 and op_active is not None:
        # Check Kadabra finisher: opponent active HP <= 30
        if op_active_hp <= 30 and (field_counts[Kadabra] >= 1 or active_id == Kadabra):
            target_idx = 0
            target_pokemon = op_active
            target_use_boss = False
            target_can_kill = True
            target_prize_gain = prize_count(op_active)
            use_kadabra_finish = True
        else:
            # Evaluate all opponent pokemon
            all_op = [(0, op_active)]
            for bi, bp in enumerate(op_state.bench):
                if bp is not None:
                    all_op.append((bi + 1, bp))

            candidates = []
            for oi, pkmn in all_op:
                pz = prize_count(pkmn)
                sp_e = count_special_defense_energies(pkmn)
                eff_max_dmg = max_damage
                hm_need = 0
                if sp_e > 0:
                    if hand_counts[Enhanced_Hammer] >= sp_e:
                        hm_need = sp_e
                        eff_max_dmg = (max_hand_size - hm_need) * 20
                    else:
                        eff_max_dmg = 0
                ck = pkmn.hp <= eff_max_dmg and eff_max_dmg > 0
                candidates.append((oi, pkmn, pz, ck, hm_need))

            # Priority 1: kill wins the game
            win_cands = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck and my_prize_count <= pz]
            if win_cands:
                # Among winners, prefer active (no boss needed), then highest HP
                best = min(win_cands, key=lambda x: (0 if x[0] == 0 else 1, -x[1].hp))
                target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best
                target_use_boss = target_idx != 0
            else:
                # Priority 2: killable target with most prizes
                killable = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck]
                if killable:
                    best = max(killable, key=lambda x: (x[2], x[1].hp))
                    target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best
                    target_use_boss = target_idx != 0
                else:
                    # Priority 3: just hit active
                    target_idx = 0
                    target_pokemon = op_active
                    target_use_boss = False
                    target_can_kill = False
                    target_prize_gain = 0

    # Should we use Dudunsparce's ability?
    need_dudunsparce_draw = False
    if target_pokemon is not None and target_can_kill:
        needed = target_pokemon.hp
        current_dmg = (hand_size - target_hammer_needed) * 20
        if current_dmg < needed:
            need_dudunsparce_draw = True

    # Do we need to attach energy to the active to retreat?
    need_retreat_energy = False
    if active_pokemon is not None and state.turn >= 2:
        active_is_attacker = (active_id == Alakazam and active_has_psychic) or (use_kadabra_finish and active_id == Kadabra)
        if not active_is_attacker:
            # Check if there's a better attacker on bench
            has_bench_attacker = False
            if use_kadabra_finish and field_counts[Kadabra] >= 1 and active_id != Kadabra:
                has_bench_attacker = True
            elif field_counts[Alakazam] >= 1 and active_id != Alakazam:
                has_bench_attacker = True
            elif field_counts[Kadabra] >= 1 and active_id != Kadabra:
                has_bench_attacker = True
            if has_bench_attacker:
                retreat_cost = card_table[active_pokemon.id].retreatCost
                active_energy_count = len(active_pokemon.energies)
                if active_energy_count < retreat_cost:
                    need_retreat_energy = True

    # Do we need Fezandipiti ex's Flip the Script to kill the target?
    fez_hand_contribution = 0
    if field_counts[Fezandipiti_ex] >= 1 and not ability_used_fezandipiti:
        fez_hand_contribution = 3
    elif hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:
        fez_hand_contribution = 2  # play -1, ability +3 = net +2
    need_fezandipiti_draw = False
    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0:
        max_damage_without_fez = (max_hand_size - fez_hand_contribution - target_hammer_needed) * 20
        if max_damage_without_fez < target_pokemon.hp:
            need_fezandipiti_draw = True

    # Also allow Fezandipiti if drawing could find key enablers (Boss, Rare Candy, Alakazam, Energy)
    need_fezandipiti_for_setup = False
    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0 and not need_fezandipiti_draw:
        # Missing Boss's Orders for bench target
        missing_boss = (target_use_boss and hand_counts[Boss_Orders] == 0
                        and not state.supporterPlayed)
        # Check if we have a ready attacker (Alakazam with psychic energy)
        has_ready_attacker = (active_id == Alakazam and active_has_psychic)
        if not has_ready_attacker:
            for _, p in my_field:
                if p.id == Alakazam and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards):
                    has_ready_attacker = True
                    break
        missing_attacker = False
        missing_energy = False
        if not has_ready_attacker:
            # Can we set up Alakazam this turn?
            can_evolve_to_alakazam = (field_counts[Kadabra] >= 1 and hand_counts[Alakazam] >= 1)
            can_rare_candy_alakazam = (field_counts[Abra] >= 1 and hand_counts[Rare_Candy] >= 1
                                       and hand_counts[Alakazam] >= 1)
            if not can_evolve_to_alakazam and not can_rare_candy_alakazam:
                # Missing evolution pieces
                if field_counts[Kadabra] >= 1 and hand_counts[Alakazam] == 0:
                    missing_attacker = True
                elif field_counts[Abra] >= 1 and (hand_counts[Rare_Candy] == 0 or hand_counts[Alakazam] == 0):
                    missing_attacker = True
            # Check if energy is available for the attacker
            energy_in_hand = (hand_counts[Basic_Psychic_Energy] + hand_counts[Telepath_Psychic_Energy]
                              + hand_counts[Enriching_Energy])
            if not state.energyAttached and energy_in_hand == 0:
                has_energized = any(
                    p.id in ABRA_LINE and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards)
                    for _, p in my_field
                )
                if not has_energized:
                    missing_energy = True
        if missing_boss or missing_attacker or missing_energy:
            need_fezandipiti_for_setup = True

    # Deck safety: don't let deck count drop to <= prize count unless winning this turn.
    can_win_this_turn = target_can_kill and my_prize_count <= target_prize_gain
    deck_count = my_state.deckCount
    # safe_draws: max cards we can draw/search from deck while keeping deck > prize count.
    # Keep one card for the next turn's mandatory draw unless we can win immediately.
    safe_draws = deck_count - my_prize_count - 1 if not can_win_this_turn else 999
    deckout_risk_strict = (safe_draws <= 0 and not can_win_this_turn)
    deckout_risk = (safe_draws <= 3 and not can_win_this_turn)
    setup_incomplete = field_counts[Alakazam] == 0
    need_abra_setup = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam] < 2
    need_dunsparce_setup = field_counts[Dunsparce] + field_counts[Dudunsparce] < 1

    # ---- Score each option ----
    scores = []
    for o in select.option:
        score = 0

        if o.type == OptionType.NUMBER:
            num = int(o.number)
            if deckout_risk and not can_win_this_turn:
                # Prefer smaller optional draw/search numbers when the deck is thin.
                limit = max(0, safe_draws)
                score = 1000 - abs(num - limit) * 150 - max(0, num - limit) * 600
            else:
                score = num

        elif o.type == OptionType.YES:
            score = -1 if deckout_risk_strict else 1

        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card is None:
                scores.append(score)
                continue
            energy_count = len(card.energies) if isinstance(card, Pokemon) else 0

            if context == SelectContext.SWITCH or context == SelectContext.TO_ACTIVE:
                if o.playerIndex == my_index:
                    if card.id == Alakazam:
                        score += 100 + energy_count * 10
                    elif card.id == Kadabra:
                        score += 90 if (op_active_hp <= 30) else 30
                    elif card.id == Abra:
                        score += 10
                    elif card.id in DUNSPARCE_LINE:
                        score += 5
                    else:
                        score += 1
                else:
                    if target_use_boss and target_pokemon is not None:
                        if o.index == target_idx - 1:
                            score += 100

            elif context == SelectContext.SETUP_ACTIVE_POKEMON:
                if card.id == Abra:
                    score = 10
                elif card.id == Dunsparce:
                    score = 5
                elif card.id == Psyduck:
                    score = 2
                elif card.id == Shaymin:
                    score = 1

            elif context == SelectContext.SETUP_BENCH_POKEMON:
                if card.id == Abra:
                    cur = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]
                    score = 200 if cur == 0 else 100 + (3 - cur) * 10
                elif card.id == Dunsparce:
                    score = 150 if dunsparce_line_on_field == 0 else 50

            elif context == SelectContext.TO_HAND:
                score = 200 - hand_counts.get(card.id, 0) * 50
                if card.id == Dudunsparce:
                    score += 80 if (field_counts[Dunsparce] >= 1 and field_counts[Dudunsparce] == 0) else -50
                elif card.id == Kadabra:
                    score += 70 if field_counts[Abra] >= 1 else -20
                elif card.id == Alakazam:
                    score += 60 if (field_counts[Kadabra] >= 1 or field_counts[Abra] >= 1) else -20
                elif card.id == Abra:
                    score += 50 if abra_line_on_field < 3 else -50
                elif card.id == Dunsparce:
                    score += 40 if dunsparce_line_on_field < 2 else -50
                elif card.id in PSYCHIC_ENERGY_IDS:
                    score += 30 if not state.energyAttached else -10
                elif card.id == Enriching_Energy:
                    score += 20
                elif card.id == Rare_Candy:
                    score += 90 if (field_counts[Abra] >= 1 and field_counts[Alakazam] == 0) else -10
                elif card.id == Buddy_Buddy_Poffin:
                    score += 80 if (need_abra_setup or need_dunsparce_setup) else -30
                elif card.id == Hilda:
                    score += 70 if setup_incomplete else -20
                elif card.id == Dawn:
                    score += 80 if setup_incomplete else -20
                elif card.id == Boss_Orders:
                    if target_use_boss and target_can_kill:
                        score += 160
                    elif opponent_archetype in ('alakazam_mirror', 'hop_control') and state.turn >= 4:
                        score += 50
                    else:
                        score -= 40
                elif card.id == Enhanced_Hammer:
                    score += 130 if target_hammer_needed > 0 else (70 if opponent_archetype in ('hop_control', 'lucario') else 20)
                elif card.id == Nighttime_Mine:
                    if stadium_id != Nighttime_Mine:
                        score += 130 if opponent_archetype in ('hop_control', 'lucario', 'alakazam_mirror') else 60
                    else:
                        score -= 80

                if deckout_risk and not can_win_this_turn:
                    key_cards = {Alakazam, Kadabra, Rare_Candy, Boss_Orders, Enhanced_Hammer, Nighttime_Mine, Basic_Psychic_Energy, Telepath_Psychic_Energy}
                    if card.id not in key_cards:
                        score -= 180
                    if deckout_risk_strict and card.id not in key_cards:
                        score = -1

            elif context == SelectContext.ATTACH_FROM:
                if isinstance(card, Pokemon):
                    if need_retreat_energy and o.area == AreaType.ACTIVE:
                        score = 150  # Must attach to active to retreat
                    elif len(card.energyCards) >= 1:
                        score = -1  # Don't attach 2+ energy to the same pokemon
                    elif card.id in ABRA_LINE:
                        score = 100
                        if card.id == Alakazam:
                            score += 20
                        elif card.id == Kadabra:
                            score += 10
                        if o.area == AreaType.ACTIVE:
                            score += 5
                    elif card.id in DUNSPARCE_LINE:
                        score = 50
                    else:
                        score = 10

            elif context == SelectContext.TO_BENCH:
                if card.id == Abra:
                    score = 100
                elif card.id == Dunsparce:
                    score = 80
                elif card.id == Psyduck:
                    if op_has_duskull:
                        score = 60
                    else:
                        score = -1
                elif card.id == Shaymin:
                    if op_has_water_threat:
                        score = 40
                    else:
                        score = -1

            elif context == SelectContext.TO_DECK:
                if card.id in ABRA_LINE:
                    score = 100
                elif card.id in DUNSPARCE_LINE:
                    score = 50
                else:
                    score = 10

        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            data = card_table[card.id]

            if data.cardType == CardType.POKEMON:
                score = 20000
                is_early = state.turn <= 2

                if card.id == Abra:
                    if is_early:
                        score += 500
                    elif abra_line_on_field < 3:
                        score += 200
                    elif bench_free <= 1:
                        score = -1
                    else:
                        score += 50

                elif card.id == Dunsparce:
                    if dunsparce_line_on_field < 1:
                        score += 400 if is_early else 100
                    elif dunsparce_line_on_field < 2:
                        score += 50
                    else:
                        score = -1

                elif card.id == Fezandipiti_ex:
                    if need_fezandipiti_draw or need_fezandipiti_for_setup:
                        score += 80 if not is_early else 30
                    else:
                        score = -1  # Don't play unless Flip the Script is needed to kill

                elif card.id == Genesect:
                    if not op_used_ace_spec and (hand_counts[Lucky_Helmet] > 0 or hand_counts[Poke_Pad] > 0):
                        score += 100
                    else:
                        score = -1

                elif card.id == Psyduck:
                    if op_has_duskull:
                        score += 300
                    else:
                        score = -1

                elif card.id == Shaymin:
                    if op_has_water_threat:
                        score += 300
                    else:
                        score = -1

                # Keep at least 1 bench slot free
                if bench_free <= 1 and score > 0:
                    score -= 5000

            else:
                score = 10000

                if card.id == Buddy_Buddy_Poffin:
                    if safe_draws < 2:
                        score = -1  # Deck too thin (searches deck)
                    elif state.turn <= 2:
                        if abra_line_on_field < 3 or dunsparce_line_on_field < 1:
                            score = 18000
                        else:
                            score = 8000
                    else:
                        if abra_line_on_field < 3 or dunsparce_line_on_field < 2:
                            score = 15000
                        elif target_can_kill:
                            score = 8000
                        else:
                            score = -1

                elif card.id == Poke_Pad:
                    if safe_draws < 1:
                        score = -1  # Deck too thin (searches deck)
                    elif state.turn <= 2:
                        score = 17000
                    else:
                        score = 14000 if abra_line_on_field < 3 else 12000

                elif card.id == Rare_Candy:
                    if field_counts[Abra] >= 1 and hand_counts[Alakazam] >= 1 and safe_draws >= 3:
                        score = 16000
                    else:
                        score = -1

                elif card.id == Night_Stretcher:
                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]
                    if dis_abra >= 1:
                        score = 13000
                    elif discard_counts[Basic_Psychic_Energy] + discard_counts[Telepath_Psychic_Energy] >= 1:
                        score = 11000
                    else:
                        score = -1

                elif card.id == Sacred_Ash:
                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]
                    if dis_abra >= 2:
                        score = 13500
                    elif dis_abra >= 1:
                        score = 11000
                    else:
                        score = -1

                elif card.id == Enhanced_Hammer:
                    if target_hammer_needed > 0:
                        score = 6500
                    else:
                        # Check if any opponent pokemon has special defense energy
                        any_special = any(count_special_defense_energies(p) > 0 for p in op_all_pokemon)
                        if any_special:
                            score = 6200 if opponent_archetype in ('hop_control', 'lucario') else 5000
                        elif opponent_archetype in ('hop_control', 'lucario') and state.turn >= 3:
                            score = 2200
                        else:
                            score = -1

                elif card.id == Lucky_Helmet:
                    score = 7000  # Will be handled via ATTACH

                elif card.id == Boss_Orders:
                    if target_use_boss and target_can_kill:
                        score = 9000 if can_win_this_turn else 6200
                    elif opponent_archetype == 'alakazam_mirror' and state.turn >= 4:
                        score = 2400
                    elif opponent_archetype == 'hop_control' and state.turn >= 4:
                        score = 2200
                    else:
                        score = -1

                elif card.id == Hilda:
                    if safe_draws >= 2:
                        score = 3000
                    else:
                        score = -1

                elif card.id == Dawn:
                    if safe_draws >= 3:
                        score = 3100
                    else:
                        score = -1

                elif card.id == Lana_Aid:
                    dis_line = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam] + discard_counts[Dunsparce] + discard_counts[Dudunsparce]
                    dis_energy = discard_counts[Basic_Psychic_Energy] + discard_counts[Telepath_Psychic_Energy]
                    if dis_line + dis_energy >= 2:
                        score = 7600
                    elif state.turn >= 5 and dis_line + dis_energy >= 1:
                        score = 4200
                    else:
                        score = -1

                elif card.id == Nighttime_Mine:
                    # Prefer Nighttime Mine in the control mirrors / Tera-energy tax matchups.
                    if stadium_id != Nighttime_Mine:
                        if opponent_archetype in ('hop_control', 'lucario'):
                            score = 15500
                        elif opponent_archetype == 'alakazam_mirror' and state.turn >= 3:
                            score = 9800
                        elif op_has_dragapult_line or target_can_kill:
                            score = 15000
                        elif stadium_id != 0:
                            score = 7600
                        elif state.turn >= 3:
                            score = 5300
                        else:
                            score = -1
                    else:
                        score = -1

                elif card.id == Battle_Cage:
                    if op_has_dragapult_line:
                        score = 9000
                    elif stadium_id != 0:
                        score = 4000
                    else:
                        score = -1

        elif o.type == OptionType.ATTACH:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)

            if card.id == Air_Balloon:
                if o.inPlayArea == AreaType.ACTIVE and active_id not in (Alakazam, Kadabra):
                    score = 8800
                elif pokemon.id in ABRA_LINE and len(getattr(pokemon, 'tools', [])) == 0:
                    score = 5800
                else:
                    score = -1

            elif card.id == Lucky_Helmet:
                score = 7000
                if pokemon.id == Genesect and not op_used_ace_spec:
                    score += 300
                elif o.inPlayArea == AreaType.ACTIVE:
                    score += 200
                else:
                    score += 50

            elif card.id in PSYCHIC_ENERGY_IDS:
                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:
                    score = 9500  # Must attach to active to retreat
                elif len(pokemon.energyCards) >= 1:
                    score = -1  # Don't attach 2+ energy to the same pokemon
                elif pokemon.id in ABRA_LINE:
                    score = 8000
                    if pokemon.id == Alakazam:
                        score += 30
                    elif pokemon.id == Kadabra:
                        score += 20
                    elif pokemon.id == Abra:
                        score += 10
                    if o.inPlayArea == AreaType.ACTIVE:
                        score += 5
                else:
                    score = -1
                # Telepath Psychic Energy searches 2 from deck
                if card.id == Telepath_Psychic_Energy and safe_draws < 2 and score > 0:
                    score = -1

            elif card.id == Enriching_Energy:
                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:
                    score = 9500  # Must attach to active to retreat
                elif len(pokemon.energyCards) >= 1:
                    score = -1  # Don't attach 2+ energy to the same pokemon
                elif pokemon.id in DUNSPARCE_LINE:
                    score = 8500
                    if pokemon.id == Dudunsparce:
                        score += 10
                else:
                    score = -1
                # Enriching Energy draws 4 from deck
                if card.id == Enriching_Energy and safe_draws < 4 and score > 0:
                    score = -1

        elif o.type == OptionType.EVOLVE:
            if context == SelectContext.TO_BENCH:
                card = get_card(obs, o.area, o.index, my_index)
                pokemon = None
            else:
                card = get_card(obs, AreaType.HAND, o.index, my_index)
                pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 9000

            if context == SelectContext.TO_BENCH:
                if card is not None:
                    if card.id == Abra:
                        score += 100    # Bench Abra for evolution chain
                    elif card.id == Dunsparce:
                        score += 80     # Draw support on bench
                    elif card.id == Psyduck:
                        score += 60 if op_has_duskull else -9100
                    elif card.id == Shaymin:
                        score += 40 if op_has_water_threat else -9100
                    elif card.id == Alakazam:
                        score -= 50     # Keep Alakazam in hand for Rare Candy
                    elif card.id == Riolu:
                        score += 20     # Bench Riolu for evolution chain

            elif card.id == Alakazam:
                if safe_draws < 3:
                    score = -1  # Deck too thin for Psychic Draw (3 cards)
                elif o.inPlayArea == AreaType.ACTIVE:
                    score += 200  # Active Alakazam = highest
                else:
                    score += 50  # Bench Alakazam
                score += len(pokemon.energies) * 10

            elif card.id == Kadabra:
                if safe_draws < 2:
                    score = -1  # Deck too thin for Psychic Draw (2 cards)
                else:
                    score += 100
                    if len(pokemon.energies) == 0:
                        score += 50  # Evolve non-energy Abra first
                    else:
                        score -= 20
                        if hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:
                            score -= 100  # Save energy Abra for Rare Candy -> Alakazam

            elif card.id == Dudunsparce:
                if safe_draws < 2:
                    score = -1  # Deck too thin for draw on evolve
                else:
                    score += 80

        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card is None:
                scores.append(score)
                continue

            if card.id == Dudunsparce:
                if need_dudunsparce_draw:
                    if safe_draws >= 3:
                        score = 30000
                    else:
                        score = -1  # Deck too thin
                else:
                    score = -1
            elif card.id == Fezandipiti_ex:
                if (need_fezandipiti_draw or need_fezandipiti_for_setup) and safe_draws >= 3:
                    score = 29000
                else:
                    score = -1  # Don't use unless needed to kill target
            elif card.id == Battle_Cage:
                score = 1
            else:
                score = 28000
                if deckout_risk_strict and not can_win_this_turn:
                    score = -1
                elif deckout_risk and not can_win_this_turn:
                    score -= 12000

        elif o.type == OptionType.RETREAT:
            if active_id == Alakazam and active_has_psychic:
                score = -1
            elif use_kadabra_finish and active_id != Kadabra and field_counts[Kadabra] >= 1:
                score = 2500  # Retreat to bring Kadabra forward for finish
            elif active_id in (Abra, Dunsparce, Dudunsparce, Psyduck, Shaymin, Genesect):
                if field_counts[Alakazam] >= 1 or field_counts[Kadabra] >= 1:
                    score = 2000
                else:
                    score = -1
            else:
                score = -1

        elif o.type == OptionType.ATTACK:
            score = 1000
            if o.attackId == ATTACK_POWERFUL_HAND:
                score += 500
                if target_can_kill:
                    score += 5000 if can_win_this_turn else 2600
                elif active_id == Alakazam and active_has_psychic:
                    score += 300
            elif o.attackId == ATTACK_SUPER_PSY_BOLT:
                if op_active_hp <= 30:
                    score += 600  # Kadabra finisher
                else:
                    score += 100
            elif o.attackId == ATTACK_TELEPORTATION:
                score += 50

        scores.append(score)

    # Select in descending order of score.
    # Do not blindly return maxCount: Top control agents often choose fewer than allowed.
    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    min_count = max(0, int(select.minCount))
    max_count = min(len(select.option), max(0, int(select.maxCount)))
    if max_count <= 0:
        return []
    threshold = context_threshold(context, safe_draws, can_win_this_turn, opponent_archetype, field_counts[Alakazam] > 0)
    eligible = [i for i in desc_indices if scores[i] >= threshold]
    # For optional contexts, allow no-op when all choices are weak.
    selected = eligible[:max_count]
    if len(selected) < min_count:
        selected = desc_indices[:min_count]
    selected = safe_unique_action(selected, len(select.option), min_count, max_count)

    if context == SelectContext.MAIN and selected:
        o = select.option[selected[0]]
        if o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card is not None:
                if card.id == Dudunsparce:
                    ability_used_dudunsparce = True
                elif card.id == Fezandipiti_ex:
                    ability_used_fezandipiti = True

    return selected
'''


def make_alakazam_agent_source(deck: list[int] | None = None) -> str:
    """Return a complete main.py source with the selected 60-card deck embedded."""
    src = ALAKAZAM_RULE_AGENT_SOURCE
    chosen = DEFAULT_ALAKAZAM_DECK if deck is None else list(deck)
    if len(chosen) != 60 or not all(isinstance(x, int) for x in chosen):
        raise ValueError(f'Alakazam deck must be 60 ints. Got len={len(chosen)}')
    src = re.sub(r"my_deck = \[[^\]]*\]", f"my_deck = {chosen!r}", src, count=1)
    compile(src, 'alakazam_rule_agent.py', 'exec')
    return src


def assert_no_import_time_deck_io(source: str, label: str):
    dangerous_patterns = [
        'DECK_PATH =',
        'file_path = "deck.csv"',
        "file_path = 'deck.csv'",
        'open(DECK_PATH',
        'open(file_path',
        'with open(DECK_PATH',
        'with open(file_path',
        '/kaggle_simulations/agent/deck.csv',
    ]
    found = [p for p in dangerous_patterns if p in source]
    if found:
        raise AssertionError(f'{label} contains import-time deck I/O patterns: {found}')

_assert_src = make_alakazam_agent_source(DEFAULT_ALAKAZAM_DECK)
assert_no_import_time_deck_io(_assert_src, 'ALAKAZAM_RULE_AGENT_SOURCE')
print('alakazam source chars:', len(_assert_src))
print('no import-time deck I/O: OK')


alakazam source chars: 43379
no import-time deck I/O: OK


In [3]:
# -----------------------------
# Generic utilities
# -----------------------------

def prefixed_artifact_path(path: str | Path) -> Path:
    path = Path(path)
    if path.parent == OUTPUT_DIR and not path.name.startswith(f'{RUN_PREFIX}-'):
        return path.with_name(f'{RUN_PREFIX}-{path.name}')
    return path


def write_text(path: str | Path, text: str) -> str:
    path = prefixed_artifact_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    return str(path)


def write_json(path: str | Path, obj: Any) -> str:
    return write_text(path, json.dumps(obj, ensure_ascii=False, indent=2))


def safe_save_table(df: pd.DataFrame, base_path: str | Path) -> str:
    base_path = prefixed_artifact_path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)
    if df is None:
        df = pd.DataFrame()
    parquet_path = base_path.with_suffix('.parquet')
    csv_path = base_path.with_suffix('.csv')
    try:
        df.to_parquet(parquet_path, index=False)
        return str(parquet_path)
    except Exception:
        df.to_csv(csv_path, index=False)
        return str(csv_path)


def norm_name(s: Any) -> str:
    if s is None:
        return ''
    return unicodedata.normalize('NFKC', str(s)).strip()


def safe_get(obj: Any, path: list[Any], default=None):
    cur = obj
    for p in path:
        try:
            if isinstance(cur, dict):
                cur = cur[p]
            elif isinstance(cur, list) and isinstance(p, int):
                cur = cur[p]
            else:
                return default
        except Exception:
            return default
    return cur


def as_list(x: Any) -> list:
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def to_int_or_none(x: Any):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return None
        return int(x)
    except Exception:
        return None


def enum_name(x: Any, fallback_prefix: str = 'UNKNOWN') -> str:
    if x is None:
        return f'{fallback_prefix}_NONE'
    if hasattr(x, 'name'):
        return str(x.name)
    if isinstance(x, dict):
        for key in ['name', 'value', 'id', 'type']:
            if key in x:
                return enum_name(x[key], fallback_prefix)
    s = str(x)
    # Common serialized forms: "SelectContext.MAIN", "OptionType.PLAY".
    if '.' in s:
        s = s.split('.')[-1]
    return s.upper()


def stable_deck_hash(deck: list[int]) -> str:
    counts = Counter(int(x) for x in deck)
    payload = '|'.join(f'{k}:{counts[k]}' for k in sorted(counts))
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:12]


def valid_deck_list(x: Any) -> bool:
    if not isinstance(x, list) or len(x) != 60:
        return False
    try:
        return all(isinstance(int(v), int) for v in x)
    except Exception:
        return False


def normalize_deck(x: Any) -> list[int] | None:
    if not valid_deck_list(x):
        return None
    return [int(v) for v in x]


def infer_archetype_from_deck(deck: list[int] | None) -> str:
    if not deck:
        return 'Unknown'
    c = Counter(deck)
    if c[Alakazam] >= 1 and c[Abra] >= 2:
        return 'Alakazam'
    if c[Hop_Trevenant] >= 1 or c[Hop_Phantump] >= 2:
        return 'Hop_Trevenant'
    if c[Mega_Lucario_ex] >= 1 or c[Riolu] >= 2:
        return 'Mega_Lucario'
    return 'Other'


def normalize_action(action: Any) -> list[int]:
    if action is None:
        return []
    if isinstance(action, int):
        return [int(action)]
    if isinstance(action, list):
        out = []
        for v in action:
            iv = to_int_or_none(v)
            if iv is not None:
                out.append(iv)
        return out
    iv = to_int_or_none(action)
    return [] if iv is None else [iv]


def import_module_from_path(module_name: str, path: str | Path):
    path = str(path)
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module

print('utilities ready')


utilities ready


In [4]:
# -----------------------------
# Top200 and Episode discovery
# -----------------------------

def load_top200(path: Path = TOP200_CSV_PATH) -> pd.DataFrame:
    if not path.exists():
        log_error('top200_missing', path=str(path))
        return pd.DataFrame(columns=['ranking', 'name', 'rate', 'name_norm'])
    df = pd.read_csv(path)
    rename_map = {}
    lower = {str(c).lower(): c for c in df.columns}
    if 'rank' in lower and 'ranking' not in df.columns:
        rename_map[lower['rank']] = 'ranking'
    if 'teamname' in lower and 'name' not in df.columns:
        rename_map[lower['teamname']] = 'name'
    df = df.rename(columns=rename_map).copy()
    for col in ['ranking', 'name', 'rate']:
        if col not in df.columns:
            if col == 'rate':
                df[col] = None
            else:
                raise ValueError(f'Top200 CSV missing required column: {col}; columns={list(df.columns)}')
    df['ranking'] = pd.to_numeric(df['ranking'], errors='coerce')
    df['rate'] = pd.to_numeric(df['rate'], errors='coerce')
    df['name_norm'] = df['name'].map(norm_name)
    df = df.sort_values(['ranking', 'rate'], na_position='last')
    return df


def build_top200_lookup(df: pd.DataFrame) -> dict[str, dict]:
    lookup = {}
    for _, row in df.iterrows():
        nm = norm_name(row.get('name'))
        if not nm:
            continue
        lookup[nm] = {
            'ranking': None if pd.isna(row.get('ranking')) else int(row.get('ranking')),
            'rate': None if pd.isna(row.get('rate')) else float(row.get('rate')),
        }
    return lookup


def find_episode_files(roots: list[Path] = EPISODE_ROOT_CANDIDATES, max_files: int | None = MAX_EPISODES) -> list[Path]:
    files = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        patterns = [
            str(root / '**' / '*.json'),
            str(root / '**' / 'episodes' / '*.json'),
            str(root / '**' / 'episode*.json'),
        ]
        for pat in patterns:
            for f in glob.glob(pat, recursive=True):
                p = Path(f)
                if p in seen:
                    continue
                name = p.name.lower()
                # Avoid obvious metadata jsons when possible, but keep broad recall.
                if name in {'dataset-metadata.json'}:
                    continue
                seen.add(p)
                files.append(p)
    files = sorted(files, key=lambda p: (str(p.parent), p.name))
    if max_files is not None:
        files = files[:max_files]
    return files


def read_json_file(path: Path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except UnicodeDecodeError:
        return json.loads(path.read_text(encoding='utf-8-sig'))

TOP200_DF = load_top200()
TOP200_LOOKUP = build_top200_lookup(TOP200_DF)
EPISODE_FILES = find_episode_files()
print('top200 rows:', len(TOP200_DF))
print('episode files:', len(EPISODE_FILES))


top200 rows: 200
episode files: 5516


In [5]:
# -----------------------------
# Episode metadata and deck extraction
# -----------------------------

def extract_team_names(js: dict) -> list[str]:
    candidates = [
        safe_get(js, ['info', 'TeamNames']),
        safe_get(js, ['info', 'teamNames']),
        safe_get(js, ['metadata', 'TeamNames']),
        safe_get(js, ['metadata', 'teamNames']),
        safe_get(js, ['agents']),
        safe_get(js, ['teamNames']),
    ]
    for cand in candidates:
        if isinstance(cand, list) and len(cand) >= 2:
            if isinstance(cand[0], dict):
                names = [norm_name(x.get('name') or x.get('team') or x.get('TeamName')) for x in cand[:2]]
            else:
                names = [norm_name(x) for x in cand[:2]]
            if any(names):
                return names
    # Fallback: look into first step payload names.
    names = ['', '']
    steps = js.get('steps') if isinstance(js, dict) else None
    if isinstance(steps, list):
        for step in steps[:3]:
            if isinstance(step, list):
                for pi, payload in enumerate(step[:2]):
                    if isinstance(payload, dict):
                        nm = safe_get(payload, ['info', 'name']) or safe_get(payload, ['observation', 'playerName'])
                        if nm and pi < 2:
                            names[pi] = norm_name(nm)
    return names


def extract_rewards(js: dict) -> list[Any]:
    rewards = js.get('rewards') or safe_get(js, ['info', 'rewards']) or safe_get(js, ['metadata', 'rewards'])
    if isinstance(rewards, list) and len(rewards) >= 2:
        return rewards[:2]
    # Try final step payloads.
    steps = js.get('steps') if isinstance(js, dict) else None
    if isinstance(steps, list) and steps:
        last = steps[-1]
        if isinstance(last, list):
            out = []
            for payload in last[:2]:
                if isinstance(payload, dict):
                    out.append(payload.get('reward'))
            if len(out) == 2:
                return out
    return [None, None]


def winner_from_rewards(rewards: list[Any]) -> int | None:
    vals = []
    for v in rewards[:2]:
        try:
            vals.append(float(v))
        except Exception:
            vals.append(float('nan'))
    if len(vals) < 2 or any(math.isnan(v) for v in vals):
        return None
    if vals[0] > vals[1]:
        return 0
    if vals[1] > vals[0]:
        return 1
    return None


def _collect_deck_candidates(obj: Any, out: list[list[int]], max_hits: int = 20):
    if len(out) >= max_hits:
        return
    if valid_deck_list(obj):
        out.append([int(v) for v in obj])
        return
    if isinstance(obj, dict):
        for v in obj.values():
            _collect_deck_candidates(v, out, max_hits=max_hits)
            if len(out) >= max_hits:
                return
    elif isinstance(obj, list):
        # Avoid scanning entire huge logs too deeply once enough candidates found.
        for v in obj[:200]:
            _collect_deck_candidates(v, out, max_hits=max_hits)
            if len(out) >= max_hits:
                return


def extract_decks(js: dict) -> dict[int, list[int] | None]:
    decks = {0: None, 1: None}
    steps = js.get('steps') if isinstance(js, dict) else None
    # Known path: steps[1][player]['action'] can be the submitted decklist.
    if isinstance(steps, list) and len(steps) > 1:
        for player in [0, 1]:
            cand = safe_get(steps, [1, player, 'action'])
            decks[player] = normalize_deck(cand)
    # Known fallback: visualize can carry decklists. Use only for deck extraction.
    if any(decks[p] is None for p in [0, 1]):
        cands = []
        scan_targets = []
        if isinstance(steps, list):
            scan_targets.extend(steps[:3])
        scan_targets.extend([js.get('visualize'), safe_get(js, ['steps', 0])])
        for target in scan_targets:
            _collect_deck_candidates(target, cands, max_hits=10)
        # Assign first two distinct 60-card lists if known paths failed.
        uniq = []
        seen = set()
        for d in cands:
            h = tuple(d)
            if h not in seen:
                seen.add(h)
                uniq.append(d)
        for player in [0, 1]:
            if decks[player] is None and len(uniq) > player:
                decks[player] = uniq[player]
    return decks


def episode_id_from_path_or_json(path: Path, js: dict) -> str:
    for p in [['EpisodeId'], ['episode_id'], ['info', 'EpisodeId'], ['metadata', 'EpisodeId'], ['id']]:
        v = safe_get(js, p)
        if v is not None:
            return str(v)
    return path.stem


def episode_meta_rows(path: Path, js: dict, top_lookup: dict[str, dict]):
    episode_id = episode_id_from_path_or_json(path, js)
    names = extract_team_names(js)
    rewards = extract_rewards(js)
    winner = winner_from_rewards(rewards)
    decks = extract_decks(js)
    rows_deck = []
    ranks = []
    archetypes = []
    for player in [0, 1]:
        name = names[player] if player < len(names) else ''
        info = top_lookup.get(norm_name(name), {})
        rank = info.get('ranking')
        rate = info.get('rate')
        ranks.append(rank)
        deck = decks.get(player)
        archetype = infer_archetype_from_deck(deck)
        archetypes.append(archetype)
        rows_deck.append({
            'episode_id': episode_id,
            'player_index': player,
            'player_name': name,
            'name_norm': norm_name(name),
            'rank': rank,
            'rate': rate,
            'is_top200': rank is not None and rank <= TOP_RANK_CUTOFF,
            'deck_valid': deck is not None,
            'deck_hash': stable_deck_hash(deck) if deck else None,
            'deck_list': json.dumps(deck, ensure_ascii=False) if deck else None,
            'archetype': archetype,
            'won': None if winner is None else int(player == winner),
        })
    t200_count = sum(1 for r in ranks if r is not None and r <= TOP_RANK_CUTOFF)
    tier = 'T200_T200' if t200_count == 2 else 'T200_OTHER' if t200_count == 1 else 'OTHER_OTHER'
    matchup = f'{archetypes[0]}_vs_{archetypes[1]}'
    meta = {
        'episode_id': episode_id,
        'path': str(path),
        'player0_name': names[0] if len(names) > 0 else '',
        'player1_name': names[1] if len(names) > 1 else '',
        'player0_rank': ranks[0],
        'player1_rank': ranks[1],
        'player0_archetype': archetypes[0],
        'player1_archetype': archetypes[1],
        'tier': tier,
        'matchup': matchup,
        'winner': winner,
        'reward0': rewards[0] if len(rewards) > 0 else None,
        'reward1': rewards[1] if len(rewards) > 1 else None,
        'n_steps': len(js.get('steps', [])) if isinstance(js.get('steps'), list) else None,
    }
    return meta, rows_deck

print('episode metadata helpers ready')


episode metadata helpers ready


In [6]:
# -----------------------------
# Decision / option / state extraction
# -----------------------------

# Fallback names for observed serialized enum IDs. These can be refined from reports.
SELECT_CONTEXT_FALLBACK = {
    '0': 'UNKNOWN_0',
    '1': 'SETUP_ACTIVE_POKEMON',
    '2': 'MAIN',
    '3': 'SETUP_BENCH_POKEMON',
    '4': 'SWITCH',
    '5': 'TO_HAND',
    '6': 'DISCARD',
    '7': 'TO_BENCH',
    '8': 'ATTACH_FROM',
    '9': 'TO_ACTIVE',
    '10': 'TO_DECK',
    '11': 'EVOLVE',
    '12': 'ATTACK',
    '22': 'NUMBER_OR_CHOICE',
}
OPTION_TYPE_FALLBACK = {
    '0': 'CARD', '1': 'PLAY', '2': 'ATTACH', '3': 'EVOLVE', '4': 'ABILITY',
    '5': 'RETREAT', '6': 'ATTACK', '7': 'END', '8': 'YES', '9': 'NO', '10': 'NUMBER'
}


def decode_context(x: Any) -> str:
    raw = enum_name(x, 'CONTEXT')
    return SELECT_CONTEXT_FALLBACK.get(raw, raw)


def decode_option_type(x: Any) -> str:
    raw = enum_name(x, 'OPTION')
    return OPTION_TYPE_FALLBACK.get(raw, raw)


def get_option_list(select: Any) -> list:
    if not isinstance(select, dict):
        return []
    opts = select.get('option')
    if opts is None:
        opts = select.get('options')
    return opts if isinstance(opts, list) else []


def option_get(option: Any, keys: list[str], default=None):
    if isinstance(option, dict):
        for k in keys:
            if k in option:
                return option[k]
    else:
        for k in keys:
            if hasattr(option, k):
                return getattr(option, k)
    return default


def summarize_pokemon(p: Any) -> dict:
    if p is None:
        return {}
    if isinstance(p, dict):
        pid = p.get('id') or p.get('cardId') or p.get('card_id')
        hp = p.get('hp')
        energies = p.get('energies') or p.get('energyCards') or []
        tools = p.get('tools') or []
    else:
        pid = getattr(p, 'id', getattr(p, 'cardId', None))
        hp = getattr(p, 'hp', None)
        energies = getattr(p, 'energies', getattr(p, 'energyCards', []))
        tools = getattr(p, 'tools', [])
    energy_ids = []
    for e in as_list(energies):
        if isinstance(e, dict):
            energy_ids.append(e.get('id') or e.get('cardId'))
        else:
            energy_ids.append(getattr(e, 'id', getattr(e, 'cardId', e)))
    tool_ids = []
    for t in as_list(tools):
        if isinstance(t, dict):
            tool_ids.append(t.get('id') or t.get('cardId'))
        else:
            tool_ids.append(getattr(t, 'id', getattr(t, 'cardId', t)))
    return {
        'id': to_int_or_none(pid),
        'hp': to_int_or_none(hp),
        'energy_count': len([x for x in energy_ids if x is not None]),
        'tool_count': len([x for x in tool_ids if x is not None]),
        'energy_ids': [to_int_or_none(x) for x in energy_ids if to_int_or_none(x) is not None],
        'tool_ids': [to_int_or_none(x) for x in tool_ids if to_int_or_none(x) is not None],
    }


def player_state_from_obs(obs: dict, player_index: int) -> dict:
    players = safe_get(obs, ['current', 'players'], [])
    if isinstance(players, list) and len(players) > player_index:
        ps = players[player_index]
    else:
        ps = {}
    return ps if isinstance(ps, dict) else {}


def count_zone(ps: dict, key: str, count_key: str | None = None) -> int | None:
    val = ps.get(key)
    if isinstance(val, list):
        return len([x for x in val if x is not None])
    if count_key and count_key in ps:
        return to_int_or_none(ps.get(count_key))
    return None


def state_summary_from_obs(obs: dict, decision_id: str, episode_id: str, player_index: int) -> dict:
    cur = obs.get('current', {}) if isinstance(obs, dict) else {}
    your_index = to_int_or_none(cur.get('yourIndex'))
    if your_index is None:
        your_index = player_index
    op_index = 1 - player_index
    my = player_state_from_obs(obs, player_index)
    op = player_state_from_obs(obs, op_index)
    my_active = summarize_pokemon((my.get('active') or [None])[0] if isinstance(my.get('active'), list) and my.get('active') else None)
    op_active = summarize_pokemon((op.get('active') or [None])[0] if isinstance(op.get('active'), list) and op.get('active') else None)
    my_bench = [summarize_pokemon(x) for x in as_list(my.get('bench')) if x is not None]
    op_bench = [summarize_pokemon(x) for x in as_list(op.get('bench')) if x is not None]
    stadium = safe_get(cur, ['stadium'], [])
    stadium_id = None
    if isinstance(stadium, list) and stadium:
        stadium_id = summarize_pokemon(stadium[0]).get('id')
    return {
        'decision_id': decision_id,
        'episode_id': episode_id,
        'player_index': player_index,
        'turn': to_int_or_none(cur.get('turn')),
        'turn_action_count': to_int_or_none(cur.get('turnActionCount')),
        'supporter_played': bool(cur.get('supporterPlayed')) if 'supporterPlayed' in cur else None,
        'stadium_played': bool(cur.get('stadiumPlayed')) if 'stadiumPlayed' in cur else None,
        'energy_attached': bool(cur.get('energyAttached')) if 'energyAttached' in cur else None,
        'my_active_id': my_active.get('id'),
        'my_active_hp': my_active.get('hp'),
        'my_active_energy_count': my_active.get('energy_count'),
        'op_active_id': op_active.get('id'),
        'op_active_hp': op_active.get('hp'),
        'op_active_energy_count': op_active.get('energy_count'),
        'my_bench_count': len(my_bench),
        'op_bench_count': len(op_bench),
        'my_alakazam_count': sum(1 for x in [my_active] + my_bench if x.get('id') == Alakazam),
        'my_kadabra_count': sum(1 for x in [my_active] + my_bench if x.get('id') == Kadabra),
        'my_abra_count': sum(1 for x in [my_active] + my_bench if x.get('id') == Abra),
        'my_dudunsparce_count': sum(1 for x in [my_active] + my_bench if x.get('id') == Dudunsparce),
        'my_hand_count': count_zone(my, 'hand', 'handCount'),
        'op_hand_count': count_zone(op, 'hand', 'handCount'),
        'my_deck_count': count_zone(my, 'deck', 'deckCount'),
        'op_deck_count': count_zone(op, 'deck', 'deckCount'),
        'my_prizes_left': count_zone(my, 'prize', 'prizeCount'),
        'op_prizes_left': count_zone(op, 'prize', 'prizeCount'),
        'my_discard_count': count_zone(my, 'discard', 'discardCount'),
        'op_discard_count': count_zone(op, 'discard', 'discardCount'),
        'stadium_id': stadium_id,
        'powerful_hand_damage_est': (count_zone(my, 'hand', 'handCount') or 0) * 20,
        'deckout_risk': 1 if (count_zone(my, 'deck', 'deckCount') or 99) <= 4 else 0,
    }


def extract_option_features(option: Any) -> dict:
    typ = decode_option_type(option_get(option, ['type', 'optionType']))
    row = {
        'option_type': typ,
        'area': enum_name(option_get(option, ['area']), 'AREA'),
        'index': to_int_or_none(option_get(option, ['index'])),
        'player_index_option': to_int_or_none(option_get(option, ['playerIndex', 'player_index'])),
        'card_id': to_int_or_none(option_get(option, ['cardId', 'card_id', 'id'])),
        'target_card_id': to_int_or_none(option_get(option, ['targetCardId', 'target_card_id'])),
        'attack_id': to_int_or_none(option_get(option, ['attackId', 'attack_id'])),
        'number_value': to_int_or_none(option_get(option, ['number', 'value'])),
        'in_play_area': enum_name(option_get(option, ['inPlayArea', 'in_play_area']), 'AREA'),
        'in_play_index': to_int_or_none(option_get(option, ['inPlayIndex', 'in_play_index'])),
        'remain_damage_counter': to_int_or_none(option_get(option, ['remainDamageCounter', 'remain_damage_counter'])),
        'remain_energy_cost': to_int_or_none(option_get(option, ['remainEnergyCost', 'remain_energy_cost'])),
    }
    return row


def iter_episode_decisions(js: dict, meta: dict, deck_row_by_player: dict[int, dict]):
    steps = js.get('steps') if isinstance(js, dict) else None
    if not isinstance(steps, list):
        return [], [], []
    decision_rows, option_rows, state_rows = [], [], []
    episode_id = meta['episode_id']
    for step_idx, step in enumerate(steps):
        if not isinstance(step, list):
            continue
        for player_index, payload in enumerate(step[:2]):
            if not isinstance(payload, dict):
                continue
            obs = payload.get('observation')
            if not isinstance(obs, dict):
                continue
            select = obs.get('select')
            if not isinstance(select, dict):
                continue
            options = get_option_list(select)
            if not options:
                continue
            action = normalize_action(payload.get('action'))
            min_count = to_int_or_none(select.get('minCount'))
            max_count = to_int_or_none(select.get('maxCount'))
            if min_count is None:
                min_count = 0
            if max_count is None:
                max_count = len(options)
            chosen_valid = (
                len(action) >= min_count and len(action) <= max_count and
                len(set(action)) == len(action) and all(0 <= a < len(options) for a in action)
            )
            context_name = decode_context(select.get('context'))
            decision_id = f'{episode_id}:{step_idx}:{player_index}'
            deck_info = deck_row_by_player.get(player_index, {})
            drow = {
                'decision_id': decision_id,
                'episode_id': episode_id,
                'step': step_idx,
                'player_index': player_index,
                'player_name': deck_info.get('player_name'),
                'rank': deck_info.get('rank'),
                'rate': deck_info.get('rate'),
                'is_top200': deck_info.get('is_top200'),
                'player_archetype': deck_info.get('archetype'),
                'opponent_archetype': deck_row_by_player.get(1 - player_index, {}).get('archetype'),
                'tier': meta.get('tier'),
                'matchup': meta.get('matchup'),
                'context_raw': select.get('context'),
                'context_name': context_name,
                'min_count': min_count,
                'max_count': max_count,
                'num_options': len(options),
                'chosen_action': json.dumps(action),
                'chosen_count': len(action),
                'chosen_valid': chosen_valid,
                'won': deck_info.get('won'),
            }
            decision_rows.append(drow)
            state_rows.append(state_summary_from_obs(obs, decision_id, episode_id, player_index))
            chosen_set = set(action)
            for oi, opt in enumerate(options):
                orow = {
                    'decision_id': decision_id,
                    'episode_id': episode_id,
                    'step': step_idx,
                    'player_index': player_index,
                    'context_name': context_name,
                    'option_index': oi,
                    'is_chosen': int(oi in chosen_set),
                }
                try:
                    orow.update(extract_option_features(opt))
                except Exception as exc:
                    orow.update({'option_type': 'EXTRACT_ERROR', 'extract_error': repr(exc)})
                option_rows.append(orow)
    return decision_rows, option_rows, state_rows

print('decision extraction helpers ready')


decision extraction helpers ready


In [7]:
# -----------------------------
# Run replay mining
# -----------------------------
EPISODE_INDEX_DF = pd.DataFrame()
DECKLISTS_DF = pd.DataFrame()
DECISION_ROWS_DF = pd.DataFrame()
OPTION_ROWS_DF = pd.DataFrame()
STATE_SUMMARY_DF = pd.DataFrame()
NON_EPISODE_JSON_ROWS = []

if RUN_REPLAY_MINING:
    episode_rows = []
    deck_rows_all = []
    decision_rows_all = []
    option_rows_all = []
    state_rows_all = []
    t0 = time.time()
    for i, path in enumerate(EPISODE_FILES):
        try:
            js = read_json_file(path)
            steps = js.get('steps') if isinstance(js, dict) else None
            if not isinstance(steps, list) or len(steps) < 2 or not isinstance(steps[0], list):
                NON_EPISODE_JSON_ROWS.append({'path': str(path), 'reason': 'not_episode_like_steps'})
                continue
            meta, deck_rows = episode_meta_rows(path, js, TOP200_LOOKUP)
            episode_rows.append(meta)
            deck_rows_all.extend(deck_rows)
            deck_by_player = {r['player_index']: r for r in deck_rows}
            drows, orows, srows = iter_episode_decisions(js, meta, deck_by_player)
            decision_rows_all.extend(drows)
            option_rows_all.extend(orows)
            state_rows_all.extend(srows)
        except Exception as exc:
            log_error('episode_parse', path=str(path), error=repr(exc), traceback=traceback.format_exc(limit=3))
        if (i + 1) % 250 == 0:
            print(f'parsed {i+1}/{len(EPISODE_FILES)} episodes; decisions={len(decision_rows_all):,}; options={len(option_rows_all):,}')
    EPISODE_INDEX_DF = pd.DataFrame(episode_rows)
    DECKLISTS_DF = pd.DataFrame(deck_rows_all)
    DECISION_ROWS_DF = pd.DataFrame(decision_rows_all)
    OPTION_ROWS_DF = pd.DataFrame(option_rows_all)
    STATE_SUMMARY_DF = pd.DataFrame(state_rows_all)
    print('mining seconds:', round(time.time() - t0, 2))
else:
    print('RUN_REPLAY_MINING=False; using empty dataframes')

ARTIFACT_PATHS = {}

# Deterministic episode-level split for future loop promotion checks.
# This is reporting-only in v0-05d3; model training and agent behavior are unchanged.
def stable_episode_bucket(episode_id: Any) -> int:
    import hashlib
    digest = hashlib.sha1(str(episode_id).encode('utf-8')).hexdigest()
    return int(digest[:12], 16) % 10000


def stable_episode_split(bucket: int) -> str:
    if bucket < 7000:
        return 'train'
    if bucket < 8500:
        return 'valid'
    return 'holdout'


def build_episode_split_df(episodes: pd.DataFrame) -> pd.DataFrame:
    cols = ['episode_id', 'split_bucket', 'split_pct', 'split', 'tier', 'matchup', 'winner', 'n_steps', 'stress_long_game', 'stress_top200_tier']
    if episodes is None or episodes.empty or 'episode_id' not in episodes.columns:
        return pd.DataFrame(columns=cols)
    out = episodes.copy()
    out['episode_id'] = out['episode_id'].astype(str)
    out['split_bucket'] = out['episode_id'].map(stable_episode_bucket).astype(int)
    out['split_pct'] = out['split_bucket'] / 10000.0
    out['split'] = out['split_bucket'].map(stable_episode_split)
    n_steps = pd.to_numeric(out.get('n_steps', pd.Series(index=out.index, dtype='float64')), errors='coerce')
    long_threshold = float(n_steps.quantile(0.75)) if n_steps.notna().any() else float('inf')
    out['stress_long_game'] = n_steps >= long_threshold
    out['stress_top200_tier'] = out.get('tier', '').astype(str).str.contains('T200', na=False)
    for c in cols:
        if c not in out.columns:
            out[c] = None
    return out[cols].sort_values(['split', 'episode_id']).reset_index(drop=True)


def build_split_summary_df(episode_split: pd.DataFrame, decisions: pd.DataFrame, options: pd.DataFrame) -> pd.DataFrame:
    if episode_split is None or episode_split.empty:
        return pd.DataFrame()
    ep = episode_split.copy()
    decision_counts = pd.DataFrame(columns=['episode_id', 'decision_rows', 'unknown0_decisions'])
    if decisions is not None and not decisions.empty and 'episode_id' in decisions.columns:
        d = decisions.copy()
        d['episode_id'] = d['episode_id'].astype(str)
        decision_counts = d.groupby('episode_id').agg(
            decision_rows=('decision_id', 'size'),
            unknown0_decisions=('context_name', lambda s: int((s.astype(str) == 'UNKNOWN_0').sum())),
        ).reset_index()
    option_counts = pd.DataFrame(columns=['episode_id', 'option_rows'])
    if options is not None and not options.empty and 'episode_id' in options.columns:
        o = options[['episode_id']].copy()
        o['episode_id'] = o['episode_id'].astype(str)
        option_counts = o.groupby('episode_id').size().rename('option_rows').reset_index()
    work = ep.merge(decision_counts, on='episode_id', how='left').merge(option_counts, on='episode_id', how='left')
    for c in ['decision_rows', 'unknown0_decisions', 'option_rows']:
        work[c] = pd.to_numeric(work[c], errors='coerce').fillna(0).astype(int)
    return work.groupby('split', dropna=False).agg(
        episodes=('episode_id', 'nunique'),
        split_bucket_min=('split_bucket', 'min'),
        split_bucket_max=('split_bucket', 'max'),
        decision_rows=('decision_rows', 'sum'),
        option_rows=('option_rows', 'sum'),
        unknown0_decisions=('unknown0_decisions', 'sum'),
        top200_tier_episodes=('stress_top200_tier', 'sum'),
        long_game_episodes=('stress_long_game', 'sum'),
        avg_n_steps=('n_steps', 'mean'),
    ).reset_index()


def build_stress_slice_summary_df(episode_split: pd.DataFrame, decisions: pd.DataFrame) -> pd.DataFrame:
    if episode_split is None or episode_split.empty or decisions is None or decisions.empty or 'episode_id' not in decisions.columns:
        return pd.DataFrame()
    split_cols = ['episode_id', 'split', 'stress_long_game', 'stress_top200_tier']
    dcols = [c for c in ['episode_id', 'decision_id', 'context_name', 'tier', 'matchup', 'won', 'chosen_valid', 'num_options'] if c in decisions.columns]
    d = decisions[dcols].copy()
    d['episode_id'] = d['episode_id'].astype(str)
    work = d.merge(episode_split[split_cols], on='episode_id', how='left')
    work['split'] = work['split'].fillna('unknown')
    rows = []

    def add_slice(name: str, mask):
        sub = work.loc[mask].copy()
        if sub.empty:
            return
        for split, grp in sub.groupby('split', dropna=False):
            rows.append({
                'slice': name,
                'split': split,
                'episodes': int(grp['episode_id'].nunique()),
                'decisions': int(len(grp)),
                'unknown0_decisions': int((grp.get('context_name', '').astype(str) == 'UNKNOWN_0').sum()) if 'context_name' in grp else 0,
                'win_rate': float(pd.to_numeric(grp.get('won', pd.Series(dtype='float64')), errors='coerce').mean()) if 'won' in grp else None,
                'chosen_valid_rate': float(pd.to_numeric(grp.get('chosen_valid', pd.Series(dtype='float64')), errors='coerce').mean()) if 'chosen_valid' in grp else None,
                'avg_num_options': float(pd.to_numeric(grp.get('num_options', pd.Series(dtype='float64')), errors='coerce').mean()) if 'num_options' in grp else None,
            })

    add_slice('all_decisions', pd.Series(True, index=work.index))
    if 'context_name' in work.columns:
        add_slice('unknown0_decisions', work['context_name'].astype(str) == 'UNKNOWN_0')
    add_slice('top200_tier_decisions', work['stress_top200_tier'].fillna(False).astype(bool))
    add_slice('long_game_decisions', work['stress_long_game'].fillna(False).astype(bool))
    if 'won' in work.columns:
        add_slice('losing_player_decisions', pd.to_numeric(work['won'], errors='coerce') == 0)
    if 'tier' in work.columns:
        for tier in sorted(x for x in work['tier'].dropna().astype(str).unique() if x):
            add_slice(f'tier::{tier}', work['tier'].astype(str) == tier)
    if 'matchup' in work.columns:
        top_matchups = work['matchup'].dropna().astype(str).value_counts().head(12).index.tolist()
        for matchup in top_matchups:
            add_slice(f'matchup::{matchup}', work['matchup'].astype(str) == matchup)
    return pd.DataFrame(rows).sort_values(['slice', 'split']).reset_index(drop=True)


EPISODE_SPLIT_DF = build_episode_split_df(EPISODE_INDEX_DF)
SPLIT_SUMMARY_DF = build_split_summary_df(EPISODE_SPLIT_DF, DECISION_ROWS_DF, OPTION_ROWS_DF)
STRESS_SLICE_SUMMARY_DF = build_stress_slice_summary_df(EPISODE_SPLIT_DF, DECISION_ROWS_DF)

ARTIFACT_PATHS['episode_index'] = safe_save_table(EPISODE_INDEX_DF, OUTPUT_DIR / 'episode_index')
ARTIFACT_PATHS['decklists'] = safe_save_table(DECKLISTS_DF, OUTPUT_DIR / 'decklists')
ARTIFACT_PATHS['decision_rows'] = safe_save_table(DECISION_ROWS_DF, OUTPUT_DIR / 'decision_rows')
ARTIFACT_PATHS['option_rows'] = safe_save_table(OPTION_ROWS_DF, OUTPUT_DIR / 'option_rows')
ARTIFACT_PATHS['state_summary'] = safe_save_table(STATE_SUMMARY_DF, OUTPUT_DIR / 'state_summary')
NON_EPISODE_JSON_DF = pd.DataFrame(NON_EPISODE_JSON_ROWS)
ARTIFACT_PATHS['non_episode_json'] = safe_save_table(NON_EPISODE_JSON_DF, OUTPUT_DIR / 'non_episode_json')
ARTIFACT_PATHS['episode_split'] = safe_save_table(EPISODE_SPLIT_DF, OUTPUT_DIR / 'episode_split')

# --- v0-05d9: Monte Carlo return weights ---
if USE_MC_RETURN_WEIGHTS and not DECISION_ROWS_DF.empty and not EPISODE_SPLIT_DF.empty:
    _ep_steps = EPISODE_SPLIT_DF[['episode_id','n_steps']].drop_duplicates('episode_id').copy()
    _ep_steps['episode_id'] = _ep_steps['episode_id'].astype(str)
    DECISION_ROWS_DF['episode_id'] = DECISION_ROWS_DF['episode_id'].astype(str)
    DECISION_ROWS_DF = DECISION_ROWS_DF.merge(
        _ep_steps.rename(columns={'n_steps': '_mc_n_steps'}), on='episode_id', how='left'
    )
    _step = DECISION_ROWS_DF.get('step', DECISION_ROWS_DF.index.to_series())
    _T = DECISION_ROWS_DF['_mc_n_steps'].fillna(_step + 1)
    if USE_PHASE_AWARE_WEIGHT:
        _frac = (_step / _T.clip(lower=1)).clip(0, 1)
        DECISION_ROWS_DF['step_frac'] = _frac
        _pw = _frac.map(lambda x: 0.5 if x < 1/3 else (1.0 if x <= 2/3 else 0.2)).astype(float)
        DECISION_ROWS_DF['phase_weight'] = _pw
        DECISION_ROWS_DF['mc_step_weight'] = _pw
    else:
        DECISION_ROWS_DF['mc_step_weight'] = MC_DISCOUNT_GAMMA ** (_T - _step).clip(lower=0)
        DECISION_ROWS_DF['phase_weight'] = DECISION_ROWS_DF['mc_step_weight']
    DECISION_ROWS_DF = DECISION_ROWS_DF.drop(columns=['_mc_n_steps'])
    _w = DECISION_ROWS_DF['mc_step_weight']
    print(f'mc_step_weight (phase_aware={USE_PHASE_AWARE_WEIGHT}): min={_w.min():.4f} mean={_w.mean():.4f} max={_w.max():.4f}')
else:
    DECISION_ROWS_DF['mc_step_weight'] = 1.0
    DECISION_ROWS_DF['phase_weight'] = 1.0
    print('mc_step_weight: uniform 1.0')

# v05d16: Top200 episode filter
if TOP200_EPISODE_FILTER and not DECISION_ROWS_DF.empty and 'tier' in DECISION_ROWS_DF.columns:
    _n_before = len(DECISION_ROWS_DF)
    DECISION_ROWS_DF = DECISION_ROWS_DF[DECISION_ROWS_DF['tier'] == 'T200_T200'].copy()
    print(f'T200_T200 filter: {_n_before} → {len(DECISION_ROWS_DF)} rows')

ARTIFACT_PATHS['split_summary'] = safe_save_table(SPLIT_SUMMARY_DF, OUTPUT_DIR / 'split_summary')
ARTIFACT_PATHS['stress_slice_summary'] = safe_save_table(STRESS_SLICE_SUMMARY_DF, OUTPUT_DIR / 'stress_slice_summary')
print('episode_index:', EPISODE_INDEX_DF.shape)
print('decklists:', DECKLISTS_DF.shape)
print('decision_rows:', DECISION_ROWS_DF.shape)
print('option_rows:', OPTION_ROWS_DF.shape)
print('state_summary:', STATE_SUMMARY_DF.shape)
print('episode_split:', EPISODE_SPLIT_DF.shape)
print('split_summary:', SPLIT_SUMMARY_DF.shape)
print('stress_slice_summary:', STRESS_SLICE_SUMMARY_DF.shape)
print('non_episode_json_skipped:', len(NON_EPISODE_JSON_ROWS))
print('errors:', len(V05_ERROR_ROWS))


parsed 250/5516 episodes; decisions=71,286; options=400,337
parsed 500/5516 episodes; decisions=137,725; options=770,271
parsed 750/5516 episodes; decisions=205,291; options=1,149,528
parsed 1000/5516 episodes; decisions=273,011; options=1,521,445
parsed 1250/5516 episodes; decisions=339,349; options=1,881,909
parsed 1500/5516 episodes; decisions=408,118; options=2,270,084
parsed 1750/5516 episodes; decisions=478,187; options=2,660,402
parsed 2000/5516 episodes; decisions=547,723; options=3,041,567
parsed 2250/5516 episodes; decisions=630,061; options=3,460,056
parsed 2500/5516 episodes; decisions=698,111; options=3,834,862
parsed 2750/5516 episodes; decisions=766,103; options=4,209,205
parsed 3000/5516 episodes; decisions=832,599; options=4,578,513
parsed 3250/5516 episodes; decisions=900,070; options=4,959,039
parsed 3500/5516 episodes; decisions=969,054; options=5,346,668
parsed 3750/5516 episodes; decisions=1,034,441; options=5,709,338
parsed 4000/5516 episodes; decisions=1,103,353

In [8]:
# -----------------------------
# Alakazam deck selection and reports
# -----------------------------

def deck_list_from_json_cell(x: Any) -> list[int] | None:
    if isinstance(x, list):
        return normalize_deck(x)
    if isinstance(x, str) and x:
        try:
            return normalize_deck(json.loads(x))
        except Exception:
            return None
    return None


def deck_ranking_report(decklists: pd.DataFrame) -> pd.DataFrame:
    """Rank observed Alakazam deck hashes.

    v0-05b1 fix: this function must use the representative deck stored in
    ``sample_deck``. The previous v0-05b cell accidentally referenced a stale
    local variable named ``deck`` and also reused candidate-comparison variables
    such as ``row``, ``hop_games`` and ``mirror_games`` before they existed.
    This version keeps deck-ranking self-contained and fail-soft.
    """
    if decklists is None or decklists.empty:
        return pd.DataFrame()
    df = decklists[(decklists.get('deck_valid') == True) & (decklists.get('archetype') == 'Alakazam')].copy()
    if df.empty:
        return pd.DataFrame()
    rows = []
    for h, g in df.groupby('deck_hash'):
        wins = pd.to_numeric(g.get('won'), errors='coerce') if 'won' in g.columns else pd.Series(dtype=float)
        ranks = pd.to_numeric(g.get('rank'), errors='coerce') if 'rank' in g.columns else pd.Series(dtype=float)

        sample_deck = None
        if 'deck_list' in g.columns and g['deck_list'].notna().any():
            try:
                sample_deck = deck_list_from_json_cell(g['deck_list'].dropna().iloc[0])
            except Exception:
                sample_deck = None
        if not isinstance(sample_deck, list):
            sample_deck = []

        deck_counts = Counter(sample_deck)
        main_cards_present = {
            'Abra': int(deck_counts.get(Abra, 0)),
            'Kadabra': int(deck_counts.get(Kadabra, 0)),
            'Alakazam': int(deck_counts.get(Alakazam, 0)),
            'Rare_Candy': int(deck_counts.get(Rare_Candy, 0)),
            'Dudunsparce': int(deck_counts.get(Dudunsparce, 0)),
            'Enhanced_Hammer': int(deck_counts.get(Enhanced_Hammer, 0)),
            'Nighttime_Mine': int(deck_counts.get(Nighttime_Mine, 0)),
            'Boss_Orders': int(deck_counts.get(Boss_Orders, 0)),
        }
        deck_valid_60 = len(sample_deck) == 60
        games = int(len(g))
        risk_flags = []
        if not deck_valid_60:
            risk_flags.append('invalid_or_missing_decklist')
        if games < 50:
            risk_flags.append('low_sample')
        if main_cards_present['Abra'] < 3 or main_cards_present['Alakazam'] < 2:
            risk_flags.append('thin_alakazam_line')

        rows.append({
            'deck_hash': str(h),
            'games': games,
            'wins': int(wins.fillna(0).sum()) if len(wins) else 0,
            'winrate': float(wins.mean()) if len(wins) and wins.notna().any() else None,
            'top10_uses': int((ranks <= 10).sum()) if len(ranks) else 0,
            'top50_uses': int((ranks <= 50).sum()) if len(ranks) else 0,
            'top200_uses': int((ranks <= 200).sum()) if len(ranks) else 0,
            'avg_rank': float(ranks.mean()) if len(ranks) and ranks.notna().any() else None,
            'deck_size_valid': bool(deck_valid_60),
            'main_cards_present': json.dumps(main_cards_present, ensure_ascii=False),
            'risk_flags': json.dumps(risk_flags, ensure_ascii=False),
            'deck_list': json.dumps(sample_deck, ensure_ascii=False) if sample_deck else None,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out['selection_score'] = (
            pd.to_numeric(out['games'], errors='coerce').clip(upper=1000).fillna(0) / 1000.0 +
            pd.to_numeric(out['winrate'], errors='coerce').fillna(0.5) * 2.0 +
            pd.to_numeric(out['top10_uses'], errors='coerce').fillna(0) * 0.15 +
            pd.to_numeric(out['top50_uses'], errors='coerce').fillna(0) * 0.03
        )
        out = out.sort_values(['selection_score', 'games', 'winrate'], ascending=False)
    return out


def matchup_report(decklists: pd.DataFrame) -> pd.DataFrame:
    if decklists.empty:
        return pd.DataFrame()
    rows = []
    for eid, grp in decklists.groupby('episode_id'):
        if len(grp) < 2:
            continue
        for _, me in grp.iterrows():
            if me.get('archetype') != 'Alakazam':
                continue
            opp = grp[grp['player_index'] != me['player_index']]
            if opp.empty:
                continue
            opp = opp.iloc[0]
            rows.append({
                'episode_id': eid,
                'my_deck_hash': me.get('deck_hash'),
                'my_rank': me.get('rank'),
                'opponent_archetype': opp.get('archetype'),
                'opponent_rank': opp.get('rank'),
                'won': me.get('won'),
                'tier': 'T200_T200' if bool(me.get('is_top200')) and bool(opp.get('is_top200')) else 'T200_OTHER' if bool(me.get('is_top200')) or bool(opp.get('is_top200')) else 'OTHER_OTHER',
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame()
    g = df.groupby(['my_deck_hash', 'opponent_archetype', 'tier'], dropna=False).agg(
        games=('won', 'count'),
        winrate=('won', 'mean'),
        avg_my_rank=('my_rank', 'mean'),
    ).reset_index().sort_values(['games', 'winrate'], ascending=False)
    return g


def candidate_deck_comparison(ranking: pd.DataFrame, matchup: pd.DataFrame, candidate_hashes: list[str]) -> pd.DataFrame:
    hashes = []
    for h in list(candidate_hashes or []) + ([stable_deck_hash(DEFAULT_ALAKAZAM_DECK)]):
        if h and h not in hashes:
            hashes.append(h)
    if not ranking.empty:
        for h in ranking.head(10)['deck_hash'].astype(str).tolist():
            if h not in hashes:
                hashes.append(h)
    rows = []
    for h in hashes:
        base = ranking[ranking['deck_hash'].astype(str) == str(h)].head(1) if not ranking.empty else pd.DataFrame()
        row = base.iloc[0].to_dict() if not base.empty else {}
        sub = matchup[matchup['my_deck_hash'].astype(str) == str(h)].copy() if not matchup.empty else pd.DataFrame()
        def rate_for(arch: str):
            ss = sub[sub['opponent_archetype'] == arch]
            if ss.empty:
                return None, 0
            return float(np.average(pd.to_numeric(ss['winrate'], errors='coerce').fillna(0.5), weights=pd.to_numeric(ss['games'], errors='coerce').fillna(0))) if np is not None else float(ss['winrate'].mean()), int(pd.to_numeric(ss['games'], errors='coerce').fillna(0).sum())
        hop_wr, hop_games = rate_for('Hop_Trevenant')
        mirror_wr, mirror_games = rate_for('Alakazam')
        lucario_wr, lucario_games = rate_for('Mega_Lucario')
        deck = deck_list_from_json_cell(row.get('deck_list')) if row else None
        if h == stable_deck_hash(DEFAULT_ALAKAZAM_DECK) and deck is None:
            deck = list(DEFAULT_ALAKAZAM_DECK)
        deck_counts = Counter(deck or [])
        main_cards_present = {
            'Abra': int(deck_counts.get(Abra, 0)),
            'Kadabra': int(deck_counts.get(Kadabra, 0)),
            'Alakazam': int(deck_counts.get(Alakazam, 0)),
            'Rare_Candy': int(deck_counts.get(Rare_Candy, 0)),
            'Dudunsparce': int(deck_counts.get(Dudunsparce, 0)),
            'Enhanced_Hammer': int(deck_counts.get(Enhanced_Hammer, 0)),
            'Nighttime_Mine': int(deck_counts.get(Nighttime_Mine, 0)),
            'Boss_Orders': int(deck_counts.get(Boss_Orders, 0)),
        }
        risk_flags = []
        if deck is None or len(deck) != 60:
            risk_flags.append('invalid_or_missing_decklist')
        if int(row.get('games', 0) or 0) < 50:
            risk_flags.append('low_sample')
        if hop_games < 20:
            risk_flags.append('low_hop_sample')
        if mirror_games < 20:
            risk_flags.append('low_mirror_sample')
        if main_cards_present['Abra'] < 3 or main_cards_present['Alakazam'] < 2:
            risk_flags.append('thin_alakazam_line')
        rows.append({
            'deck_hash': h,
            'is_default_fallback': h == stable_deck_hash(DEFAULT_ALAKAZAM_DECK),
            'games': int(row.get('games', 0) or 0),
            'winrate': row.get('winrate'),
            'top10_uses': int(row.get('top10_uses', 0) or 0),
            'top50_uses': int(row.get('top50_uses', 0) or 0),
            'avg_rank': row.get('avg_rank'),
            'selection_score': row.get('selection_score'),
            'vs_hop_games': hop_games,
            'vs_hop_winrate': hop_wr,
            'mirror_games': mirror_games,
            'mirror_winrate': mirror_wr,
            'vs_lucario_games': lucario_games,
            'vs_lucario_winrate': lucario_wr,
            'deck_size_valid': bool(deck is not None and len(deck) == 60),
            'main_cards_present': json.dumps(main_cards_present, ensure_ascii=False),
            'risk_flags': json.dumps(risk_flags, ensure_ascii=False),
            'deck_list': json.dumps(deck, ensure_ascii=False) if deck else None,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out['review_score'] = (
            pd.to_numeric(out['top10_uses'], errors='coerce').fillna(0) * 0.20 +
            pd.to_numeric(out['top50_uses'], errors='coerce').fillna(0) * 0.04 +
            pd.to_numeric(out['games'], errors='coerce').clip(upper=500).fillna(0) * 0.002 +
            pd.to_numeric(out['vs_hop_winrate'], errors='coerce').fillna(0.5) * 2.0 +
            pd.to_numeric(out['mirror_winrate'], errors='coerce').fillna(0.5) * 2.0
        )
        out = out.sort_values(['review_score', 'selection_score', 'games'], ascending=False)
    return out


ALAKAZAM_DECK_RANKING_DF = deck_ranking_report(DECKLISTS_DF) if RUN_DECK_SELECTION else pd.DataFrame()
ALAKAZAM_MATCHUP_REPORT_DF = matchup_report(DECKLISTS_DF) if RUN_DECK_SELECTION else pd.DataFrame()
ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF = candidate_deck_comparison(ALAKAZAM_DECK_RANKING_DF, ALAKAZAM_MATCHUP_REPORT_DF, CANDIDATE_DECK_HASHES)

ARTIFACT_PATHS['alakazam_deck_ranking'] = safe_save_table(ALAKAZAM_DECK_RANKING_DF, OUTPUT_DIR / 'alakazam_deck_ranking')
ARTIFACT_PATHS['alakazam_matchup_report'] = safe_save_table(ALAKAZAM_MATCHUP_REPORT_DF, OUTPUT_DIR / 'alakazam_matchup_report')
ARTIFACT_PATHS['alakazam_candidate_deck_comparison'] = safe_save_table(ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF, OUTPUT_DIR / 'alakazam_candidate_deck_comparison')

SELECTED_ALAKAZAM_DECK = list(DEFAULT_ALAKAZAM_DECK)
SELECTED_ALAKAZAM_DECK_SOURCE = 'default_kadoraba_fallback'
SELECTED_ALAKAZAM_DECK_HASH = stable_deck_hash(SELECTED_ALAKAZAM_DECK)


def try_select_deck_by_hash(deck_hash: str) -> tuple[list[int] | None, str]:
    if not deck_hash:
        return None, ''
    pools = [ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF, ALAKAZAM_DECK_RANKING_DF]
    for pool in pools:
        if pool is None or pool.empty or 'deck_hash' not in pool.columns:
            continue
        hit = pool[pool['deck_hash'].astype(str) == str(deck_hash)]
        if not hit.empty:
            deck = deck_list_from_json_cell(hit.iloc[0].get('deck_list'))
            if deck and len(deck) == 60:
                return deck, f'hash_override:{deck_hash}'
    if deck_hash == stable_deck_hash(DEFAULT_ALAKAZAM_DECK):
        return list(DEFAULT_ALAKAZAM_DECK), f'hash_override_default:{deck_hash}'
    return None, ''

if SUBMISSION_DECK_HASH_OVERRIDE:
    cand, src = try_select_deck_by_hash(SUBMISSION_DECK_HASH_OVERRIDE)
    if cand:
        SELECTED_ALAKAZAM_DECK = cand
        SELECTED_ALAKAZAM_DECK_SOURCE = src
        SELECTED_ALAKAZAM_DECK_HASH = stable_deck_hash(cand)
    else:
        log_error('deck_hash_override_not_found', deck_hash=SUBMISSION_DECK_HASH_OVERRIDE)
elif AUTO_SELECT_SUBMISSION_DECK and not ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF.empty:
    for _, row in ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF.iterrows():
        cand = deck_list_from_json_cell(row.get('deck_list'))
        if cand and len(cand) == 60:
            SELECTED_ALAKAZAM_DECK = cand
            SELECTED_ALAKAZAM_DECK_SOURCE = 'auto_selected_from_candidate_comparison'
            SELECTED_ALAKAZAM_DECK_HASH = stable_deck_hash(cand)
            break

print('selected deck source:', SELECTED_ALAKAZAM_DECK_SOURCE)
print('selected deck hash:', SELECTED_ALAKAZAM_DECK_HASH)
print('candidate deck comparison preview:')
display(ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF.head(12)) if 'display' in globals() else print(ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF.head(12))
print('deck ranking preview:')
display(ALAKAZAM_DECK_RANKING_DF.head(10)) if 'display' in globals() else print(ALAKAZAM_DECK_RANKING_DF.head(10))


selected deck source: default_kadoraba_fallback
selected deck hash: e702674c1864
candidate deck comparison preview:
       deck_hash  is_default_fallback  games   winrate  top10_uses  \
5   8297d036a5ee                False     33  0.636364          28   
1   2420d2c8ce4b                False    725  0.522759           0   
6   96275182a9da                False    107  0.588785           0   
0   e702674c1864                 True    138  0.500000          10   
7   b83fa38ef906                False     39  0.666667           0   
9   82d5654c2507                False     38  0.605263           0   
8   c9999e6386ea                False     36  0.666667           0   
10  5ae8a2d920e0                False    111  0.576577           0   
11  c47d1b0affa6                False     25  0.640000           0   
12  3093e9a1e018                False      1  1.000000           0   
2   5430416cbea4                False     58  0.517241           0   
4   2d8ce01487f9                False     15

In [9]:
# -----------------------------
# Alakazam-specific model dataset
# -----------------------------

def matchup_label(player_archetype: str, opponent_archetype: str) -> str:
    if player_archetype == 'Alakazam' and opponent_archetype == 'Hop_Trevenant':
        return 'Alakazam_vs_Hop_Trevenant'
    if player_archetype == 'Alakazam' and opponent_archetype == 'Alakazam':
        return 'Alakazam_mirror'
    if player_archetype == 'Alakazam' and opponent_archetype == 'Mega_Lucario':
        return 'Alakazam_vs_Mega_Lucario'
    return f'{player_archetype}_vs_{opponent_archetype}'


def prepare_alakazam_option_dataset(options: pd.DataFrame, decisions: pd.DataFrame, states: pd.DataFrame) -> pd.DataFrame:
    if options.empty or decisions.empty:
        return pd.DataFrame()
    dcols = [
        'decision_id', 'episode_id', 'player_index', 'rank', 'rate', 'is_top200',
        'player_archetype', 'opponent_archetype', 'tier', 'matchup', 'context_name',
        'min_count', 'max_count', 'num_options', 'chosen_valid', 'won', 'chosen_count',
        'mc_step_weight', 'phase_weight', 'step_frac',
    ]
    dcols = [c for c in dcols if c in decisions.columns]
    df = options.merge(decisions[dcols], on=['decision_id', 'episode_id', 'player_index', 'context_name'], how='left')
    if not states.empty:
        scols = [c for c in states.columns if c not in {'episode_id', 'player_index'}]
        df = df.merge(states[scols], on='decision_id', how='left')
    df = df[df.get('player_archetype') == 'Alakazam'].copy()
    if 'rank' in df.columns:
        df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
        df = df[(df['rank'].isna()) | (df['rank'] <= TOP_RANK_CUTOFF)].copy()
    if 'chosen_valid' in df.columns:
        df = df[df['chosen_valid'] == True].copy()
    df['matchup_label'] = [matchup_label(a, b) for a, b in zip(df.get('player_archetype', ''), df.get('opponent_archetype', ''))]
    df['context_name'] = df['context_name'].fillna('UNKNOWN')
    df['option_type'] = df['option_type'].fillna('UNKNOWN')
    # Alakazam-specific phase features.
    for c in ['my_hand_count', 'my_deck_count', 'op_active_hp', 'powerful_hand_damage_est', 'my_alakazam_count', 'my_kadabra_count', 'my_abra_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    if 'powerful_hand_damage_est' in df.columns and 'op_active_hp' in df.columns:
        df['powerful_hand_can_ko_active'] = (df['powerful_hand_damage_est'] >= df['op_active_hp']).astype(float)
    else:
        df['powerful_hand_can_ko_active'] = 0.0
    if 'my_deck_count' in df.columns:
        df['deckout_risk_feature'] = (df['my_deck_count'].fillna(99) <= 4).astype(float)
    else:
        df['deckout_risk_feature'] = 0.0
    return df

ALAKAZAM_OPTION_MODEL_DF = prepare_alakazam_option_dataset(OPTION_ROWS_DF, DECISION_ROWS_DF, STATE_SUMMARY_DF)
ARTIFACT_PATHS['alakazam_option_model_df'] = safe_save_table(ALAKAZAM_OPTION_MODEL_DF, OUTPUT_DIR / 'alakazam_option_model_df')
print('alakazam option model df:', ALAKAZAM_OPTION_MODEL_DF.shape)
if not ALAKAZAM_OPTION_MODEL_DF.empty:
    print(ALAKAZAM_OPTION_MODEL_DF[['context_name', 'option_type', 'is_chosen', 'matchup_label']].head())


alakazam option model df: (524472, 66)
               context_name option_type  is_chosen    matchup_label
19461  SETUP_ACTIVE_POKEMON      EVOLVE          1  Alakazam_mirror
19462  SETUP_ACTIVE_POKEMON      EVOLVE          0  Alakazam_mirror
19463  SETUP_ACTIVE_POKEMON      EVOLVE          0  Alakazam_mirror
19464  SETUP_ACTIVE_POKEMON      EVOLVE          1  Alakazam_mirror
19469  SETUP_ACTIVE_POKEMON      EVOLVE          1  Alakazam_mirror


In [10]:
# -----------------------------
# UNKNOWN / numeric context diagnostics
# -----------------------------

def _top_counter_json(s: pd.Series, n: int = 20) -> str:
    vals = []
    for x in s.dropna().tolist():
        if isinstance(x, float) and math.isnan(x):
            continue
        try:
            vals.append(int(x))
        except Exception:
            vals.append(str(x))
    return json.dumps(Counter(vals).most_common(n), ensure_ascii=False)


def build_unknown_context_diagnostics(df: pd.DataFrame, max_examples: int = 300) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if df.empty or 'context_name' not in df.columns:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    work = df.copy()
    ctx = work['context_name'].astype(str)
    unknown = ctx.str.startswith('UNKNOWN') | ctx.str.fullmatch(r'\d+') | ctx.isin(['30', '37', '38', '41', '43'])
    work = work[unknown].copy()
    if work.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    rows = []
    deep_rows = []
    for ctx_name, g in work.groupby('context_name', dropna=False):
        dec = g.drop_duplicates('decision_id') if 'decision_id' in g else g
        chosen = g[pd.to_numeric(g.get('is_chosen'), errors='coerce').fillna(0).astype(int) == 1] if 'is_chosen' in g else pd.DataFrame()
        option_type_top = Counter(g.get('option_type', pd.Series(dtype=object)).astype(str)).most_common(12)
        chosen_type_top = Counter(chosen.get('option_type', pd.Series(dtype=object)).astype(str)).most_common(12) if not chosen.empty else []
        rows.append({
            'context_name': str(ctx_name),
            'option_rows': int(len(g)),
            'decisions': int(dec['decision_id'].nunique()) if 'decision_id' in dec else int(len(dec)),
            'min_count_mean': float(pd.to_numeric(dec.get('min_count'), errors='coerce').mean()) if 'min_count' in dec else None,
            'max_count_mean': float(pd.to_numeric(dec.get('max_count'), errors='coerce').mean()) if 'max_count' in dec else None,
            'chosen_count_mean': float(pd.to_numeric(dec.get('chosen_count'), errors='coerce').mean()) if 'chosen_count' in dec else None,
            'full_rate': float((pd.to_numeric(dec.get('chosen_count'), errors='coerce') >= pd.to_numeric(dec.get('max_count'), errors='coerce')).mean()) if {'chosen_count','max_count'}.issubset(dec.columns) else None,
            'option_type_top12': json.dumps(option_type_top, ensure_ascii=False),
            'chosen_option_type_top12': json.dumps(chosen_type_top, ensure_ascii=False),
            'card_id_top20': _top_counter_json(g.get('card_id', pd.Series(dtype=object)), 20) if 'card_id' in g else '[]',
            'chosen_card_id_top20': _top_counter_json(chosen.get('card_id', pd.Series(dtype=object)), 20) if 'card_id' in chosen else '[]',
            'attack_id_top10': _top_counter_json(g.get('attack_id', pd.Series(dtype=object)), 10) if 'attack_id' in g else '[]',
            'number_value_top10': _top_counter_json(g.get('number_value', pd.Series(dtype=object)), 10) if 'number_value' in g else '[]',
        })
        example_decisions = g['decision_id'].drop_duplicates().head(5).tolist() if 'decision_id' in g else []
        for did in example_decisions:
            ex = g[g['decision_id'] == did].head(25).copy()
            keep = [c for c in ['decision_id','episode_id','player_index','context_name','option_index','option_type','card_id','target_card_id','attack_id','number_value','area','in_play_area','is_chosen','min_count','max_count','chosen_count','matchup_label'] if c in ex.columns]
            deep_rows.append({
                'context_name': str(ctx_name),
                'decision_id': did,
                'example_options_json': ex[keep].to_json(orient='records', force_ascii=False) if keep else ex.to_json(orient='records', force_ascii=False),
            })
    report = pd.DataFrame(rows).sort_values(['option_rows', 'decisions'], ascending=False)
    example_cols = [c for c in ['decision_id', 'episode_id', 'player_index', 'context_name', 'option_index', 'option_type', 'card_id', 'target_card_id', 'attack_id', 'number_value', 'area', 'in_play_area', 'is_chosen', 'min_count', 'max_count', 'chosen_count', 'matchup_label'] if c in work.columns]
    examples = work[example_cols].head(max_examples).copy() if example_cols else work.head(max_examples).copy()
    deep = pd.DataFrame(deep_rows)
    return report, examples, deep

UNKNOWN_CONTEXT_REPORT_DF, UNKNOWN_CONTEXT_EXAMPLES_DF, UNKNOWN_CONTEXT_DEEP_DIVE_DF = build_unknown_context_diagnostics(ALAKAZAM_OPTION_MODEL_DF)
ARTIFACT_PATHS['unknown_context_report'] = safe_save_table(UNKNOWN_CONTEXT_REPORT_DF, OUTPUT_DIR / 'unknown_context_report')
ARTIFACT_PATHS['unknown_context_examples'] = safe_save_table(UNKNOWN_CONTEXT_EXAMPLES_DF, OUTPUT_DIR / 'unknown_context_examples')
ARTIFACT_PATHS['unknown_context_deep_dive'] = safe_save_table(UNKNOWN_CONTEXT_DEEP_DIVE_DF, OUTPUT_DIR / 'unknown_context_deep_dive')
print('unknown context report:')
display(UNKNOWN_CONTEXT_REPORT_DF.head(20)) if 'display' in globals() else print(UNKNOWN_CONTEXT_REPORT_DF.head(20))
if not UNKNOWN_CONTEXT_DEEP_DIVE_DF.empty:
    print('unknown context deep-dive examples:')
    display(UNKNOWN_CONTEXT_DEEP_DIVE_DF.head(20)) if 'display' in globals() else print(UNKNOWN_CONTEXT_DEEP_DIVE_DF.head(20))


unknown context report:
   context_name  option_rows  decisions  min_count_mean  max_count_mean  \
11    UNKNOWN_0       441649      38159             1.0             1.0   
10           43         6156       3078             1.0             1.0   
7            37         1581        516             1.0             1.0   
5            30          538        366             1.0             1.0   
9            41          286        143             1.0             1.0   
8            38          232         93             1.0             1.0   
1            21          129         70             1.0             1.0   
0            15            3          1             1.0             1.0   
4            27            2          2             1.0             1.0   
2            23            2          1             1.0             1.0   
6            34            2          1             2.0             2.0   
3            26            1          1             1.0             1.0   



In [11]:
# -----------------------------
# Alakazam BC / weighted BC training
# -----------------------------
ALAKAZAM_BC_MODELS = {}
ALAKAZAM_WEIGHTED_BC_MODELS = {}
ALAKAZAM_MODEL_REPORT_ROWS = []
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = [
    'option_index', 'num_options', 'min_count', 'max_count', 'chosen_count',
    'rank', 'rate', 'card_id', 'target_card_id', 'attack_id', 'number_value',
    'in_play_index', 'remain_damage_counter', 'remain_energy_cost',
    'turn', 'turn_action_count', 'my_active_id', 'my_active_hp', 'my_active_energy_count',
    'op_active_id', 'op_active_hp', 'op_active_energy_count', 'my_bench_count', 'op_bench_count',
    'my_alakazam_count', 'my_kadabra_count', 'my_abra_count', 'my_dudunsparce_count',
    'my_hand_count', 'op_hand_count', 'my_deck_count', 'op_deck_count',
    'my_prizes_left', 'op_prizes_left', 'stadium_id', 'powerful_hand_damage_est',
    'powerful_hand_can_ko_active', 'deckout_risk_feature'
]
CATEGORICAL_FEATURES = [
    'context_name', 'option_type', 'area', 'in_play_area', 'tier', 'matchup_label', 'opponent_archetype'
]


def model_input(df: pd.DataFrame, numeric_features=NUMERIC_FEATURES, categorical_features=CATEGORICAL_FEATURES):
    num_cols = [c for c in numeric_features if c in df.columns]
    cat_cols = [c for c in categorical_features if c in df.columns]
    X = pd.DataFrame(index=df.index)
    for c in num_cols:
        X[c] = pd.to_numeric(df[c], errors='coerce')
    if np is not None and len(num_cols):
        X[num_cols] = X[num_cols].replace([np.inf, -np.inf], np.nan)
    for c in cat_cols:
        X[c] = df[c].astype('string').fillna('UNK')
    return X, num_cols, cat_cols


def sample_option_rows(df: pd.DataFrame, max_rows: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.copy()
    # Preserve positives and matchup/context diversity.
    parts = []
    rng = random.Random(seed)
    group_cols = ['context_name', 'is_chosen'] if 'context_name' in df.columns else ['is_chosen']
    n_groups = max(1, df.groupby(group_cols, dropna=False).ngroups)
    per_group = max(1000, max_rows // n_groups)
    for _, g in df.groupby(group_cols, dropna=False):
        take = min(len(g), per_group)
        parts.append(g.sample(n=take, random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > max_rows:
        out = out.sample(n=max_rows, random_state=seed).reset_index(drop=True)
    return out


def make_bc_weights(df: pd.DataFrame) -> pd.Series:
    w = pd.Series(1.0, index=df.index, dtype=float)
    if 'rank' in df.columns:
        rank = pd.to_numeric(df['rank'], errors='coerce')
        w *= np.where(rank <= 10, 4.0, np.where(rank <= 50, 2.0, 1.0)) if np is not None else 1.0
    if 'won' in df.columns:
        won = pd.to_numeric(df['won'], errors='coerce').fillna(0)
        w *= np.where(won > 0, 1.5, 1.0) if np is not None else 1.0
    if 'matchup_label' in df.columns:
        w *= df['matchup_label'].isin(PRIMARY_MATCHUPS).map(lambda x: 1.5 if x else 1.0).astype(float)
    return w.fillna(1.0)


def grouped_option_metrics(df_valid: pd.DataFrame, prob: Any) -> dict:
    tmp = df_valid[['decision_id', 'option_index', 'is_chosen']].copy()
    tmp['prob'] = prob
    ranks = []
    top1 = []
    top3 = []
    n_dec = 0
    for _, g in tmp.groupby('decision_id'):
        if g['is_chosen'].sum() <= 0:
            continue
        g = g.sort_values('prob', ascending=False).reset_index(drop=True)
        chosen_positions = [i + 1 for i, v in enumerate(g['is_chosen'].tolist()) if v == 1]
        if not chosen_positions:
            continue
        n_dec += 1
        r = min(chosen_positions)
        ranks.append(r)
        top1.append(1 if r <= 1 else 0)
        top3.append(1 if r <= 3 else 0)
    if n_dec == 0:
        return {'eval_decisions': 0, 'top1_acc': None, 'top3_acc': None, 'mean_chosen_rank': None}
    return {
        'eval_decisions': int(n_dec),
        'top1_acc': float(sum(top1) / n_dec),
        'top3_acc': float(sum(top3) / n_dec),
        'mean_chosen_rank': float(sum(ranks) / len(ranks)),
    }


def train_option_classifier(df: pd.DataFrame, context: str | None = None, weighted: bool = False):
    if df.empty:
        return None, {'context': context or 'ALL', 'status': 'empty'}
    work = df.copy()
    if context is not None:
        work = work[work['context_name'] == context].copy()
    if work.empty or work['is_chosen'].nunique() < 2 or work['decision_id'].nunique() < 50:
        return None, {'context': context or 'ALL', 'status': 'insufficient', 'rows': int(len(work))}
    max_rows = MAX_CONTEXT_TRAIN_ROWS if context is not None else MAX_GLOBAL_TRAIN_ROWS
    work = sample_option_rows(work, max_rows=max_rows)
    try:
        from sklearn.compose import ColumnTransformer
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import log_loss, roc_auc_score
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
    except Exception as exc:
        return None, {'context': context or 'ALL', 'status': 'sklearn_missing', 'error': repr(exc)}

    # Split by decision_id to avoid leaking options from the same decision into both sides.
    decision_ids = work['decision_id'].drop_duplicates()
    train_ids, valid_ids = train_test_split(decision_ids, test_size=0.15, random_state=RANDOM_SEED)
    train = work[work['decision_id'].isin(train_ids)].copy()
    valid = work[work['decision_id'].isin(valid_ids)].copy()
    if train['is_chosen'].nunique() < 2 or valid.empty:
        return None, {'context': context or 'ALL', 'status': 'bad_split', 'rows': int(len(work))}
    X_train, num_cols, cat_cols = model_input(train)
    X_valid, _, _ = model_input(valid, numeric_features=num_cols, categorical_features=cat_cols)
    y_train = train['is_chosen'].astype(int)
    y_valid = valid['is_chosen'].astype(int)
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0.0)), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='UNK')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), cat_cols),
    ], remainder='drop')
    clf = LogisticRegression(max_iter=700, class_weight='balanced', solver='lbfgs')
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    sample_weight = make_bc_weights(train) if weighted else None
    try:
        if sample_weight is not None:
            pipe.fit(X_train, y_train, clf__sample_weight=sample_weight)
        else:
            pipe.fit(X_train, y_train)
    except TypeError:
        pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_valid)[:, 1]
    metrics = grouped_option_metrics(valid, prob)
    metrics.update({
        'context': context or 'ALL',
        'weighted': bool(weighted),
        'status': 'ok',
        'rows': int(len(work)),
        'train_rows': int(len(train)),
        'valid_rows': int(len(valid)),
        'positive_rate': float(work['is_chosen'].mean()),
    })
    try:
        metrics['auc'] = float(roc_auc_score(y_valid, prob)) if y_valid.nunique() == 2 else None
        metrics['logloss'] = float(log_loss(y_valid, prob, labels=[0, 1]))
    except Exception:
        metrics['auc'] = None
        metrics['logloss'] = None
    bundle = {'model': pipe, 'numeric_features': num_cols, 'categorical_features': cat_cols, 'context': context, 'weighted': weighted, 'metrics': metrics}
    return bundle, metrics

if TRAIN_MODE and RUN_ALAKAZAM_BC and not ALAKAZAM_OPTION_MODEL_DF.empty:
    contexts = ALAKAZAM_OPTION_MODEL_DF['context_name'].value_counts().head(14).index.tolist()
    for ctx in contexts:
        for weighted in [False, True]:
            try:
                bundle, metrics = train_option_classifier(ALAKAZAM_OPTION_MODEL_DF, context=ctx, weighted=weighted)
                ALAKAZAM_MODEL_REPORT_ROWS.append({'model': 'weighted_bc' if weighted else 'bc', **metrics})
                if bundle is not None:
                    target = ALAKAZAM_WEIGHTED_BC_MODELS if weighted else ALAKAZAM_BC_MODELS
                    target[ctx] = bundle
            except Exception as exc:
                ALAKAZAM_MODEL_REPORT_ROWS.append({'model': 'weighted_bc' if weighted else 'bc', 'context': ctx, 'status': 'error', 'error': repr(exc)})
    # Global model is optional; never stop notebook on global failure.
    try:
        bundle, metrics = train_option_classifier(ALAKAZAM_OPTION_MODEL_DF, context=None, weighted=False)
        ALAKAZAM_MODEL_REPORT_ROWS.append({'model': 'bc', **metrics})
        if bundle is not None:
            ALAKAZAM_BC_MODELS['ALL'] = bundle
    except Exception as exc:
        ALAKAZAM_MODEL_REPORT_ROWS.append({'model': 'bc', 'context': 'ALL', 'status': 'error', 'error': repr(exc)})
else:
    print('BC training skipped')

ALAKAZAM_MODEL_REPORT_DF = pd.DataFrame(ALAKAZAM_MODEL_REPORT_ROWS)
ARTIFACT_PATHS['alakazam_model_report'] = safe_save_table(ALAKAZAM_MODEL_REPORT_DF, OUTPUT_DIR / 'alakazam_model_report')
if ALAKAZAM_BC_MODELS:
    with open(MODEL_DIR / f'{RUN_PREFIX}-alakazam_bc_models.pkl', 'wb') as f:
        pickle.dump(ALAKAZAM_BC_MODELS, f)
    ARTIFACT_PATHS['alakazam_bc_models'] = str(MODEL_DIR / f'{RUN_PREFIX}-alakazam_bc_models.pkl')
if ALAKAZAM_WEIGHTED_BC_MODELS:
    with open(MODEL_DIR / f'{RUN_PREFIX}-alakazam_weighted_bc_models.pkl', 'wb') as f:
        pickle.dump(ALAKAZAM_WEIGHTED_BC_MODELS, f)
    ARTIFACT_PATHS['alakazam_weighted_bc_models'] = str(MODEL_DIR / f'{RUN_PREFIX}-alakazam_weighted_bc_models.pkl')
print(ALAKAZAM_MODEL_REPORT_DF.head(30))


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:572: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:572: FutureWarning: Curr

          model  eval_decisions  top1_acc  top3_acc  mean_chosen_rank  \
0            bc            5724  0.571803  0.842418          2.174179   
1   weighted_bc            5724  0.577568  0.843641          2.168064   
2            bc            1753  0.640616  0.874501          1.811751   
3   weighted_bc            1753  0.640616  0.874501          1.811751   
4            bc             462  0.774892  1.000000          1.225108   
5   weighted_bc             462  0.774892  1.000000          1.225108   
6            bc             233  0.643777  0.909871          1.669528   
7   weighted_bc             233  0.643777  0.909871          1.669528   
8            bc             184  0.467391  0.902174          1.902174   
9   weighted_bc             184  0.467391  0.902174          1.902174   
10           bc             210  0.923810  0.995238          1.100000   
11  weighted_bc             210  0.923810  0.995238          1.100000   
12           bc              57  0.947368  1.000000

In [12]:
# -----------------------------
# Alakazam-specific value model and phase diagnostics
# -----------------------------
ALAKAZAM_VALUE_MODEL = None
ALAKAZAM_VALUE_REPORT = {}

VALUE_NUMERIC_FEATURES = [
    'turn', 'turn_action_count', 'my_active_id', 'my_active_hp', 'my_active_energy_count',
    'op_active_id', 'op_active_hp', 'op_active_energy_count', 'my_bench_count', 'op_bench_count',
    'my_alakazam_count', 'my_kadabra_count', 'my_abra_count', 'my_dudunsparce_count',
    'my_hand_count', 'op_hand_count', 'my_deck_count', 'op_deck_count', 'my_prizes_left', 'op_prizes_left',
    'stadium_id', 'powerful_hand_damage_est', 'deckout_risk'
]
VALUE_CATEGORICAL_FEATURES = ['context_name', 'tier', 'matchup_label', 'opponent_archetype']


def prepare_value_dataset(option_model_df: pd.DataFrame) -> pd.DataFrame:
    if option_model_df.empty:
        return pd.DataFrame()
    cols = ['decision_id', 'won'] + [c for c in VALUE_NUMERIC_FEATURES + VALUE_CATEGORICAL_FEATURES if c in option_model_df.columns]
    df = option_model_df[cols].drop_duplicates('decision_id').copy()
    df['won'] = pd.to_numeric(df['won'], errors='coerce')
    df = df[df['won'].isin([0, 1])].copy()
    if 'powerful_hand_damage_est' in df.columns and 'op_active_hp' in df.columns:
        df['powerful_hand_can_ko_active'] = (pd.to_numeric(df['powerful_hand_damage_est'], errors='coerce') >= pd.to_numeric(df['op_active_hp'], errors='coerce')).astype(float)
    if 'my_deck_count' in df.columns:
        df['deckout_risk_feature'] = (pd.to_numeric(df['my_deck_count'], errors='coerce').fillna(99) <= 4).astype(float)
    return df


def train_value_model(df: pd.DataFrame):
    if df.empty or df['won'].nunique() < 2 or len(df) < 200:
        return None, {'status': 'insufficient', 'rows': int(len(df))}
    try:
        from sklearn.compose import ColumnTransformer
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import log_loss, roc_auc_score
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
    except Exception as exc:
        return None, {'status': 'sklearn_missing', 'error': repr(exc)}
    work = df.sample(n=min(len(df), MAX_GLOBAL_TRAIN_ROWS), random_state=RANDOM_SEED).copy()
    train, valid = train_test_split(work, test_size=0.15, random_state=RANDOM_SEED, stratify=work['won'] if work['won'].nunique() == 2 else None)
    X_train, num_cols, cat_cols = model_input(train, numeric_features=VALUE_NUMERIC_FEATURES + ['powerful_hand_can_ko_active', 'deckout_risk_feature'], categorical_features=VALUE_CATEGORICAL_FEATURES)
    X_valid, _, _ = model_input(valid, numeric_features=num_cols, categorical_features=cat_cols)
    y_train = train['won'].astype(int)
    y_valid = valid['won'].astype(int)
    pre = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0.0)), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='UNK')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))]), cat_cols),
    ], remainder='drop')
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=700, class_weight='balanced'))])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_valid)[:, 1]
    metrics = {
        'status': 'ok',
        'rows': int(len(work)),
        'train_rows': int(len(train)),
        'valid_rows': int(len(valid)),
        'auc': float(roc_auc_score(y_valid, prob)) if y_valid.nunique() == 2 else None,
        'logloss': float(log_loss(y_valid, prob, labels=[0, 1])),
    }
    return {'model': pipe, 'numeric_features': num_cols, 'categorical_features': cat_cols, 'metrics': metrics}, metrics

VALUE_DATASET_DF = prepare_value_dataset(ALAKAZAM_OPTION_MODEL_DF)
ARTIFACT_PATHS['alakazam_value_dataset'] = safe_save_table(VALUE_DATASET_DF, OUTPUT_DIR / 'alakazam_value_dataset')
if TRAIN_MODE and RUN_ALAKAZAM_VALUE:
    try:
        ALAKAZAM_VALUE_MODEL, ALAKAZAM_VALUE_REPORT = train_value_model(VALUE_DATASET_DF)
        if ALAKAZAM_VALUE_MODEL is not None:
            with open(MODEL_DIR / f'{RUN_PREFIX}-alakazam_value_model.pkl', 'wb') as f:
                pickle.dump(ALAKAZAM_VALUE_MODEL, f)
            ARTIFACT_PATHS['alakazam_value_model'] = str(MODEL_DIR / f'{RUN_PREFIX}-alakazam_value_model.pkl')
    except Exception as exc:
        ALAKAZAM_VALUE_REPORT = {'status': 'error', 'error': repr(exc), 'traceback': traceback.format_exc(limit=5)}
print('value report:', ALAKAZAM_VALUE_REPORT)

# Phase diagnostics for human rule work.
PHASE_DIAGNOSTIC_ROWS = []
if not VALUE_DATASET_DF.empty:
    d = VALUE_DATASET_DF.copy()
    for label, q in [
        ('has_alakazam', d.get('my_alakazam_count', 0).fillna(0) >= 1 if 'my_alakazam_count' in d else pd.Series(False, index=d.index)),
        ('powerful_hand_can_ko_active', d.get('powerful_hand_can_ko_active', 0).fillna(0) >= 1 if 'powerful_hand_can_ko_active' in d else pd.Series(False, index=d.index)),
        ('deckout_risk', d.get('deckout_risk_feature', 0).fillna(0) >= 1 if 'deckout_risk_feature' in d else pd.Series(False, index=d.index)),
    ]:
        sub = d[q]
        if len(sub):
            PHASE_DIAGNOSTIC_ROWS.append({'phase_signal': label, 'rows': int(len(sub)), 'winrate': float(sub['won'].mean())})
PHASE_DIAGNOSTIC_DF = pd.DataFrame(PHASE_DIAGNOSTIC_ROWS)
ARTIFACT_PATHS['alakazam_phase_diagnostics'] = safe_save_table(PHASE_DIAGNOSTIC_DF, OUTPUT_DIR / 'alakazam_phase_diagnostics')
print(PHASE_DIAGNOSTIC_DF)


value report: {'status': 'ok', 'rows': 60483, 'train_rows': 51410, 'valid_rows': 9073, 'auc': 0.7452168739081948, 'logloss': 0.592318079998845}
                  phase_signal   rows   winrate
0                 has_alakazam  36087  0.585613
1  powerful_hand_can_ko_active  42348  0.533603
2                 deckout_risk   3473  0.519148


In [13]:
# ── v05d16: Delta-V N-step lookahead + mc_step_weight update ────────────────
N_STEP = N_STEP_LOOKAHEAD  # default=3
DELTA_V_REPORT = {'status': 'not_run'}

if ALAKAZAM_VALUE_MODEL is not None and not ALAKAZAM_OPTION_MODEL_DF.empty:
    try:
        _val_mdl = ALAKAZAM_VALUE_MODEL['model']
        _vnum = ALAKAZAM_VALUE_MODEL.get('numeric_features', VALUE_NUMERIC_FEATURES)
        _vcat = ALAKAZAM_VALUE_MODEL.get('categorical_features', VALUE_CATEGORICAL_FEATURES)
        _dec = ALAKAZAM_OPTION_MODEL_DF.drop_duplicates('decision_id').copy()
        _avail = [c for c in _vnum + _vcat if c in _dec.columns]
        _dec['v_t'] = _val_mdl.predict_proba(_dec[_avail])[:, 1]
        if 'step' in _dec.columns:
            _dec = _dec.sort_values(['episode_id', 'player_index', 'step'])
            _dec['v_t_plus_N'] = _dec.groupby(['episode_id', 'player_index'])['v_t'].shift(-N_STEP)
            _dec['delta_v'] = _dec['v_t_plus_N'] - _dec['v_t']
        else:
            _dec['delta_v'] = float('nan')
        _dec['delta_v_weight'] = _dec['delta_v'].clip(lower=0).fillna(0.0)
        _pw_col = 'phase_weight' if 'phase_weight' in _dec.columns else 'mc_step_weight'
        _dec['combined_weight'] = _dec.get(_pw_col, pd.Series(0.5, index=_dec.index)).fillna(0.5) * _dec['delta_v_weight']
        _wmap = _dec.set_index('decision_id')['combined_weight'].to_dict()
        DECISION_ROWS_DF['mc_step_weight'] = DECISION_ROWS_DF['decision_id'].map(_wmap).fillna(0.0)
        ALAKAZAM_OPTION_MODEL_DF['mc_step_weight'] = ALAKAZAM_OPTION_MODEL_DF['decision_id'].map(_wmap).fillna(0.0)
        _n_pos = (_dec['combined_weight'] > 0).sum()
        _dv_pos = (_dec['delta_v'].fillna(-1) > 0).mean()
        DELTA_V_REPORT = {
            'status': 'ok', 'N_step': N_STEP,
            'n_decisions': int(len(_dec)),
            'positive_weight': int(_n_pos),
            'delta_v_mean': float(_dec['delta_v'].mean()),
            'delta_v_positive_pct': float(_dv_pos),
        }
        print(f'Delta-V (N={N_STEP}): decisions={len(_dec)}, positive_dv={100*_dv_pos:.1f}%, weight>0={_n_pos}/{len(_dec)}')
        print(f'  delta_v: mean={_dec["delta_v"].mean():.4f} std={_dec["delta_v"].std():.4f}')
        with open(OUTPUT_DIR / 'delta_v_report.json', 'w') as _f:
            json.dump(DELTA_V_REPORT, _f, indent=2)
        ARTIFACT_PATHS['delta_v_report'] = str(OUTPUT_DIR / 'delta_v_report.json')
    except Exception as _exc:
        import traceback as _tb
        DELTA_V_REPORT = {'status': 'error', 'error': repr(_exc)}
        print(f'Delta-V computation failed: {_exc}')
        print(_tb.format_exc(limit=4))
else:
    print('Delta-V skipped (no value model or empty ALAKAZAM_OPTION_MODEL_DF)')
    DELTA_V_REPORT = {'status': 'skipped', 'reason': 'no_value_model_or_empty_data'}
print('DELTA_V_REPORT:', DELTA_V_REPORT.get('status'))


Delta-V (N=3): decisions=60483, positive_dv=50.5%, weight>0=30567/60483
  delta_v: mean=0.0010 std=0.1003
DELTA_V_REPORT: ok


In [14]:
# -----------------------------
# Rule-gap diagnostics for Alakazam control
# -----------------------------
ALAKAZAM_PATCH_CANDIDATES = {'source': EXPERIMENT_NAME, 'candidates': []}

if RUN_RULE_GAP_DIAGNOSTICS and not ALAKAZAM_OPTION_MODEL_DF.empty:
    df = ALAKAZAM_OPTION_MODEL_DF.copy()
    # 1. Do not always use maxCount.
    if {'decision_id', 'chosen_count', 'max_count', 'context_name'}.issubset(df.columns):
        dec = df.drop_duplicates('decision_id').copy()
        dec['chosen_ratio'] = pd.to_numeric(dec['chosen_count'], errors='coerce') / pd.to_numeric(dec['max_count'], errors='coerce').replace(0, np.nan)
        for ctx, g in dec.groupby('context_name'):
            if len(g) < 100:
                continue
            full_rate = float((g['chosen_count'] >= g['max_count']).mean())
            if full_rate < 0.80:
                ALAKAZAM_PATCH_CANDIDATES['candidates'].append({
                    'priority': 'high',
                    'theme': 'select_fewer_than_maxCount',
                    'context': str(ctx),
                    'decisions': int(len(g)),
                    'full_rate': full_rate,
                    'avg_chosen': float(pd.to_numeric(g['chosen_count'], errors='coerce').mean()),
                    'avg_max': float(pd.to_numeric(g['max_count'], errors='coerce').mean()),
                    'recommendation': 'Use positive-score threshold; do not blindly return maxCount.'
                })
    # 2. Control card timing.
    for card_id, theme in [(Enhanced_Hammer, 'enhanced_hammer_timing'), (Nighttime_Mine, 'nighttime_mine_timing'), (Boss_Orders, 'boss_for_ko'), (Rare_Candy, 'rare_candy_timing')]:
        sub = df[pd.to_numeric(df.get('card_id'), errors='coerce') == card_id] if 'card_id' in df else pd.DataFrame()
        if not sub.empty:
            ALAKAZAM_PATCH_CANDIDATES['candidates'].append({
                'priority': 'medium',
                'theme': theme,
                'card_id': int(card_id),
                'option_rows': int(len(sub)),
                'chosen_rate': float(pd.to_numeric(sub['is_chosen'], errors='coerce').mean()),
                'recommendation': 'Compare chosen contexts by matchup and phase; encode high-confidence timing into rule scores.'
            })

ARTIFACT_PATHS['alakazam_patch_candidates'] = write_json(OUTPUT_DIR / 'alakazam_patch_candidates.json', ALAKAZAM_PATCH_CANDIDATES)
print(json.dumps(ALAKAZAM_PATCH_CANDIDATES, ensure_ascii=False, indent=2)[:4000])


{
  "source": "v0_05d25_lgbm_winner_wt4",
  "candidates": [
    {
      "priority": "high",
      "theme": "select_fewer_than_maxCount",
      "context": "MAIN",
      "decisions": 1468,
      "full_rate": 0.19754768392370572,
      "avg_chosen": 0.2656675749318801,
      "avg_max": 1.2881471389645776,
      "recommendation": "Use positive-score threshold; do not blindly return maxCount."
    },
    {
      "priority": "high",
      "theme": "select_fewer_than_maxCount",
      "context": "TO_ACTIVE",
      "decisions": 251,
      "full_rate": 0.06374501992031872,
      "avg_chosen": 1.0,
      "avg_max": 3.549800796812749,
      "recommendation": "Use positive-score threshold; do not blindly return maxCount."
    },
    {
      "priority": "high",
      "theme": "select_fewer_than_maxCount",
      "context": "TO_HAND",
      "decisions": 1285,
      "full_rate": 0.19455252918287938,
      "avg_chosen": 0.9712062256809338,
      "avg_max": 1.7766536964980544,
      "recommendation": "Us

In [15]:
# -----------------------------
# v0-05b complete-baseline review tables
# -----------------------------

def build_candidate_deck_review(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    for col in ['games','top10_uses','top50_uses','vs_hop_games','mirror_games','vs_lucario_games']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0).astype(int)
    for col in ['winrate','vs_hop_winrate','mirror_winrate','vs_lucario_winrate','avg_rank','review_score']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    if 'risk_flags' not in out.columns:
        out['risk_flags'] = '[]'
    # A manual-review score: conservative, sample-aware, Hop/mirror-heavy.
    def _num_col(name, default=0.0):
        if name in out.columns:
            return pd.to_numeric(out[name], errors='coerce').fillna(default)
        return pd.Series(default, index=out.index)
    out['complete_baseline_score'] = (
        _num_col('top10_uses') * 0.25 +
        _num_col('top50_uses') * 0.05 +
        _num_col('games').clip(upper=600) * 0.002 +
        _num_col('vs_hop_winrate', 0.5) * 2.5 +
        _num_col('mirror_winrate', 0.5) * 2.5 -
        (_num_col('vs_hop_games') < 20).astype(int) * 0.50 -
        (_num_col('mirror_games') < 20).astype(int) * 0.50
    )
    return out.sort_values(['complete_baseline_score','review_score','games'], ascending=False)


def build_context_threshold_recommendations(report: pd.DataFrame) -> pd.DataFrame:
    if report is None or report.empty:
        return pd.DataFrame()
    rows = []
    for _, r in report.iterrows():
        ctx = str(r.get('context_name'))
        full_rate = r.get('full_rate')
        chosen_mean = r.get('chosen_count_mean')
        max_mean = r.get('max_count_mean')
        if pd.isna(full_rate):
            conservatism = 'unknown'
        elif full_rate < 0.25:
            conservatism = 'strong_threshold'
        elif full_rate < 0.60:
            conservatism = 'medium_threshold'
        else:
            conservatism = 'light_threshold'
        rows.append({
            'context_name': ctx,
            'observed_full_rate': full_rate,
            'observed_chosen_count_mean': chosen_mean,
            'observed_max_count_mean': max_mean,
            'recommended_selection_policy': conservatism,
            'note': 'Use diagnostics to tune rule thresholds; do not enable neural submission from this table alone.'
        })
    return pd.DataFrame(rows)

CANDIDATE_DECK_REVIEW_DF = build_candidate_deck_review(ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF)
CONTEXT_THRESHOLD_RECOMMENDATIONS_DF = build_context_threshold_recommendations(UNKNOWN_CONTEXT_REPORT_DF)
ARTIFACT_PATHS['candidate_deck_review'] = safe_save_table(CANDIDATE_DECK_REVIEW_DF, OUTPUT_DIR / 'candidate_deck_review')
ARTIFACT_PATHS['context_threshold_recommendations'] = safe_save_table(CONTEXT_THRESHOLD_RECOMMENDATIONS_DF, OUTPUT_DIR / 'context_threshold_recommendations')
print('candidate deck review:')
display(CANDIDATE_DECK_REVIEW_DF.head(12)) if 'display' in globals() else print(CANDIDATE_DECK_REVIEW_DF.head(12))
print('context threshold recommendations from unknown/numeric contexts:')
display(CONTEXT_THRESHOLD_RECOMMENDATIONS_DF.head(20)) if 'display' in globals() else print(CONTEXT_THRESHOLD_RECOMMENDATIONS_DF.head(20))


candidate deck review:
       deck_hash  is_default_fallback  games   winrate  top10_uses  \
5   8297d036a5ee                False     33  0.636364          28   
1   2420d2c8ce4b                False    725  0.522759           0   
6   96275182a9da                False    107  0.588785           0   
0   e702674c1864                 True    138  0.500000          10   
10  5ae8a2d920e0                False    111  0.576577           0   
9   82d5654c2507                False     38  0.605263           0   
7   b83fa38ef906                False     39  0.666667           0   
8   c9999e6386ea                False     36  0.666667           0   
11  c47d1b0affa6                False     25  0.640000           0   
12  3093e9a1e018                False      1  1.000000           0   
2   5430416cbea4                False     58  0.517241           0   
4   2d8ce01487f9                False     15  0.400000           0   

    top50_uses    avg_rank  selection_score  vs_hop_games  vs_hop_

## v0-05c: Offline PyTorch MLP reranker and disagreement diagnostics

This stage is intentionally placed after replay mining, deck review, BC/value training, and rule-gap diagnostics, but before final submission generation. The MLP is trained and evaluated offline. The default final submission remains rule-only unless the neural assist flags are explicitly changed and pass safety checks.


In [16]:

# -----------------------------
# v0-05c: B2 diagnostics + PyTorch MLP offline option reranker
# -----------------------------
MLP_RERANKER_REPORT = {'status': 'not_run'}
MLP_CONTEXT_REPORT_DF = pd.DataFrame()
MLP_DECISION_REPORT_DF = pd.DataFrame()
MLP_RULE_PROXY_REPORT_DF = pd.DataFrame()
MLP_BC_COMPARISON_REPORT_DF = pd.DataFrame()
MLP_DISAGREEMENT_EXAMPLES_DF = pd.DataFrame()
UNKNOWN0_ACTION_REPORT_DF = pd.DataFrame()
ERROR_CLASSIFICATION_DF = pd.DataFrame()
MLP_SUBMISSION_SAFETY = {
    'enabled': bool(USE_MLP_RERANKER_FOR_SUBMISSION),
    'contexts': list(MLP_SUBMISSION_CONTEXTS),
    'alpha': float(MLP_ALPHA),
    'torch_in_submission': False,
    'fallback': 'rule_only',
}

# Submission-safe-ish feature set: fields expected to be reconstructible from observation/select/options.
MLP_NUMERIC_FEATURES = [
    'option_index', 'num_options', 'min_count', 'max_count',
    'card_id', 'target_card_id', 'attack_id', 'number_value',
    'in_play_index', 'remain_damage_counter', 'remain_energy_cost',
    'turn', 'turn_action_count', 'my_active_id', 'my_active_hp', 'my_active_energy_count',
    'op_active_id', 'op_active_hp', 'op_active_energy_count', 'my_bench_count', 'op_bench_count',
    'my_alakazam_count', 'my_kadabra_count', 'my_abra_count', 'my_dudunsparce_count',
    'my_hand_count', 'op_hand_count', 'my_deck_count', 'op_deck_count',
    'my_prizes_left', 'op_prizes_left', 'stadium_id', 'powerful_hand_damage_est',
    'powerful_hand_can_ko_active', 'deckout_risk_feature'
]
MLP_CATEGORICAL_FEATURES = [
    'context_name', 'option_type', 'area', 'in_play_area', 'opponent_archetype'
]
MLP_PRIORITY_CONTEXTS = ['UNKNOWN_0', 'TO_HAND', 'TO_BENCH', 'TO_ACTIVE', 'SWITCH', 'MAIN']


def classify_parse_errors(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame(columns=['stage', 'path', 'error', 'classification'])
    out = []
    for r in rows:
        err = str(r.get('error', ''))
        path = str(r.get('path', ''))
        if 'steps' in err or 'KeyError' in err or 'TypeError' in err:
            cls = 'episode_shape_or_non_episode'
        elif path.lower().endswith('.json'):
            cls = 'json_parse_or_schema'
        else:
            cls = 'unknown'
        rr = dict(r)
        rr['classification'] = cls
        out.append(rr)
    return pd.DataFrame(out)


def build_unknown0_action_report(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty or 'context_name' not in df.columns:
        return pd.DataFrame()
    work = df[df['context_name'].astype(str).eq('UNKNOWN_0')].copy()
    if work.empty:
        return pd.DataFrame()
    work['is_chosen_i'] = pd.to_numeric(work.get('is_chosen'), errors='coerce').fillna(0).astype(int)
    if 'option_type' not in work.columns:
        work['option_type'] = 'UNK'
    if 'decision_id' not in work.columns:
        work['decision_id'] = (work.get('episode_id', '').astype(str) + ':' + work.get('step', '').astype(str) + ':' + work.get('player_index', '').astype(str))
    rows = []
    # Coarse option-set signature per decision, useful for YES/NO/END/ATTACK contexts.
    for did, g in work.groupby('decision_id', sort=False):
        chosen = g[g['is_chosen_i'] == 1]
        opt_types = sorted(g['option_type'].astype(str).fillna('UNK').unique().tolist())
        chosen_types = chosen['option_type'].astype(str).fillna('UNK').tolist() if not chosen.empty else []
        rows.append({
            'decision_id': did,
            'episode_id': str(g['episode_id'].iloc[0]) if 'episode_id' in g.columns else '',
            'turn': float(pd.to_numeric(g.get('turn'), errors='coerce').dropna().iloc[0]) if 'turn' in g.columns and pd.to_numeric(g.get('turn'), errors='coerce').notna().any() else None,
            'my_deck_count': float(pd.to_numeric(g.get('my_deck_count'), errors='coerce').dropna().iloc[0]) if 'my_deck_count' in g.columns and pd.to_numeric(g.get('my_deck_count'), errors='coerce').notna().any() else None,
            'my_hand_count': float(pd.to_numeric(g.get('my_hand_count'), errors='coerce').dropna().iloc[0]) if 'my_hand_count' in g.columns and pd.to_numeric(g.get('my_hand_count'), errors='coerce').notna().any() else None,
            'num_options': int(len(g)),
            'option_signature': '|'.join(opt_types),
            'chosen_signature': '|'.join(chosen_types),
            'yes_available': int('YES' in opt_types),
            'no_available': int('NO' in opt_types),
            'end_available': int('END' in opt_types),
            'attack_available': int('ATTACK' in opt_types),
            'yes_chosen': int('YES' in chosen_types),
            'no_chosen': int('NO' in chosen_types),
            'end_chosen': int('END' in chosen_types),
            'attack_chosen': int('ATTACK' in chosen_types),
            'chosen_card_top': _top_counter_json(chosen.get('card_id', pd.Series(dtype=object)), 5) if not chosen.empty else '[]',
            'chosen_attack_top': _top_counter_json(chosen.get('attack_id', pd.Series(dtype=object)), 5) if not chosen.empty else '[]',
        })
    dec_df = pd.DataFrame(rows)
    if dec_df.empty:
        return dec_df
    summary_rows = []
    for sig, g in dec_df.groupby('option_signature', dropna=False):
        summary_rows.append({
            'option_signature': sig,
            'decisions': int(len(g)),
            'avg_turn': float(pd.to_numeric(g['turn'], errors='coerce').mean()) if 'turn' in g else None,
            'avg_my_deck_count': float(pd.to_numeric(g['my_deck_count'], errors='coerce').mean()) if 'my_deck_count' in g else None,
            'yes_available_rate': float(g['yes_available'].mean()),
            'no_available_rate': float(g['no_available'].mean()),
            'end_available_rate': float(g['end_available'].mean()),
            'attack_available_rate': float(g['attack_available'].mean()),
            'yes_chosen_rate': float(g['yes_chosen'].mean()),
            'no_chosen_rate': float(g['no_chosen'].mean()),
            'end_chosen_rate': float(g['end_chosen'].mean()),
            'attack_chosen_rate': float(g['attack_chosen'].mean()),
        })
    return pd.DataFrame(summary_rows).sort_values(['decisions'], ascending=False)


def ensure_mlp_decision_id(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    if 'decision_id' not in work.columns or work['decision_id'].isna().all():
        parts = []
        for c in ['episode_id', 'step', 'player_index', 'context_name']:
            if c in work.columns:
                parts.append(work[c].astype(str))
            else:
                parts.append(pd.Series('', index=work.index))
        work['decision_id'] = parts[0] + ':' + parts[1] + ':' + parts[2] + ':' + parts[3]
    work['decision_id'] = work['decision_id'].astype(str)
    return work


def prepare_mlp_training_frame(df: pd.DataFrame, max_rows: int = MLP_MAX_ROWS, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    work = ensure_mlp_decision_id(df)
    if 'is_chosen' not in work.columns:
        return pd.DataFrame()
    work['is_chosen'] = pd.to_numeric(work['is_chosen'], errors='coerce').fillna(0).astype(int)
    work = work[work['is_chosen'].isin([0, 1])].copy()
    if work.empty:
        return work
    # Keep only decisions with at least one positive and at least two legal options.
    dec = work.groupby('decision_id').agg(
        rows=('is_chosen', 'size'),
        positives=('is_chosen', 'sum'),
        context_name=('context_name', 'first') if 'context_name' in work.columns else ('is_chosen', 'size')
    ).reset_index()
    dec = dec[(dec['rows'] >= 2) & (dec['positives'] >= 1)].copy()
    if dec.empty:
        return pd.DataFrame()
    if len(work) > max_rows:
        avg_rows = max(2.0, float(dec['rows'].mean()))
        target_decisions = max(1000, int(max_rows / avg_rows))
        target_decisions = min(target_decisions, len(dec))
        rng = np.random.default_rng(seed) if np is not None else None
        if rng is not None:
            weights = np.ones(len(dec), dtype=float)
            if 'context_name' in dec.columns:
                weights += dec['context_name'].astype(str).isin(MLP_PRIORITY_CONTEXTS).astype(float).to_numpy() * 2.5
                # UNKNOWN_0 and TO_HAND are the first neural targets.
                weights += dec['context_name'].astype(str).isin(['UNKNOWN_0', 'TO_HAND']).astype(float).to_numpy() * 1.5
            weights = weights / weights.sum()
            chosen_ids = rng.choice(dec['decision_id'].to_numpy(), size=target_decisions, replace=False, p=weights)
        else:
            chosen_ids = random.sample(dec['decision_id'].tolist(), target_decisions)
        work = work[work['decision_id'].isin(set(chosen_ids))].copy()
    return work.reset_index(drop=True)


def split_by_decision(work: pd.DataFrame, valid_size: float = MLP_VALID_SIZE, seed: int = RANDOM_SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    decision_ids = work['decision_id'].drop_duplicates().tolist()
    if len(decision_ids) < 20:
        return pd.DataFrame(), pd.DataFrame()
    rng = random.Random(seed)
    rng.shuffle(decision_ids)
    n_valid = max(1, int(len(decision_ids) * valid_size))
    valid_ids = set(decision_ids[:n_valid])
    train_ids = set(decision_ids[n_valid:])
    train = work[work['decision_id'].isin(train_ids)].copy()
    valid = work[work['decision_id'].isin(valid_ids)].copy()
    return train, valid


def fit_mlp_preprocessor(train: pd.DataFrame) -> dict:
    num_cols = [c for c in MLP_NUMERIC_FEATURES if c in train.columns]
    cat_cols = [c for c in MLP_CATEGORICAL_FEATURES if c in train.columns]
    num_stats = {}
    for c in num_cols:
        s = pd.to_numeric(train[c], errors='coerce')
        if np is not None:
            s = s.replace([np.inf, -np.inf], np.nan)
        mean = float(s.mean()) if s.notna().any() else 0.0
        std = float(s.std()) if s.notna().sum() > 1 else 1.0
        if not math.isfinite(std) or std <= 1e-6:
            std = 1.0
        if not math.isfinite(mean):
            mean = 0.0
        num_stats[c] = {'mean': mean, 'std': std}
    cat_maps = {}
    for c in cat_cols:
        vals = train[c].astype('string').fillna('UNK').astype(str)
        counts = vals.value_counts(dropna=False)
        # Cap extremely high cardinality while keeping common card/attack IDs.
        keep = counts.head(2000).index.tolist()
        mapping = {'__UNK__': 0}
        for v in keep:
            if v not in mapping:
                mapping[v] = len(mapping)
        cat_maps[c] = mapping
    return {'numeric_cols': num_cols, 'categorical_cols': cat_cols, 'num_stats': num_stats, 'cat_maps': cat_maps}


def transform_mlp_frame(df: pd.DataFrame, prep: dict):
    num_cols = prep.get('numeric_cols', [])
    cat_cols = prep.get('categorical_cols', [])
    num_arr = []
    for c in num_cols:
        s = pd.to_numeric(df.get(c, pd.Series(0, index=df.index)), errors='coerce')
        if np is not None:
            s = s.replace([np.inf, -np.inf], np.nan)
        mean = prep['num_stats'][c]['mean']
        std = prep['num_stats'][c]['std']
        num_arr.append(((s.fillna(mean).astype(float) - mean) / std).to_numpy(dtype='float32'))
    if num_arr:
        X_num = np.vstack(num_arr).T.astype('float32')
    else:
        X_num = np.zeros((len(df), 0), dtype='float32')
    cat_arr = []
    for c in cat_cols:
        mapping = prep['cat_maps'][c]
        vals = df.get(c, pd.Series('UNK', index=df.index)).astype('string').fillna('UNK').astype(str)
        cat_arr.append(vals.map(lambda x: mapping.get(x, 0)).to_numpy(dtype='int64'))
    if cat_arr:
        X_cat = np.vstack(cat_arr).T.astype('int64')
    else:
        X_cat = np.zeros((len(df), 0), dtype='int64')
    return X_num, X_cat


def grouped_prediction_metrics(df: pd.DataFrame, pred_col: str) -> dict:
    if df is None or df.empty or pred_col not in df.columns or 'decision_id' not in df.columns:
        return {'decisions': 0, 'top1': None, 'top3': None, 'avg_chosen_rank': None}
    top1 = []
    top3 = []
    ranks = []
    for _, g in df.groupby('decision_id', sort=False):
        y = pd.to_numeric(g.get('is_chosen'), errors='coerce').fillna(0).astype(int).to_numpy()
        if y.sum() <= 0 or len(y) == 0:
            continue
        pred = pd.to_numeric(g[pred_col], errors='coerce').fillna(-1e9).to_numpy(dtype=float)
        order = list(np.argsort(-pred)) if np is not None else sorted(range(len(pred)), key=lambda i: -pred[i])
        chosen = set([i for i, v in enumerate(y) if v == 1])
        top1.append(int(order[0] in chosen))
        top3.append(int(any(i in chosen for i in order[:min(3, len(order))])))
        rank_map = {idx: rank + 1 for rank, idx in enumerate(order)}
        ranks.append(min(rank_map[i] for i in chosen))
    if not top1:
        return {'decisions': 0, 'top1': None, 'top3': None, 'avg_chosen_rank': None}
    return {
        'decisions': int(len(top1)),
        'top1': float(sum(top1) / len(top1)),
        'top3': float(sum(top3) / len(top3)),
        'avg_chosen_rank': float(sum(ranks) / len(ranks)),
    }


def context_prediction_report(df: pd.DataFrame, pred_cols: list[str]) -> pd.DataFrame:
    rows = []
    if df is None or df.empty:
        return pd.DataFrame()
    for ctx, g in df.groupby('context_name' if 'context_name' in df.columns else 'decision_id', dropna=False):
        row = {'context_name': str(ctx), 'rows': int(len(g)), 'decisions': int(g['decision_id'].nunique()) if 'decision_id' in g.columns else int(len(g))}
        for pc in pred_cols:
            if pc in g.columns:
                m = grouped_prediction_metrics(g, pc)
                row[f'{pc}_top1'] = m['top1']
                row[f'{pc}_top3'] = m['top3']
                row[f'{pc}_avg_chosen_rank'] = m['avg_chosen_rank']
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['decisions'], ascending=False)


def offline_rule_proxy_scores(df: pd.DataFrame) -> pd.Series:
    # A deliberately simple, transparent proxy. It is NOT the full main.py rule score.
    key_setup = {Abra, Kadabra, Alakazam, Rare_Candy, Buddy_Buddy_Poffin, Hilda, Dawn}
    key_control = {Boss_Orders, Enhanced_Hammer, Nighttime_Mine}
    energy_ids = {Basic_Psychic_Energy, Telepath_Psychic_Energy, Enriching_Energy}
    out = []
    for _, r in df.iterrows():
        score = 0.0
        ctx = str(r.get('context_name', ''))
        ot = str(r.get('option_type', ''))
        card = to_int_or_none(r.get('card_id'))
        attack = to_int_or_none(r.get('attack_id'))
        deckout = float(pd.to_numeric(pd.Series([r.get('deckout_risk_feature', 0)]), errors='coerce').fillna(0).iloc[0]) > 0
        has_alakazam = float(pd.to_numeric(pd.Series([r.get('my_alakazam_count', 0)]), errors='coerce').fillna(0).iloc[0]) > 0
        can_ko = float(pd.to_numeric(pd.Series([r.get('powerful_hand_can_ko_active', 0)]), errors='coerce').fillna(0).iloc[0]) > 0
        if ot == 'ATTACK':
            score += 1000 + (2500 if can_ko else 0)
        elif ot == 'END':
            score += -20 + (80 if deckout else 0)
        elif ot == 'YES':
            score += -50 if deckout else 20
        elif ot == 'NO':
            score += 30 if deckout else 0
        elif ot in ('PLAY', 'EVOLVE', 'ABILITY', 'ATTACH'):
            score += 200
        if card is not None:
            if card in key_setup:
                score += 300 if not has_alakazam else 40
            if card in key_control:
                score += 220
            if card in energy_ids:
                score += 100
            if deckout and card not in key_control and card not in {Alakazam, Boss_Orders}:
                score -= 250
        if ctx == 'TO_HAND' and card in key_setup:
            score += 150
        if ctx == 'TO_BENCH' and card in {Abra, Dunsparce}:
            score += 200
        if ctx == 'UNKNOWN_0' and ot in ('ATTACK', 'END', 'YES', 'NO'):
            score += 60
        out.append(score)
    return pd.Series(out, index=df.index, dtype=float)


def predict_with_bc_models(df: pd.DataFrame) -> pd.Series:
    probs = pd.Series(np.nan, index=df.index, dtype=float) if np is not None else pd.Series([math.nan] * len(df), index=df.index, dtype=float)
    if 'ALAKAZAM_BC_MODELS' not in globals() or not ALAKAZAM_BC_MODELS:
        return probs
    for ctx, g in df.groupby('context_name' if 'context_name' in df.columns else df.index, dropna=False):
        bundle = ALAKAZAM_BC_MODELS.get(str(ctx)) or ALAKAZAM_BC_MODELS.get(ctx) or ALAKAZAM_BC_MODELS.get('ALL')
        if not bundle:
            continue
        try:
            X, _, _ = model_input(g, numeric_features=bundle.get('numeric_features', NUMERIC_FEATURES), categorical_features=bundle.get('categorical_features', CATEGORICAL_FEATURES))
            probs.loc[g.index] = bundle['model'].predict_proba(X)[:, 1]
        except Exception as exc:
            log_error('bc_prediction_for_mlp_report', context=str(ctx), error=repr(exc))
    return probs


def build_disagreement_examples(df: pd.DataFrame, max_examples: int = 200) -> pd.DataFrame:
    if df is None or df.empty or 'mlp_pred' not in df.columns or 'rule_proxy_score' not in df.columns:
        return pd.DataFrame()
    rows = []
    for did, g in df.groupby('decision_id', sort=False):
        y = pd.to_numeric(g.get('is_chosen'), errors='coerce').fillna(0).astype(int).to_numpy()
        if y.sum() <= 0:
            continue
        mlp_pred = pd.to_numeric(g['mlp_pred'], errors='coerce').fillna(-1e9).to_numpy(dtype=float)
        rule_pred = pd.to_numeric(g['rule_proxy_score'], errors='coerce').fillna(-1e9).to_numpy(dtype=float)
        mlp_i = int(np.argmax(mlp_pred))
        rule_i = int(np.argmax(rule_pred))
        chosen_set = set([i for i, v in enumerate(y) if v == 1])
        if mlp_i == rule_i:
            continue
        label = 'rule_wrong_mlp_right' if (mlp_i in chosen_set and rule_i not in chosen_set) else 'mlp_wrong_rule_right' if (rule_i in chosen_set and mlp_i not in chosen_set) else 'both_or_neither'
        gg = g.reset_index(drop=True)
        rows.append({
            'decision_id': did,
            'context_name': str(gg.get('context_name', pd.Series([''])).iloc[0]),
            'label': label,
            'mlp_confidence': float(mlp_pred[mlp_i]),
            'mlp_option_index': int(gg.get('option_index', pd.Series([mlp_i])).iloc[mlp_i]) if 'option_index' in gg else mlp_i,
            'mlp_option_type': str(gg.get('option_type', pd.Series([''])).iloc[mlp_i]) if 'option_type' in gg else '',
            'mlp_card_id': to_int_or_none(gg.get('card_id', pd.Series([None])).iloc[mlp_i]) if 'card_id' in gg else None,
            'rule_option_index': int(gg.get('option_index', pd.Series([rule_i])).iloc[rule_i]) if 'option_index' in gg else rule_i,
            'rule_option_type': str(gg.get('option_type', pd.Series([''])).iloc[rule_i]) if 'option_type' in gg else '',
            'rule_card_id': to_int_or_none(gg.get('card_id', pd.Series([None])).iloc[rule_i]) if 'card_id' in gg else None,
            'chosen_option_types': '|'.join(gg.loc[list(chosen_set), 'option_type'].astype(str).tolist()) if 'option_type' in gg else '',
            'chosen_card_ids': '|'.join(str(to_int_or_none(x)) for x in gg.loc[list(chosen_set), 'card_id'].tolist()) if 'card_id' in gg else '',
        })
        if len(rows) >= max_examples:
            break
    return pd.DataFrame(rows)


def make_optional_mlp_submission_agent_source(base_source: str) -> str:
    """Safety-gated v0-05d hook.

    The notebook trains a PyTorch MLP offline. The final submission remains rule-only by
    default. If MLP submission assist is requested before a pure-Python forward package is
    explicitly enabled, fail loudly rather than silently shipping an unsafe torch dependency.
    """
    if not USE_MLP_RERANKER_FOR_SUBMISSION:
        return base_source
    raise RuntimeError(
        'USE_MLP_RERANKER_FOR_SUBMISSION=True was requested, but this safe notebook does not '
        'ship torch inside main.py. Review MLP reports and add a pure-Python/numpy forward '
        'package before enabling neural submission assist.'
    )

# Always build B2 reports, even if torch is unavailable.
ERROR_CLASSIFICATION_DF = classify_parse_errors(V05_ERROR_ROWS)
UNKNOWN0_ACTION_REPORT_DF = build_unknown0_action_report(ALAKAZAM_OPTION_MODEL_DF if 'ALAKAZAM_OPTION_MODEL_DF' in globals() else pd.DataFrame())
ARTIFACT_PATHS['error_classification'] = safe_save_table(ERROR_CLASSIFICATION_DF, OUTPUT_DIR / 'error_classification')
ARTIFACT_PATHS['unknown0_action_report'] = safe_save_table(UNKNOWN0_ACTION_REPORT_DF, OUTPUT_DIR / 'unknown0_action_report')

print('unknown0 action report rows:', len(UNKNOWN0_ACTION_REPORT_DF))
if not UNKNOWN0_ACTION_REPORT_DF.empty:
    display(UNKNOWN0_ACTION_REPORT_DF.head(20)) if 'display' in globals() else print(UNKNOWN0_ACTION_REPORT_DF.head(20))

# PyTorch MLP training/evaluation. Soft-fail by design.
try:
    import torch
    from torch import nn
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = repr(exc)

if not RUN_MLP_RERANKER:
    MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'RUN_MLP_RERANKER_FALSE'}
elif not TORCH_AVAILABLE:
    MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'torch_unavailable', 'error': TORCH_IMPORT_ERROR}
elif 'ALAKAZAM_OPTION_MODEL_DF' not in globals() or ALAKAZAM_OPTION_MODEL_DF.empty:
    MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'empty_option_model_df'}
elif np is None:
    MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'numpy_unavailable'}
else:
    try:
        work = prepare_mlp_training_frame(ALAKAZAM_OPTION_MODEL_DF, max_rows=MLP_MAX_ROWS)
        if work.empty or work['is_chosen'].nunique() < 2 or work['decision_id'].nunique() < 20:
            MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'insufficient_mlp_rows', 'rows': int(len(work))}
        else:
            train_df, valid_df = split_by_decision(work)
            if train_df.empty or valid_df.empty or train_df['is_chosen'].nunique() < 2:
                MLP_RERANKER_REPORT = {'status': 'skipped', 'reason': 'bad_decision_split', 'rows': int(len(work))}
            else:
                prep = fit_mlp_preprocessor(train_df)
                Xn_train, Xc_train = transform_mlp_frame(train_df, prep)
                Xn_valid, Xc_valid = transform_mlp_frame(valid_df, prep)
                y_train = train_df['is_chosen'].astype('float32').to_numpy()
                y_valid = valid_df['is_chosen'].astype('float32').to_numpy()
                # MC return weights
                if USE_MC_RETURN_WEIGHTS and 'mc_step_weight' in train_df.columns:
                    _w_raw = train_df['mc_step_weight'].fillna(1.0).to_numpy().astype('float32')
                    weights_train = _w_raw / _w_raw.mean()  # normalize so mean=1
                else:
                    weights_train = None

                class TabularOptionMLP(nn.Module):
                    def __init__(self, num_dim: int, cat_cardinalities: list[int]):
                        super().__init__()
                        self.embeddings = nn.ModuleList()
                        emb_total = 0
                        for card in cat_cardinalities:
                            dim = int(min(32, max(4, round(math.sqrt(max(card, 2))))))
                            self.embeddings.append(nn.Embedding(card, dim))
                            emb_total += dim
                        in_dim = int(num_dim + emb_total)
                        self.net = nn.Sequential(
                            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.10),
                            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.05),
                            nn.Linear(64, 1),
                        )
                    def forward(self, x_num, x_cat):
                        parts = [x_num]
                        if len(self.embeddings) > 0 and x_cat.shape[1] > 0:
                            for j, emb in enumerate(self.embeddings):
                                parts.append(emb(x_cat[:, j]))
                        x = torch.cat(parts, dim=1) if len(parts) > 1 else x_num
                        return self.net(x).squeeze(1)

                device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                cat_cards = [len(prep['cat_maps'][c]) for c in prep.get('categorical_cols', [])]
                model = TabularOptionMLP(Xn_train.shape[1], cat_cards).to(device)
                opt = torch.optim.AdamW(model.parameters(), lr=MLP_LEARNING_RATE, weight_decay=MLP_WEIGHT_DECAY)
                pos = float(y_train.sum())
                neg = float(len(y_train) - pos)
                pos_weight_val = min(20.0, max(1.0, neg / max(pos, 1.0)))
                _loss_reduction = 'none' if (USE_MC_RETURN_WEIGHTS and weights_train is not None) else 'mean'
                loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], device=device), reduction=_loss_reduction)
                w_t = torch.tensor(weights_train, dtype=torch.float32) if weights_train is not None else None
                Xn_t = torch.tensor(Xn_train, dtype=torch.float32)
                Xc_t = torch.tensor(Xc_train, dtype=torch.long)
                y_t = torch.tensor(y_train, dtype=torch.float32)
                batch_size = max(256, min(int(MLP_BATCH_SIZE), len(train_df)))
                epoch_losses = []
                for epoch in range(int(MLP_EPOCHS)):
                    model.train()
                    perm = torch.randperm(len(train_df))
                    total_loss = 0.0
                    seen = 0
                    for start in range(0, len(train_df), batch_size):
                        idx = perm[start:start + batch_size]
                        xb_num = Xn_t[idx].to(device)
                        xb_cat = Xc_t[idx].to(device)
                        yb = y_t[idx].to(device)
                        opt.zero_grad(set_to_none=True)
                        logits = model(xb_num, xb_cat)
                        _raw_loss = loss_fn(logits, yb)
                        if w_t is not None:
                            wb = w_t[idx].to(device)
                            loss = (_raw_loss * wb).sum() / wb.sum().clamp(min=1e-8)
                        else:
                            loss = _raw_loss
                        if not torch.isfinite(loss):
                            raise RuntimeError(f'MLP loss became non-finite at epoch={epoch}')
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                        opt.step()
                        total_loss += float(loss.detach().cpu()) * len(idx)
                        seen += len(idx)
                    epoch_losses.append(total_loss / max(1, seen))
                    print(f'mlp epoch {epoch+1}/{MLP_EPOCHS}: loss={epoch_losses[-1]:.5f}')

                model.eval()
                with torch.no_grad():
                    Xn_v = torch.tensor(Xn_valid, dtype=torch.float32, device=device)
                    Xc_v = torch.tensor(Xc_valid, dtype=torch.long, device=device)
                    logits = model(Xn_v, Xc_v).detach().cpu().numpy()
                    mlp_pred = 1.0 / (1.0 + np.exp(-np.clip(logits, -40, 40)))
                pred_df = valid_df.copy()
                pred_df['mlp_pred'] = mlp_pred.astype(float)
                pred_df['rule_proxy_score'] = offline_rule_proxy_scores(pred_df)
                pred_df['bc_pred'] = predict_with_bc_models(pred_df)

                pred_cols = ['mlp_pred', 'rule_proxy_score'] + (['bc_pred'] if pred_df['bc_pred'].notna().any() else [])
                MLP_CONTEXT_REPORT_DF = context_prediction_report(pred_df, pred_cols)
                overall_mlp_metrics = grouped_prediction_metrics(pred_df, 'mlp_pred')
                overall_rule_metrics = grouped_prediction_metrics(pred_df, 'rule_proxy_score')
                overall_bc_metrics = grouped_prediction_metrics(pred_df[pred_df['bc_pred'].notna()].copy(), 'bc_pred') if pred_df['bc_pred'].notna().any() else {'decisions': 0}
                MLP_DISAGREEMENT_EXAMPLES_DF = build_disagreement_examples(pred_df, max_examples=250)
                MLP_DECISION_REPORT_DF = pred_df[['decision_id','context_name','option_index','option_type','card_id','attack_id','is_chosen','mlp_pred','rule_proxy_score','bc_pred']].copy() if set(['decision_id','context_name']).issubset(pred_df.columns) else pred_df.copy()

                # Save artifacts. Model is offline only; submission remains rule-only by default.
                mlp_model_path = MODEL_DIR / f'{RUN_PREFIX}-alakazam_mlp_reranker.pt'
                mlp_prep_path = MODEL_DIR / f'{RUN_PREFIX}-alakazam_mlp_preprocessor.pkl'
                torch.save({'state_dict': model.state_dict(), 'num_dim': Xn_train.shape[1], 'cat_cardinalities': cat_cards, 'prep': prep}, mlp_model_path)
                with open(mlp_prep_path, 'wb') as f:
                    pickle.dump(prep, f)
                ARTIFACT_PATHS['alakazam_mlp_reranker'] = str(mlp_model_path)
                ARTIFACT_PATHS['alakazam_mlp_preprocessor'] = str(mlp_prep_path)
                ARTIFACT_PATHS['mlp_context_report'] = safe_save_table(MLP_CONTEXT_REPORT_DF, OUTPUT_DIR / 'mlp_context_report')
                ARTIFACT_PATHS['mlp_valid_predictions'] = safe_save_table(MLP_DECISION_REPORT_DF, OUTPUT_DIR / 'mlp_valid_predictions')
                ARTIFACT_PATHS['mlp_disagreement_examples'] = safe_save_table(MLP_DISAGREEMENT_EXAMPLES_DF, OUTPUT_DIR / 'mlp_disagreement_examples')

                try:
                    from sklearn.metrics import roc_auc_score, log_loss
                    row_auc = float(roc_auc_score(y_valid, mlp_pred)) if len(set(y_valid.tolist())) == 2 else None
                    row_logloss = float(log_loss(y_valid, mlp_pred, labels=[0, 1]))
                except Exception:
                    row_auc = None
                    row_logloss = None
                MLP_RERANKER_REPORT = {
                    'status': 'ok',
                    'rows': int(len(work)),
                    'train_rows': int(len(train_df)),
                    'valid_rows': int(len(valid_df)),
                    'train_decisions': int(train_df['decision_id'].nunique()),
                    'valid_decisions': int(valid_df['decision_id'].nunique()),
                    'epochs': int(MLP_EPOCHS),
                    'batch_size': int(batch_size),
                    'device': str(device),
                    'positive_rate': float(work['is_chosen'].mean()),
                    'pos_weight': float(pos_weight_val),
                    'epoch_losses': [float(x) for x in epoch_losses],
                    'row_auc': row_auc,
                    'row_logloss': row_logloss,
                    'mlp_decision_metrics': overall_mlp_metrics,
                    'rule_proxy_decision_metrics': overall_rule_metrics,
                    'bc_decision_metrics': overall_bc_metrics,
                    'feature_safety': 'submission_safe_features_only_except opponent_archetype_if_replay_inferred',
                    'submission_enabled': bool(USE_MLP_RERANKER_FOR_SUBMISSION),
                }
                print('mlp report:', MLP_RERANKER_REPORT)
                if not MLP_CONTEXT_REPORT_DF.empty:
                    display(MLP_CONTEXT_REPORT_DF.head(20)) if 'display' in globals() else print(MLP_CONTEXT_REPORT_DF.head(20))
                if not MLP_DISAGREEMENT_EXAMPLES_DF.empty:
                    display(MLP_DISAGREEMENT_EXAMPLES_DF.head(20)) if 'display' in globals() else print(MLP_DISAGREEMENT_EXAMPLES_DF.head(20))
    except Exception as exc:
        MLP_RERANKER_REPORT = {'status': 'error', 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
        log_error('mlp_reranker', error=repr(exc), traceback=traceback.format_exc(limit=8))

ARTIFACT_PATHS['mlp_reranker_report'] = write_json(OUTPUT_DIR / 'mlp_reranker_report.json', MLP_RERANKER_REPORT)
ARTIFACT_PATHS['mlp_submission_safety'] = write_json(OUTPUT_DIR / 'mlp_submission_safety.json', MLP_SUBMISSION_SAFETY)
print('MLP_RERANKER_REPORT status:', MLP_RERANKER_REPORT.get('status'))
print('MLP submission assist enabled:', USE_MLP_RERANKER_FOR_SUBMISSION)


unknown0 action report rows: 64
              option_signature  decisions   avg_turn  avg_my_deck_count  \
1                 12|13|14|END       3480   7.236494          23.703448   
6          12|13|14|END|NUMBER       3190   9.763323          14.607210   
53               14|END|NO|YES       2492   7.630417          24.024880   
49                      14|END       2362   5.777307          30.490262   
7      12|13|14|END|NUMBER|YES       2225  11.170337          12.955506   
5          12|13|14|END|NO|YES       2189   8.657835          20.505710   
56                  14|END|YES       2118   5.965061          30.857413   
2              12|13|14|END|NO       2082   7.486071          23.059558   
8             12|13|14|END|YES       1819   8.391424          22.103903   
50                   14|END|NO       1478   6.427605          28.804465   
48                          14       1351   3.985936          34.924500   
54               14|END|NUMBER       1325   8.521509          21.806

## v0-05d1: UNKNOWN_0 policy-table assist with runtime hit tracking

This stage trains a submission-safe UNKNOWN_0 MLP, distills high-confidence behavior into a policy table, and injects runtime counters into the policy-table `main.py` variant.

The final `submission.tar.gz` defaults to `unknown0_policy_table`, but still falls back to `rule_only` if the policy artifact fails safety checks.


In [17]:
# -----------------------------
# v0-05d1: UNKNOWN_0 submission-safe MLP, calibration, policy table, and runtime hit tracking
# -----------------------------
UNKNOWN0_MLP_REPORT = {'status': 'not_run'}
UNKNOWN0_MLP_SIGNATURE_REPORT_DF = pd.DataFrame()
UNKNOWN0_MLP_CALIBRATION_DF = pd.DataFrame()
UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF = pd.DataFrame()
UNKNOWN0_MLP_VALID_PRED_DF = pd.DataFrame()
UNKNOWN0_POLICY_TABLE_DF = pd.DataFrame()
UNKNOWN0_POLICY_TABLE = {}
UNKNOWN0_POLICY_FIT_PRED_DF = pd.DataFrame()
UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF = pd.DataFrame()
UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY = {'status': 'not_run'}
UNKNOWN0_POLICY_SUMMARY = {'status': 'not_run'}
UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS = {
    'status': 'not_built',
    'reason': 'main.py remains torch-free; the submission variant uses a distilled policy table with rule fallback',
}

UNKNOWN0_NUMERIC_FEATURES = [
    'option_index', 'num_options', 'min_count', 'max_count',
    'card_id', 'target_card_id', 'attack_id', 'number_value',
    'in_play_index', 'remain_damage_counter', 'remain_energy_cost',
    'turn', 'turn_action_count', 'my_active_id', 'my_active_hp', 'my_active_energy_count',
    'op_active_id', 'op_active_hp', 'op_active_energy_count', 'my_bench_count', 'op_bench_count',
    'my_alakazam_count', 'my_kadabra_count', 'my_abra_count', 'my_dudunsparce_count',
    'my_hand_count', 'op_hand_count', 'my_deck_count', 'op_deck_count',
    'my_prizes_left', 'op_prizes_left', 'stadium_id', 'powerful_hand_damage_est',
    'powerful_hand_can_ko_active', 'deckout_risk_feature',
]
UNKNOWN0_CATEGORICAL_FEATURES = ['context_name', 'option_type', 'area', 'in_play_area', 'option_signature']
UNKNOWN0_FORBIDDEN_FEATURE_PATTERNS = ['winner', 'won', 'reward', 'future', 'visualize', 'deck_list', 'deck_hash']


def assert_unknown0_feature_safety(num_cols, cat_cols):
    requested = list(num_cols) + list(cat_cols)
    forbidden = []
    for c in requested:
        lo = str(c).lower()
        if any(p in lo for p in UNKNOWN0_FORBIDDEN_FEATURE_PATTERNS):
            forbidden.append(c)
    if forbidden:
        raise AssertionError(f'UNKNOWN0 submission-safe feature list has forbidden columns: {forbidden}')
    return {
        'requested_feature_count': int(len(requested)),
        'forbidden_feature_count': int(len(forbidden)),
        'uses_visualize_features_for_mlp': False,
        'offline_only_feature_count': 0,
    }



def _to_abstract_option_sig(option_types):
    types = set(str(t) for t in option_types)
    numeric = [t for t in types if t.lstrip('-').isdigit()]
    n = len(numeric)
    n_bucket = 'N' + ('0' if n == 0 else '1' if n == 1 else '2' if n == 2 else '3p')
    parts = [n_bucket]
    for kw in ['END', 'YES', 'NO', 'NUMBER']:
        if kw in types:
            parts.append(kw)
    return '|'.join(sorted(parts))

def add_unknown0_signatures(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if 'decision_id' not in out.columns:
        out['decision_id'] = (
            out.get('episode_id', pd.Series('', index=out.index)).astype(str) + ':' +
            out.get('step', pd.Series('', index=out.index)).astype(str) + ':' +
            out.get('player_index', pd.Series('', index=out.index)).astype(str)
        )
    out['decision_id'] = out['decision_id'].astype(str)
    if 'option_type' not in out.columns:
        out['option_type'] = 'UNK'
    sig_map = out.groupby('decision_id')['option_type'].apply(
        lambda s: '|'.join(sorted(s.astype(str).fillna('UNK').unique().tolist()))
    ).to_dict()
    out['option_signature'] = out['decision_id'].map(sig_map).fillna('')
    # v0-05d10: replace with abstract signature when flag is set
    if 'USE_ABSTRACT_OPTION_SIGNATURE' in globals() and USE_ABSTRACT_OPTION_SIGNATURE:
        _abs_map = out.groupby('decision_id')['option_type'].apply(
            lambda s: _to_abstract_option_sig(s.astype(str).fillna('UNK').unique().tolist())
        ).to_dict()
        out['option_signature'] = out['decision_id'].map(_abs_map).fillna('N0')
    turn_s = pd.to_numeric(out.get('turn', pd.Series(0, index=out.index)), errors='coerce').fillna(0)
    deck_s = pd.to_numeric(out.get('my_deck_count', pd.Series(999, index=out.index)), errors='coerce').fillna(999)
    ko_s = pd.to_numeric(out.get('powerful_hand_can_ko_active', pd.Series(0, index=out.index)), errors='coerce').fillna(0)
    deckout_s = pd.to_numeric(out.get('deckout_risk_feature', pd.Series(0, index=out.index)), errors='coerce').fillna(0)
    out['unknown0_turn_bucket'] = pd.cut(turn_s, bins=[-1, 2, 5, 999], labels=['early', 'mid', 'late']).astype(str)
    if np is not None:
        out['unknown0_deck_bucket'] = np.where(deckout_s > 0, 'risk', np.where(deck_s <= 3, 'strict', np.where(deck_s <= 6, 'thin', 'safe')))
        out['unknown0_ko_bucket'] = np.where(ko_s > 0, 'ko', 'no_ko')
    else:
        out['unknown0_deck_bucket'] = 'safe'
        out['unknown0_ko_bucket'] = 'no_ko'
    return out


def fit_unknown0_preprocessor(train):
    num_cols = [c for c in UNKNOWN0_NUMERIC_FEATURES if c in train.columns]
    cat_cols = [c for c in UNKNOWN0_CATEGORICAL_FEATURES if c in train.columns]
    num_stats = {}
    for c in num_cols:
        s = pd.to_numeric(train[c], errors='coerce')
        if np is not None:
            s = s.replace([np.inf, -np.inf], np.nan)
        mean = float(s.mean()) if s.notna().any() else 0.0
        std = float(s.std()) if s.notna().sum() > 1 else 1.0
        if not math.isfinite(mean):
            mean = 0.0
        if not math.isfinite(std) or std <= 1e-6:
            std = 1.0
        num_stats[c] = {'mean': mean, 'std': std}
    cat_maps = {}
    for c in cat_cols:
        vals = train[c].astype('string').fillna('UNK').astype(str)
        counts = vals.value_counts(dropna=False)
        mapping = {'__UNK__': 0}
        for v in counts.head(2000).index.tolist():
            if v not in mapping:
                mapping[v] = len(mapping)
        cat_maps[c] = mapping
    return {'numeric_cols': num_cols, 'categorical_cols': cat_cols, 'num_stats': num_stats, 'cat_maps': cat_maps}


def transform_unknown0_frame(df, prep):
    num_arr = []
    for c in prep.get('numeric_cols', []):
        s = pd.to_numeric(df.get(c, pd.Series(0, index=df.index)), errors='coerce')
        if np is not None:
            s = s.replace([np.inf, -np.inf], np.nan)
        mean = prep['num_stats'][c]['mean']
        std = prep['num_stats'][c]['std']
        num_arr.append(((s.fillna(mean).astype(float) - mean) / std).to_numpy(dtype='float32'))
    X_num = np.vstack(num_arr).T.astype('float32') if num_arr else np.zeros((len(df), 0), dtype='float32')
    cat_arr = []
    for c in prep.get('categorical_cols', []):
        mapping = prep['cat_maps'][c]
        vals = df.get(c, pd.Series('UNK', index=df.index)).astype('string').fillna('UNK').astype(str)
        cat_arr.append(vals.map(lambda x: mapping.get(x, 0)).to_numpy(dtype='int64'))
    X_cat = np.vstack(cat_arr).T.astype('int64') if cat_arr else np.zeros((len(df), 0), dtype='int64')
    return X_num, X_cat


def prepare_unknown0_training_frame(df, max_rows=None):
    if df is None or df.empty or 'context_name' not in df.columns or 'is_chosen' not in df.columns:
        return pd.DataFrame()
    work = df[df['context_name'].astype(str).eq('UNKNOWN_0')].copy()
    if work.empty:
        return pd.DataFrame()
    work = add_unknown0_signatures(work)
    work['is_chosen'] = pd.to_numeric(work['is_chosen'], errors='coerce').fillna(0).astype(int)
    dec = work.groupby('decision_id').agg(rows=('is_chosen','size'), positives=('is_chosen','sum')).reset_index()
    dec = dec[(dec['rows'] >= 2) & (dec['positives'] >= 1)]
    work = work[work['decision_id'].isin(set(dec['decision_id']))].copy()
    if max_rows is not None and len(work) > int(max_rows):
        # Preserve all positives when possible, and sample negatives. This keeps chosen actions visible.
        pos = work[work['is_chosen'] == 1]
        neg = work[work['is_chosen'] == 0]
        neg_n = max(0, int(max_rows) - len(pos))
        if neg_n > 0:
            neg = neg.sample(n=min(len(neg), neg_n), random_state=RANDOM_SEED)
            work = pd.concat([pos, neg], axis=0).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
        else:
            work = pos.sample(n=min(len(pos), int(max_rows)), random_state=RANDOM_SEED).reset_index(drop=True)
    return work.reset_index(drop=True)


def split_by_decision_safe(df, valid_frac=0.15):
    if df is None or df.empty or 'decision_id' not in df.columns:
        return pd.DataFrame(), pd.DataFrame()
    decisions = list(pd.Series(df['decision_id'].dropna().unique()))
    if len(decisions) < 5:
        return df.copy(), pd.DataFrame()
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(decisions)
    n_valid = max(1, int(len(decisions) * valid_frac))
    valid_ids = set(decisions[:n_valid])
    return df[~df['decision_id'].isin(valid_ids)].copy(), df[df['decision_id'].isin(valid_ids)].copy()


def decision_top_rows(df, pred_col):
    rows = []
    if df is None or df.empty or pred_col not in df.columns:
        return pd.DataFrame()
    for did, g in df.groupby('decision_id', sort=False):
        gg = g.reset_index(drop=True)
        y = pd.to_numeric(gg.get('is_chosen'), errors='coerce').fillna(0).astype(int).to_numpy()
        if len(y) == 0 or y.sum() <= 0:
            continue
        pred = pd.to_numeric(gg[pred_col], errors='coerce').fillna(-1e9).to_numpy(dtype=float)
        order = list(np.argsort(-pred)) if np is not None else sorted(range(len(pred)), key=lambda i: -pred[i])
        if not order:
            continue
        top_i = int(order[0])
        second = float(pred[order[1]]) if len(order) > 1 else -1e9
        chosen_set = {i for i, v in enumerate(y) if v == 1}
        row0 = gg.iloc[0]
        rows.append({
            'decision_id': did,
            'option_signature': str(row0.get('option_signature', '')),
            'unknown0_turn_bucket': str(row0.get('unknown0_turn_bucket', '')),
            'unknown0_deck_bucket': str(row0.get('unknown0_deck_bucket', '')),
            'unknown0_ko_bucket': str(row0.get('unknown0_ko_bucket', '')),
            'top_option_type': str(gg.get('option_type', pd.Series([''])).iloc[top_i]) if 'option_type' in gg.columns else '',
            'chosen_option_types': '|'.join(gg.loc[list(chosen_set), 'option_type'].astype(str).tolist()) if 'option_type' in gg.columns else '',
            'top_correct': int(top_i in chosen_set),
            'top_score': float(pred[top_i]),
            'margin': float(pred[top_i] - second),
            'num_options': int(len(gg)),
        })
    return pd.DataFrame(rows)


def build_unknown0_calibration(pred_df, pred_col='unknown0_mlp_pred'):
    top = decision_top_rows(pred_df, pred_col)
    if top.empty:
        return pd.DataFrame()
    bins = [-1e9, 0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 1e9]
    labels = ['<=.02', '.02-.05', '.05-.10', '.10-.20', '.20-.35', '.35-.50', '>.50']
    top['margin_bucket'] = pd.cut(top['margin'], bins=bins, labels=labels).astype(str)
    total = max(1, len(top))
    rows = []
    for b, g in top.groupby('margin_bucket', dropna=False):
        rows.append({'margin_bucket': str(b), 'decisions': int(len(g)), 'coverage': float(len(g) / total), 'top1_accuracy': float(g['top_correct'].mean()), 'avg_margin': float(g['margin'].mean())})
    return pd.DataFrame(rows)


def build_unknown0_signature_report(pred_df, margin_threshold):
    if pred_df is None or pred_df.empty:
        return pd.DataFrame()
    rows = []
    for sig, g in pred_df.groupby('option_signature', dropna=False):
        mlp_m = grouped_prediction_metrics(g, 'unknown0_mlp_pred')
        rule_m = grouped_prediction_metrics(g, 'rule_proxy_score') if 'rule_proxy_score' in g.columns else {'decisions': 0}
        bc_m = grouped_prediction_metrics(g[g['bc_pred'].notna()].copy(), 'bc_pred') if 'bc_pred' in g.columns and g['bc_pred'].notna().any() else {'decisions': 0, 'top1': None, 'top3': None, 'avg_chosen_rank': None}
        top = decision_top_rows(g, 'unknown0_mlp_pred')
        high = top[top['margin'] >= margin_threshold] if not top.empty else pd.DataFrame()
        rows.append({
            'option_signature': str(sig),
            'rows': int(len(g)),
            'decisions': int(g['decision_id'].nunique()) if 'decision_id' in g.columns else 0,
            'mlp_top1': mlp_m.get('top1'),
            'mlp_top3': mlp_m.get('top3'),
            'mlp_avg_chosen_rank': mlp_m.get('avg_chosen_rank'),
            'rule_proxy_top1': rule_m.get('top1'),
            'rule_proxy_top3': rule_m.get('top3'),
            'bc_top1': bc_m.get('top1'),
            'high_conf_decisions': int(len(high)),
            'high_conf_coverage': float(len(high) / max(1, len(top))) if not top.empty else None,
            'high_conf_accuracy': float(high['top_correct'].mean()) if len(high) else None,
            'mlp_gain_over_rule': None if mlp_m.get('top1') is None or rule_m.get('top1') is None else float(mlp_m.get('top1') - rule_m.get('top1')),
        })
    return pd.DataFrame(rows).sort_values(['decisions'], ascending=False)


def select_unknown0_policy_signatures(sig_report):
    if sig_report is None or sig_report.empty or not AUTO_SELECT_UNKNOWN0_POLICY_SIGNATURES:
        return set()
    allowed = set()
    for _, r in sig_report.iterrows():
        if int(r.get('decisions', 0) or 0) < UNKNOWN0_POLICY_MIN_SIGNATURE_DECISIONS:
            continue
        gain = r.get('mlp_gain_over_rule')
        acc = r.get('high_conf_accuracy')
        cov = r.get('high_conf_coverage')
        if gain is None or pd.isna(gain) or float(gain) < UNKNOWN0_POLICY_MIN_GAIN_OVER_RULE:
            continue
        if acc is None or pd.isna(acc) or float(acc) < UNKNOWN0_POLICY_MIN_HIGH_CONF_ACC:
            continue
        if cov is None or pd.isna(cov) or float(cov) < UNKNOWN0_POLICY_MIN_HIGH_CONF_COVERAGE:
            continue
        allowed.add(str(r.get('option_signature', '')))
    return allowed


def build_unknown0_assist_report(pred_df, allowed_signatures, margin_threshold):
    if pred_df is None or pred_df.empty or 'unknown0_mlp_pred' not in pred_df.columns or 'rule_proxy_score' not in pred_df.columns:
        return pd.DataFrame()
    out = pred_df.copy()
    out['unknown0_assist_score'] = out['rule_proxy_score']
    for _, g in out.groupby('decision_id', sort=False):
        sig = str(g['option_signature'].iloc[0]) if 'option_signature' in g.columns else ''
        if sig not in allowed_signatures:
            continue
        pred = pd.to_numeric(g['unknown0_mlp_pred'], errors='coerce').fillna(-1e9).to_numpy(dtype=float)
        order = list(np.argsort(-pred)) if np is not None else sorted(range(len(pred)), key=lambda i: -pred[i])
        if not order:
            continue
        margin = float(pred[order[0]] - (pred[order[1]] if len(order) > 1 else -1e9))
        if margin >= margin_threshold:
            out.loc[g.index, 'unknown0_assist_score'] = out.loc[g.index, 'unknown0_mlp_pred']
    rows = []
    for name, col in [('rule_proxy', 'rule_proxy_score'), ('unknown0_mlp', 'unknown0_mlp_pred'), ('rule_proxy_plus_unknown0_mlp', 'unknown0_assist_score')]:
        rows.append({'method': name, **grouped_prediction_metrics(out, col)})
    return pd.DataFrame(rows)


def build_unknown0_policy_table(pred_df, allowed_signatures, margin_threshold):
    top = decision_top_rows(pred_df, 'unknown0_mlp_pred')
    if top.empty:
        return {}, pd.DataFrame(), {'status': 'skipped', 'reason': 'empty_top_rows'}
    top = top[top['option_signature'].isin(allowed_signatures)].copy()
    top = top[top['margin'] >= margin_threshold].copy()
    if top.empty:
        return {}, pd.DataFrame(), {'status': 'skipped', 'reason': 'no_high_conf_allowed_signature_rows'}
    rows = []
    table = {}
    group_cols = ['option_signature', 'unknown0_deck_bucket', 'unknown0_ko_bucket', 'unknown0_turn_bucket']
    for key, g in top.groupby(group_cols, dropna=False):
        if len(g) < UNKNOWN0_POLICY_MIN_BUCKET_DECISIONS:
            continue
        # MC-weighted or plain accuracy (computed first for threshold check)
        if USE_MC_RETURN_WEIGHTS and 'mc_step_weight' in g.columns:
            _sw = g['mc_step_weight'].clip(lower=1e-8)
            acc = float((g['top_correct'] * _sw).sum() / _sw.sum())
            preferred = str(g.groupby('top_option_type')['mc_step_weight'].sum().idxmax())
        else:
            acc = float(g['top_correct'].mean())
            preferred = str(g['top_option_type'].value_counts().idxmax())
        if acc < UNKNOWN0_POLICY_MIN_TABLE_ACC:
            continue
        signature, deck_bucket, ko_bucket, turn_bucket = [str(x) for x in key]
        table_key = f'{signature}||{deck_bucket}||{ko_bucket}||{turn_bucket}'
        table[table_key] = preferred
        rows.append({'table_key': table_key, 'option_signature': signature, 'deck_bucket': deck_bucket, 'ko_bucket': ko_bucket, 'turn_bucket': turn_bucket, 'preferred_option_type': preferred, 'decisions': int(len(g)), 'top_correct_rate': acc, 'avg_margin': float(g['margin'].mean())})
    for sig, g in top.groupby('option_signature', dropna=False):
        if str(sig) not in allowed_signatures or len(g) < UNKNOWN0_POLICY_MIN_SIGNATURE_DECISIONS:
            continue
        acc = float(g['top_correct'].mean())
        if acc < UNKNOWN0_POLICY_MIN_TABLE_ACC:
            continue
        preferred = str(g['top_option_type'].value_counts().idxmax())
        table_key = f'{sig}||*||*||*'
        if table_key not in table:
            table[table_key] = preferred
            rows.append({'table_key': table_key, 'option_signature': str(sig), 'deck_bucket': '*', 'ko_bucket': '*', 'turn_bucket': '*', 'preferred_option_type': preferred, 'decisions': int(len(g)), 'top_correct_rate': acc, 'avg_margin': float(g['margin'].mean())})
    df = pd.DataFrame(rows)
    return table, df, {'status': 'ok' if table else 'empty', 'entries': int(len(table)), 'allowed_signatures': sorted(allowed_signatures), 'margin_threshold': float(margin_threshold), 'min_table_acc': float(UNKNOWN0_POLICY_MIN_TABLE_ACC)}


def filter_unknown0_policy_fit_frame(pred_df: pd.DataFrame, episode_split: pd.DataFrame, excluded_splits=('holdout',)):
    if pred_df is None or pred_df.empty:
        return pd.DataFrame(), {'status': 'empty_pred_df', 'excluded_splits': list(excluded_splits)}
    work = pred_df.copy()
    excluded = {str(x) for x in excluded_splits}
    if episode_split is None or episode_split.empty or 'episode_id' not in episode_split.columns or 'split' not in episode_split.columns or 'episode_id' not in work.columns:
        work['deterministic_split'] = 'unknown'
        summary = {
            'status': 'no_episode_split_join',
            'excluded_splits': list(excluded_splits),
            'input_rows': int(len(work)),
            'fit_rows': int(len(work)),
            'excluded_rows': 0,
            'fit_decisions': int(work['decision_id'].nunique()) if 'decision_id' in work.columns else 0,
            'split_rows': {'unknown': int(len(work))},
            'split_decisions': {'unknown': int(work['decision_id'].nunique()) if 'decision_id' in work.columns else 0},
        }
        return work, summary
    meta = episode_split[['episode_id', 'split']].drop_duplicates('episode_id').copy()
    meta['episode_id'] = meta['episode_id'].astype(str)
    work['episode_id'] = work['episode_id'].astype(str)
    work = work.merge(meta.rename(columns={'split': 'deterministic_split'}), on='episode_id', how='left')
    work['deterministic_split'] = work['deterministic_split'].astype('string').fillna('unknown').astype(str)
    fit = work[~work['deterministic_split'].isin(excluded)].copy()
    split_rows = work['deterministic_split'].value_counts(dropna=False).sort_index().to_dict()
    if 'decision_id' in work.columns:
        split_decisions = work.groupby('deterministic_split', dropna=False)['decision_id'].nunique().sort_index().to_dict()
    else:
        split_decisions = {}
    summary = {
        'status': 'ok' if not fit.empty else 'empty_after_split_filter',
        'excluded_splits': sorted(excluded),
        'input_rows': int(len(work)),
        'fit_rows': int(len(fit)),
        'excluded_rows': int(len(work) - len(fit)),
        'input_decisions': int(work['decision_id'].nunique()) if 'decision_id' in work.columns else 0,
        'fit_decisions': int(fit['decision_id'].nunique()) if 'decision_id' in fit.columns else 0,
        'excluded_decisions': int(work.loc[work['deterministic_split'].isin(excluded), 'decision_id'].nunique()) if 'decision_id' in work.columns else 0,
        'split_rows': {str(k): int(v) for k, v in split_rows.items()},
        'split_decisions': {str(k): int(v) for k, v in split_decisions.items()},
    }
    return fit, summary


def make_unknown0_policy_submission_agent_source(base_source, policy_table, enabled=True):
    if not enabled or not policy_table:
        return base_source
    table_literal = repr({str(k): str(v) for k, v in policy_table.items()})
    helper = f'''
# v0-05d1 UNKNOWN_0 policy-table assist, distilled from offline PyTorch MLP.
USE_ABSTRACT_OPTION_SIGNATURE = True
USE_UNKNOWN0_POLICY_TABLE = True
UNKNOWN0_POLICY_TABLE = {table_literal}
UNKNOWN0_POLICY_STATS = {{
    "calls": 0, "unknown0_context": 0, "eligible": 0,
    "key_hit": 0, "signature_fallback_hit": 0, "miss": 0,
    "no_candidate": 0, "selected": 0,
}}
UNKNOWN0_POLICY_SELECTED_OPTION_TYPE_COUNTS = {{}}
UNKNOWN0_POLICY_HIT_KEY_COUNTS = {{}}


def _unknown0_policy_stat_inc(key, amount=1):
    try:
        UNKNOWN0_POLICY_STATS[key] = UNKNOWN0_POLICY_STATS.get(key, 0) + amount
    except Exception:
        pass


def _unknown0_policy_count_dict(d, key, amount=1):
    try:
        d[key] = d.get(key, 0) + amount
    except Exception:
        pass


def _unknown0_policy_ctx_identifiers(context):
    ids = []
    for attr in ("name", "value"):
        try:
            v = getattr(context, attr, None)
            if v is not None:
                ids.append(str(v))
        except Exception:
            pass
    try:
        s = str(context)
        ids.append(s)
        if "." in s:
            ids.append(s.split(".")[-1])
    except Exception:
        pass
    return set(ids)


def _unknown0_policy_is_context(context):
    ids = _unknown0_policy_ctx_identifiers(context)
    return bool(ids.intersection({{"UNKNOWN_0", "UNKNOWN", "0"}}))


def _unknown0_policy_option_type_identifiers(option):
    ids = []
    t = getattr(option, "type", None)
    for attr in ("name", "value"):
        try:
            v = getattr(t, attr, None)
            if v is not None:
                ids.append(str(v))
        except Exception:
            pass
    try:
        s = str(t)
        ids.append(s)
        if "." in s:
            ids.append(s.split(".")[-1])
    except Exception:
        pass
    return set(x.upper() if x.isalpha() else x for x in ids if x is not None and str(x) != "")


def _unknown0_policy_signature_forms(select):
    # Build multiple forms so dataset numeric option types like 12|13|14 and runtime enum names can both hit.
    per_option = [_unknown0_policy_option_type_identifiers(o) for o in select.option]
    forms = set()
    for pick in ("name", "value", "str"):
        vals = []
        for o in select.option:
            t = getattr(o, "type", None)
            v = None
            try:
                if pick == "name":
                    v = getattr(t, "name", None)
                elif pick == "value":
                    v = getattr(t, "value", None)
                else:
                    v = str(t).split(".")[-1]
            except Exception:
                v = None
            if v is not None:
                vals.append(str(v).upper() if str(v).isalpha() else str(v))
        if vals:
            forms.add("|".join(sorted(set(vals))))
    # also a broad merged form as last resort
    merged = []
    for ids in per_option:
        if ids:
            # prefer numeric id if present to match observed UNKNOWN_0 training signatures
            nums = sorted([x for x in ids if str(x).isdigit()])
            merged.append(nums[0] if nums else sorted(ids)[0])
    if merged:
        forms.add("|".join(sorted(set(merged))))
    return forms


def _unknown0_policy_abstract_sig(select):
    KEYWORDS = set(["END", "YES", "NO", "NUMBER"])
    per_option = [_unknown0_policy_option_type_identifiers(o) for o in select.option]
    canonical = set()
    for ids in per_option:
        nums = sorted([x for x in ids if x.lstrip("-").isdigit()])
        if nums:
            canonical.add(nums[0])
        else:
            names = sorted([x for x in ids if x.isalpha()])
            if names:
                canonical.add(names[0].upper())
    keywords_found = canonical & KEYWORDS
    n_numeric = len([t for t in canonical if t.lstrip("-").isdigit()])
    n_bucket = "N" + ("0" if n_numeric == 0 else "1" if n_numeric == 1 else "2" if n_numeric == 2 else "3p")
    parts = [n_bucket]
    for kw in ["END", "YES", "NO", "NUMBER"]:
        if kw in keywords_found:
            parts.append(kw)
    return "|".join(sorted(parts))


def _unknown0_policy_select(select, scores, context, turn, deckout_risk_strict, deckout_risk, can_win_this_turn, target_can_kill, desc_indices):
    _unknown0_policy_stat_inc("calls")
    if not USE_UNKNOWN0_POLICY_TABLE or not _unknown0_policy_is_context(context):
        return None
    _unknown0_policy_stat_inc("unknown0_context")
    try:
        if int(select.minCount) != 1 or int(select.maxCount) != 1:
            return None
    except Exception:
        return None
    _unknown0_policy_stat_inc("eligible")
    deck_bucket = "strict" if deckout_risk_strict else "risk" if deckout_risk else "safe"
    ko_bucket = "ko" if (target_can_kill or can_win_this_turn) else "no_ko"
    turn_bucket = "early" if int(turn) <= 2 else "mid" if int(turn) <= 5 else "late"
    preferred = None
    hit_key = None
    signature_fallback = False
    _use_abs = globals().get("USE_ABSTRACT_OPTION_SIGNATURE", False)
    if _use_abs:
        sig = _unknown0_policy_abstract_sig(select)
        for k in [
            f"{{sig}}||{{deck_bucket}}||{{ko_bucket}}||{{turn_bucket}}",
            f"{{sig}}||{{deck_bucket}}||{{ko_bucket}}||*",
            f"{{sig}}||{{deck_bucket}}||*||*",
            f"{{sig}}||*||*||*",
        ]:
            if k in UNKNOWN0_POLICY_TABLE:
                preferred = UNKNOWN0_POLICY_TABLE[k]
                hit_key = k
                signature_fallback = k.endswith("||*||*||*")
                break
    else:
        sig_forms = _unknown0_policy_signature_forms(select)
        for sig in sorted(sig_forms):
            for k in [
                f"{{sig}}||{{deck_bucket}}||{{ko_bucket}}||{{turn_bucket}}",
                f"{{sig}}||{{deck_bucket}}||{{ko_bucket}}||*",
                f"{{sig}}||{{deck_bucket}}||*||*",
                f"{{sig}}||*||*||*",
            ]:
                if k in UNKNOWN0_POLICY_TABLE:
                    preferred = UNKNOWN0_POLICY_TABLE[k]
                    hit_key = k
                    signature_fallback = k.endswith("||*||*||*")
                    break
            if preferred is not None:
                break
    if preferred is None:
        _unknown0_policy_stat_inc("miss")
        return None
    _unknown0_policy_stat_inc("key_hit")
    if signature_fallback:
        _unknown0_policy_stat_inc("signature_fallback_hit")
    _unknown0_policy_count_dict(UNKNOWN0_POLICY_HIT_KEY_COUNTS, hit_key)
    candidates = []
    preferred_s = str(preferred).upper() if str(preferred).isalpha() else str(preferred)
    for i, o in enumerate(select.option):
        if preferred_s in _unknown0_policy_option_type_identifiers(o):
            candidates.append(i)
    if not candidates:
        _unknown0_policy_stat_inc("no_candidate")
        return None
    selected_i = max(candidates, key=lambda i: scores[i] if i < len(scores) else -10**18)
    _unknown0_policy_stat_inc("selected")
    _unknown0_policy_count_dict(UNKNOWN0_POLICY_SELECTED_OPTION_TYPE_COUNTS, preferred_s)
    return selected_i
'''
    marker = '\ndef agent(obs_dict: dict) -> list[int]:'
    if marker not in base_source:
        raise ValueError('Could not locate agent() for UNKNOWN0 policy injection')
    src = base_source.replace(marker, helper + marker, 1)
    target = """    if max_count <= 0:\n        return []\n    threshold = context_threshold(context, safe_draws, can_win_this_turn, opponent_archetype, field_counts[Alakazam] > 0)"""
    insert = """    if max_count <= 0:\n        return []\n    unknown0_policy_selected = _unknown0_policy_select(\n        select, scores, context, state.turn, deckout_risk_strict, deckout_risk,\n        can_win_this_turn, target_can_kill, desc_indices\n    ) if 'USE_UNKNOWN0_POLICY_TABLE' in globals() else None\n    if unknown0_policy_selected is not None:\n        selected = safe_unique_action([unknown0_policy_selected], len(select.option), min_count, max_count)\n        if len(selected) >= min_count:\n            return selected\n    threshold = context_threshold(context, safe_draws, can_win_this_turn, opponent_archetype, field_counts[Alakazam] > 0)"""
    if target not in src:
        raise ValueError('Could not locate final selection block for UNKNOWN0 policy injection')
    src = src.replace(target, insert, 1)
    compile(src, 'unknown0_policy_main.py', 'exec')
    assert_no_import_time_deck_io(src, 'UNKNOWN0_POLICY_MAIN_SOURCE')
    if 'import torch' in src:
        raise AssertionError('UNKNOWN0 policy submission source must not import torch')
    return src


UNKNOWN0_FEATURE_SAFETY = assert_unknown0_feature_safety(UNKNOWN0_NUMERIC_FEATURES, UNKNOWN0_CATEGORICAL_FEATURES)
# v05d18: LightGBM with card IDs as categorical features
UNKNOWN0_LGBM_CAT_OVERRIDES = [
    'card_id', 'my_active_id', 'op_active_id',
    'attack_id', 'target_card_id', 'stadium_id',
]
UNKNOWN0_LGBM_ALL_CAT = list(dict.fromkeys(
    UNKNOWN0_LGBM_CAT_OVERRIDES + UNKNOWN0_CATEGORICAL_FEATURES))
# Exclude features that can't be reliably computed in inference
UNKNOWN0_LGBM_EXCLUDE_FEATURES = ['turn_action_count']

try:
    import lightgbm as _lgb_mod
    _LGBM_OK = True
except Exception as _lgbm_import_err:
    _LGBM_OK = False
    print(f'LightGBM import failed: {_lgbm_import_err}')

if not _LGBM_OK:
    UNKNOWN0_MLP_REPORT = {'status': 'skipped', 'reason': 'lgbm_unavailable'}
elif 'ALAKAZAM_OPTION_MODEL_DF' not in globals() or ALAKAZAM_OPTION_MODEL_DF.empty:
    UNKNOWN0_MLP_REPORT = {'status': 'skipped', 'reason': 'empty_option_model_df'}
else:
    try:
        max_rows = int(os.environ.get('V05_UNKNOWN0_MLP_MAX_ROWS',
                       str(min(350_000, max(50_000, MLP_MAX_ROWS // 2)))))
        work = prepare_unknown0_training_frame(ALAKAZAM_OPTION_MODEL_DF, max_rows=max_rows)
        # Extra features from ALAKAZAM_OPTION_MODEL_DF
        _extra_numeric_candidates = ['supporter_played', 'stadium_played', 'energy_attached',
                                      'my_discard_count', 'op_discard_count']
        _extra_numeric_avail = [f for f in _extra_numeric_candidates if f in work.columns]
        print(f'Extra numeric features available: {_extra_numeric_avail}')

        if work.empty or work['is_chosen'].nunique() < 2 or work['decision_id'].nunique() < 50:
            UNKNOWN0_MLP_REPORT = {'status': 'skipped', 'reason': 'insufficient_unknown0_rows',
                                    'rows': int(len(work))}
        else:
            train_df, valid_df = split_by_decision_safe(work, valid_frac=0.15)
            # Winner-weighted training: winner decisions get 2x weight
            if 'won' in train_df.columns:
                _winner_weight = 4.0
                train_df = train_df.copy()
                train_df['_win_wt'] = train_df['won'].fillna(0).astype(float) * (_winner_weight - 1.0) + 1.0
                _n_winner = int((train_df['won'] == True).sum())
                print(f'Winner-weighted training: {_n_winner}/{len(train_df)} winner rows (weight={_winner_weight}x)')
            else:
                train_df = train_df.copy()
                train_df['_win_wt'] = 1.0
                print('won column not found, using uniform weights')
            if train_df.empty or valid_df.empty or train_df['is_chosen'].nunique() < 2:
                UNKNOWN0_MLP_REPORT = {'status': 'skipped', 'reason': 'bad_unknown0_split'}
            else:
                # Build categorical encoding maps (string→int only for string columns;
                # integer card-IDs are passed directly as ints to LightGBM categorical)
                _cat_maps = {}
                for _col in UNKNOWN0_LGBM_ALL_CAT:
                    if _col in train_df.columns and train_df[_col].dtype == object:
                        _uniq = sorted(set(
                            train_df[_col].fillna('UNK').astype(str).unique().tolist() +
                            valid_df[_col].fillna('UNK').astype(str).unique().tolist()))
                        if 'UNK' not in _uniq:
                            _uniq = ['UNK'] + _uniq
                        else:
                            _uniq = ['UNK'] + [v for v in _uniq if v != 'UNK']
                        _cat_maps[_col] = {v: i for i, v in enumerate(_uniq)}

                _all_feats = [f for f in UNKNOWN0_NUMERIC_FEATURES + UNKNOWN0_CATEGORICAL_FEATURES
                              if f not in UNKNOWN0_LGBM_EXCLUDE_FEATURES]
                # Add extra features that are available in the training frame
                _all_feats = _all_feats + [f for f in _extra_numeric_avail if f not in _all_feats]
                _lgbm_prep = {
                    'feature_names': _all_feats,
                    'cat_features': UNKNOWN0_LGBM_ALL_CAT,
                    'cat_maps': _cat_maps,
                    'exclude_features': UNKNOWN0_LGBM_EXCLUDE_FEATURES,
                    'extra_numeric': _extra_numeric_avail,
                }

                def _encode_for_lgbm(df, prep):
                    feats = prep['feature_names']
                    cm = prep['cat_maps']
                    cats = set(prep['cat_features'])
                    out = {}
                    for c in feats:
                        if c in cm:
                            out[c] = df[c].fillna('UNK').astype(str).map(
                                cm[c]).fillna(0).astype('int32')
                        elif c in cats:
                            out[c] = df[c].fillna(0).astype('int32')
                        else:
                            out[c] = df[c].fillna(0).astype('float32')
                    return pd.DataFrame(out, index=df.index)

                X_tr = _encode_for_lgbm(train_df, _lgbm_prep)
                X_va = _encode_for_lgbm(valid_df, _lgbm_prep)
                y_tr = train_df['is_chosen'].astype(int)
                y_va = valid_df['is_chosen'].astype(int)

                # Group sizes for LambdaRank: number of options per decision
                def _make_groups(df):
                    return df.groupby('decision_id', sort=False).size().values

                _g_tr = _make_groups(train_df)
                _g_va = _make_groups(valid_df)
                print(f'  LambdaRank groups: train={len(_g_tr)} decisions, valid={len(_g_va)} decisions')

                # Aggregate win weights to group level (mean per decision)
                _w_wt_tr = (train_df.groupby('decision_id', sort=False)['_win_wt']
                            .transform('mean').values.astype(float)
                            if '_win_wt' in train_df.columns else None)
                _cat_cols = [c for c in UNKNOWN0_LGBM_ALL_CAT if c in X_tr.columns]
                _dtr = _lgb_mod.Dataset(
                    X_tr, label=y_tr, group=_g_tr, weight=_w_wt_tr,
                    categorical_feature=_cat_cols, free_raw_data=False)
                _dva = _lgb_mod.Dataset(
                    X_va, label=y_va, group=_g_va, reference=_dtr,
                    categorical_feature=_cat_cols)

                _lgbm_params = {
                    'objective': 'lambdarank',
                    'metric': 'ndcg',
                    'ndcg_eval_at': [1, 3],
                    'learning_rate': 0.05,
                    'num_leaves': 63,
                    'min_child_samples': 10,
                    'feature_fraction': 0.85,
                    'bagging_fraction': 0.85,
                    'bagging_freq': 5,
                    'verbosity': -1,
                    'n_jobs': -1,
                }
                _u0_lgbm = _lgb_mod.train(
                    _lgbm_params, _dtr,
                    num_boost_round=500,
                    valid_sets=[_dtr, _dva],
                    valid_names=['train', 'valid'],
                    callbacks=[
                        _lgb_mod.early_stopping(50, verbose=False),
                        _lgb_mod.log_evaluation(50),
                    ],
                )
                print(f'unknown0 lgbm: best_iter={_u0_lgbm.best_iteration}')

                # Predict on validation set
                _lgbm_pred = _u0_lgbm.predict(X_va).astype(float)
                pred_df = valid_df.copy()
                pred_df['unknown0_mlp_pred'] = _lgbm_pred  # keep col name for downstream compat
                pred_df['rule_proxy_score'] = offline_rule_proxy_scores(pred_df)
                pred_df['bc_pred'] = predict_with_bc_models(pred_df) if 'predict_with_bc_models' in globals() else np.nan
                UNKNOWN0_MLP_VALID_PRED_DF = pred_df

                UNKNOWN0_POLICY_FIT_PRED_DF, UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY = filter_unknown0_policy_fit_frame(
                    pred_df, EPISODE_SPLIT_DF, excluded_splits=('holdout',))

                # Early-game filter (same as MLP path)
                _egf = globals().get('EARLY_GAME_ONLY_POLICY_FIT', False)
                if _egf and not UNKNOWN0_POLICY_FIT_PRED_DF.empty:
                    _ep_ns = EPISODE_SPLIT_DF[['episode_id','n_steps']].drop_duplicates('episode_id') if 'n_steps' in EPISODE_SPLIT_DF.columns else None
                    _fc = UNKNOWN0_POLICY_FIT_PRED_DF.copy()
                    if _ep_ns is not None and not _ep_ns.empty and 'episode_id' in _fc.columns and 'step' in _fc.columns:
                        _fc['episode_id'] = _fc['episode_id'].astype(str)
                        _ep_ns = _ep_ns.copy(); _ep_ns['episode_id'] = _ep_ns['episode_id'].astype(str)
                        _fc = _fc.merge(_ep_ns, on='episode_id', how='left')
                        _fc['_step_frac'] = _fc['step'].astype(float) / _fc['n_steps'].astype(float).clip(lower=1)
                        _early = _fc[_fc['_step_frac'] <= 2/3].drop(columns=['_step_frac','n_steps'], errors='ignore')
                        if len(_early) >= 20:
                            print(f'[early_game_filter] kept {len(_early)}/{len(UNKNOWN0_POLICY_FIT_PRED_DF)} rows (step_frac<=2/3)')
                            UNKNOWN0_POLICY_FIT_PRED_DF = _early
                        else:
                            print('[early_game_filter] too few early rows, using full dataset')
                    else:
                        print('[early_game_filter] missing step/n_steps columns, skipping filter')

                UNKNOWN0_MLP_SIGNATURE_REPORT_DF = build_unknown0_signature_report(
                    pred_df, UNKNOWN0_MLP_MARGIN_THRESHOLD)
                UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF = build_unknown0_signature_report(
                    UNKNOWN0_POLICY_FIT_PRED_DF, UNKNOWN0_MLP_MARGIN_THRESHOLD)
                allowed = select_unknown0_policy_signatures(UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF)
                UNKNOWN0_MLP_CALIBRATION_DF = build_unknown0_calibration(pred_df, 'unknown0_mlp_pred')
                UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF = build_unknown0_assist_report(
                    pred_df, allowed, UNKNOWN0_MLP_MARGIN_THRESHOLD)
                UNKNOWN0_POLICY_TABLE, UNKNOWN0_POLICY_TABLE_DF, UNKNOWN0_POLICY_SUMMARY =                     build_unknown0_policy_table(UNKNOWN0_POLICY_FIT_PRED_DF, allowed,
                                                UNKNOWN0_MLP_MARGIN_THRESHOLD)
                UNKNOWN0_POLICY_SUMMARY['fit_split_filter'] = UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY

                try:
                    from sklearn.metrics import roc_auc_score
                    _yva_arr = y_va.to_numpy() if hasattr(y_va, 'to_numpy') else y_va
                    row_auc = float(roc_auc_score(_yva_arr, _lgbm_pred)) if len(set(_yva_arr.tolist())) == 2 else None
                    # NDCG@1 = decision-level top1 accuracy under the ranking model
                    row_logloss = None  # not applicable for ranking
                except Exception:
                    row_auc = None
                    row_logloss = None

                # Save LightGBM model + prep
                _lgbm_txt_path = MODEL_DIR / f'{RUN_PREFIX}-unknown0_lgbm_reranker.txt'
                _lgbm_prep_path = MODEL_DIR / f'{RUN_PREFIX}-unknown0_lgbm_prep.json'
                _u0_lgbm.save_model(str(_lgbm_txt_path))
                with open(_lgbm_prep_path, 'w') as _jf:
                    json.dump(_lgbm_prep, _jf, ensure_ascii=False)

                UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS = {
                    'status': 'ok',
                    'model': str(_lgbm_txt_path),
                    'prep': str(_lgbm_prep_path),
                    'best_iteration': int(_u0_lgbm.best_iteration),
                }

                ARTIFACT_PATHS['unknown0_lgbm_model'] = str(_lgbm_txt_path)
                ARTIFACT_PATHS['unknown0_lgbm_prep'] = str(_lgbm_prep_path)
                ARTIFACT_PATHS['unknown0_mlp_valid_predictions'] = safe_save_table(
                    UNKNOWN0_MLP_VALID_PRED_DF, OUTPUT_DIR / 'unknown0_mlp_valid_predictions')
                ARTIFACT_PATHS['unknown0_mlp_signature_report'] = safe_save_table(
                    UNKNOWN0_MLP_SIGNATURE_REPORT_DF, OUTPUT_DIR / 'unknown0_mlp_signature_report')
                ARTIFACT_PATHS['unknown0_mlp_calibration'] = safe_save_table(
                    UNKNOWN0_MLP_CALIBRATION_DF, OUTPUT_DIR / 'unknown0_mlp_calibration')
                ARTIFACT_PATHS['unknown0_policy_fit_signature_report'] = safe_save_table(
                    UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF, OUTPUT_DIR / 'unknown0_policy_fit_signature_report')
                ARTIFACT_PATHS['unknown0_rule_proxy_assist_report'] = safe_save_table(
                    UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF, OUTPUT_DIR / 'unknown0_rule_proxy_assist_report')
                ARTIFACT_PATHS['unknown0_policy_table'] = safe_save_table(
                    UNKNOWN0_POLICY_TABLE_DF, OUTPUT_DIR / 'unknown0_policy_table')
                ARTIFACT_PATHS['unknown0_policy_table_json'] = write_json(
                    OUTPUT_DIR / 'unknown0_policy_table.json', UNKNOWN0_POLICY_TABLE)

                UNKNOWN0_MLP_REPORT = {
                    'status': 'ok',
                    'model': 'lightgbm_lambdarank_winner_wt4',
                    'rows': int(len(work)),
                    'train_rows': int(len(train_df)),
                    'valid_rows': int(len(valid_df)),
                    'train_decisions': int(train_df['decision_id'].nunique()),
                    'valid_decisions': int(valid_df['decision_id'].nunique()),
                    'best_iteration': int(_u0_lgbm.best_iteration),
                    'num_leaves': int(_lgbm_params['num_leaves']),
                    'row_auc': row_auc,
                    'row_logloss': row_logloss,
                    'unknown0_lgbm_decision_metrics': grouped_prediction_metrics(pred_df, 'unknown0_mlp_pred'),
                    'rule_proxy_decision_metrics': grouped_prediction_metrics(pred_df, 'rule_proxy_score'),
                    'bc_decision_metrics': grouped_prediction_metrics(
                        pred_df[pred_df['bc_pred'].notna()].copy(), 'bc_pred')
                        if 'bc_pred' in pred_df.columns and pred_df['bc_pred'].notna().any() else {'decisions': 0},
                    'feature_safety': UNKNOWN0_FEATURE_SAFETY,
                    'extra_numeric_used': _extra_numeric_avail,
                    'allowed_signatures': sorted(allowed),
                    'policy_table_entries': int(len(UNKNOWN0_POLICY_TABLE)),
                    'policy_fit_split_summary': UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY,
                    'direct_mlp_packaging_status': UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS,
                    'feature_importance_top10': dict(sorted(
                        zip(_u0_lgbm.feature_name(),
                            _u0_lgbm.feature_importance(importance_type='gain').tolist()),
                        key=lambda x: -x[1])[:10]),
                }
                print('unknown0 lgbm report:', UNKNOWN0_MLP_REPORT)
                if not UNKNOWN0_MLP_SIGNATURE_REPORT_DF.empty:
                    display(UNKNOWN0_MLP_SIGNATURE_REPORT_DF.head(20)) if 'display' in globals() else print(UNKNOWN0_MLP_SIGNATURE_REPORT_DF.head(20))
                if not UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF.empty:
                    display(UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF) if 'display' in globals() else print(UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF)
    except Exception as exc:
        UNKNOWN0_MLP_REPORT = {'status': 'error', 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
        log_error('unknown0_lgbm', error=repr(exc), traceback=traceback.format_exc(limit=8))

ARTIFACT_PATHS['unknown0_mlp_report'] = write_json(OUTPUT_DIR / 'unknown0_mlp_report.json', UNKNOWN0_MLP_REPORT)
ARTIFACT_PATHS['unknown0_policy_summary'] = write_json(OUTPUT_DIR / 'unknown0_policy_summary.json', UNKNOWN0_POLICY_SUMMARY)
ARTIFACT_PATHS['unknown0_policy_fit_split_summary'] = write_json(OUTPUT_DIR / 'unknown0_policy_fit_split_summary.json', UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY)
ARTIFACT_PATHS['unknown0_direct_mlp_packaging_status'] = write_json(OUTPUT_DIR / 'unknown0_direct_mlp_packaging_status.json', UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS)
print('UNKNOWN0_MLP_REPORT status:', UNKNOWN0_MLP_REPORT.get('status'))
print('UNKNOWN0_POLICY_SUMMARY:', UNKNOWN0_POLICY_SUMMARY)


# Split-aware UNKNOWN_0 policy diagnostics. Reporting-only; does not change submission behavior.
def _unknown0_policy_eval_keys(row: pd.Series) -> list[str]:
    sig = str(row.get('option_signature', ''))
    deck_bucket = str(row.get('unknown0_deck_bucket', ''))
    ko_bucket = str(row.get('unknown0_ko_bucket', ''))
    turn_bucket = str(row.get('unknown0_turn_bucket', ''))
    return [
        f'{sig}||{deck_bucket}||{ko_bucket}||{turn_bucket}',
        f'{sig}||{deck_bucket}||{ko_bucket}||*',
        f'{sig}||{deck_bucket}||*||*',
        f'{sig}||*||*||*',
    ]


def build_unknown0_policy_decision_eval(pred_df: pd.DataFrame, episode_split: pd.DataFrame, policy_table: dict) -> pd.DataFrame:
    cols = [
        'decision_id', 'episode_id', 'split', 'matchup', 'tier', 'won', 'num_options',
        'option_signature', 'table_key', 'preferred_option_type', 'policy_hit',
        'policy_correct', 'rule_top_correct', 'mlp_top_correct', 'bc_top_correct',
        'unknown0_mlp_margin', 'top_option_type', 'chosen_option_type',
        'stress_long_game', 'stress_top200_tier',
    ]
    if pred_df is None or pred_df.empty:
        return pd.DataFrame(columns=cols)
    top = decision_top_rows(pred_df, 'unknown0_mlp_pred')
    if top.empty:
        return pd.DataFrame(columns=cols)
    rule_top = decision_top_rows(pred_df, 'rule_proxy_score') if 'rule_proxy_score' in pred_df.columns else pd.DataFrame()
    bc_src = pred_df[pred_df['bc_pred'].notna()].copy() if 'bc_pred' in pred_df.columns and pred_df['bc_pred'].notna().any() else pd.DataFrame()
    bc_top = decision_top_rows(bc_src, 'bc_pred') if not bc_src.empty else pd.DataFrame()
    meta_cols = [c for c in ['decision_id', 'episode_id', 'matchup', 'tier', 'won'] if c in pred_df.columns]
    if 'decision_id' in meta_cols:
        meta = pred_df[meta_cols].drop_duplicates('decision_id').copy()
        work = top.merge(meta, on='decision_id', how='left')
    else:
        work = top.copy()
        work['episode_id'] = ''
    work['episode_id'] = work.get('episode_id', pd.Series('', index=work.index)).astype(str)
    if episode_split is not None and not episode_split.empty:
        split_cols = [c for c in ['episode_id', 'split', 'stress_long_game', 'stress_top200_tier'] if c in episode_split.columns]
        split_work = episode_split[split_cols].copy()
        split_work['episode_id'] = split_work['episode_id'].astype(str)
        work = work.merge(split_work, on='episode_id', how='left')
    else:
        work['split'] = 'unknown'
        work['stress_long_game'] = False
        work['stress_top200_tier'] = False
    rule_map = rule_top.set_index('decision_id')['top_correct'].to_dict() if not rule_top.empty else {}
    bc_map = bc_top.set_index('decision_id')['top_correct'].to_dict() if not bc_top.empty else {}
    chosen_type_map = pred_df[pred_df['is_chosen'].astype(bool)].drop_duplicates('decision_id').set_index('decision_id')['option_type'].astype(str).to_dict() if 'is_chosen' in pred_df.columns else {}
    records = []
    for _, r in work.iterrows():
        table_key = ''
        preferred = ''
        for k in _unknown0_policy_eval_keys(r):
            if k in policy_table:
                table_key = k
                preferred = str(policy_table[k])
                break
        policy_hit = bool(table_key)
        chosen_type = str(chosen_type_map.get(r['decision_id'], ''))
        records.append({
            'decision_id': r['decision_id'],
            'episode_id': r.get('episode_id'),
            'split': r.get('split', 'unknown'),
            'matchup': r.get('matchup'),
            'tier': r.get('tier'),
            'won': r.get('won'),
            'num_options': r.get('num_options'),
            'option_signature': r.get('option_signature'),
            'table_key': table_key,
            'preferred_option_type': preferred,
            'policy_hit': policy_hit,
            'policy_correct': bool(policy_hit and preferred == chosen_type),
            'rule_top_correct': rule_map.get(r['decision_id']),
            'mlp_top_correct': bool(r.get('top_correct')),
            'bc_top_correct': bc_map.get(r['decision_id']),
            'unknown0_mlp_margin': r.get('margin'),
            'top_option_type': r.get('top_option_type'),
            'chosen_option_type': chosen_type,
            'stress_long_game': bool(r.get('stress_long_game', False)),
            'stress_top200_tier': bool(r.get('stress_top200_tier', False)),
        })
    return pd.DataFrame(records, columns=cols)


def _aggregate_unknown0_policy_eval(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    rows = []
    for key, g in df.groupby(group_cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        hit = g[g['policy_hit'].astype(bool)]
        row = {c: v for c, v in zip(group_cols, key)}
        margin_s = pd.to_numeric(g.get('unknown0_mlp_margin'), errors='coerce')
        num_options_s = pd.to_numeric(g.get('num_options'), errors='coerce')
        valid_margin = margin_s[(num_options_s > 1) & (margin_s < 1e8)]
        row.update({
            'decisions': int(len(g)),
            'episodes': int(g['episode_id'].nunique()),
            'policy_hit_decisions': int(len(hit)),
            'single_option_decisions': int((num_options_s <= 1).sum()),
            'policy_hit_rate': float(len(hit) / max(1, len(g))),
            'policy_correct_rate_on_hits': float(hit['policy_correct'].mean()) if len(hit) else None,
            'mlp_top1': float(g['mlp_top_correct'].mean()) if 'mlp_top_correct' in g else None,
            'rule_top1': float(pd.to_numeric(g['rule_top_correct'], errors='coerce').mean()) if 'rule_top_correct' in g else None,
            'bc_top1': float(pd.to_numeric(g['bc_top_correct'], errors='coerce').mean()) if 'bc_top_correct' in g else None,
            'avg_margin': float(valid_margin.mean()) if len(valid_margin) else None,
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


def build_unknown0_policy_split_eval(decision_eval: pd.DataFrame) -> pd.DataFrame:
    return _aggregate_unknown0_policy_eval(decision_eval, ['split'])


def build_unknown0_policy_stress_eval(decision_eval: pd.DataFrame) -> pd.DataFrame:
    if decision_eval is None or decision_eval.empty:
        return pd.DataFrame()
    parts = []
    specs = [
        ('all_unknown0', pd.Series(True, index=decision_eval.index)),
        ('long_game', decision_eval['stress_long_game'].astype(bool) if 'stress_long_game' in decision_eval else pd.Series(False, index=decision_eval.index)),
        ('top200_tier', decision_eval['stress_top200_tier'].astype(bool) if 'stress_top200_tier' in decision_eval else pd.Series(False, index=decision_eval.index)),
        ('policy_hit', decision_eval['policy_hit'].astype(bool) if 'policy_hit' in decision_eval else pd.Series(False, index=decision_eval.index)),
    ]
    for name, mask in specs:
        sub = decision_eval.loc[mask].copy()
        if sub.empty:
            continue
        agg = _aggregate_unknown0_policy_eval(sub, ['split'])
        if not agg.empty:
            agg.insert(0, 'stress_slice', name)
            parts.append(agg)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


UNKNOWN0_POLICY_DECISION_EVAL_DF = build_unknown0_policy_decision_eval(UNKNOWN0_MLP_VALID_PRED_DF, EPISODE_SPLIT_DF, UNKNOWN0_POLICY_TABLE)
UNKNOWN0_POLICY_SPLIT_EVAL_DF = build_unknown0_policy_split_eval(UNKNOWN0_POLICY_DECISION_EVAL_DF)
UNKNOWN0_POLICY_STRESS_EVAL_DF = build_unknown0_policy_stress_eval(UNKNOWN0_POLICY_DECISION_EVAL_DF)
ARTIFACT_PATHS['unknown0_policy_decision_eval'] = safe_save_table(UNKNOWN0_POLICY_DECISION_EVAL_DF, OUTPUT_DIR / 'unknown0_policy_decision_eval')
ARTIFACT_PATHS['unknown0_policy_split_eval'] = safe_save_table(UNKNOWN0_POLICY_SPLIT_EVAL_DF, OUTPUT_DIR / 'unknown0_policy_split_eval')
ARTIFACT_PATHS['unknown0_policy_stress_eval'] = safe_save_table(UNKNOWN0_POLICY_STRESS_EVAL_DF, OUTPUT_DIR / 'unknown0_policy_stress_eval')
print('unknown0 policy decision eval:', UNKNOWN0_POLICY_DECISION_EVAL_DF.shape)
print('unknown0 policy split eval:', UNKNOWN0_POLICY_SPLIT_EVAL_DF.shape)
print('unknown0 policy stress eval:', UNKNOWN0_POLICY_STRESS_EVAL_DF.shape)


# ── v05d18: UNKNOWN_0 LightGBM direct submission source builder ──────────────────
_LGBM_INJ_CODE = '\n# v0-05d18: UNKNOWN_0 LightGBM inference. Falls back to policy table on failure.\n\n_U0_LGBM = None\n\ndef _u0_load_lgbm():\n    global _U0_LGBM\n    try:\n        import os as _os, lightgbm as _lgb, json as _json\n        _d = _os.path.dirname(_os.path.abspath(__file__))\n        _mp = _os.path.join(_d, \'unknown0_lgbm.txt\')\n        _pp = _os.path.join(_d, \'unknown0_lgbm_prep.json\')\n        if not _os.path.exists(_mp) or not _os.path.exists(_pp):\n            return\n        _model = _lgb.Booster(model_file=_mp)\n        with open(_pp) as _f:\n            _prep = _json.load(_f)\n        _U0_LGBM = {\'model\': _model, \'prep\': _prep}\n    except Exception:\n        pass\n\n_u0_load_lgbm()\n\n\ndef _u0_lgbm_scores(obs, select, my_index, pi_opp):\n    """Return per-option LightGBM scores for UNKNOWN_0, or None on failure."""\n    if _U0_LGBM is None:\n        return None\n    try:\n        import numpy as _np\n        my_ps = obs.current.players[my_index]\n        op_ps = obs.current.players[pi_opp]\n        my_active = (my_ps.active or [None])[0]\n        op_active = (op_ps.active or [None])[0]\n        my_bench = [p for p in (my_ps.bench or []) if p is not None]\n        op_bench = [p for p in (op_ps.bench or []) if p is not None]\n        my_all = ([my_active] if my_active else []) + my_bench\n        my_ids = [getattr(p, \'id\', 0) for p in my_all if p is not None]\n        my_hand_n = float(len(my_ps.hand or []))\n        my_deck_n = float(getattr(my_ps, \'deckCount\', len(getattr(my_ps, \'deck\', []) or [])))\n        op_hp = float(getattr(op_active, \'hp\', 0) or 0)\n        powerful_hand = my_hand_n * 20.0\n        sig = _unknown0_policy_abstract_sig(select) if \'USE_ABSTRACT_OPTION_SIGNATURE\' in globals() and USE_ABSTRACT_OPTION_SIGNATURE else \'N0\'\n        rows = []\n        for i, o in enumerate(select.option):\n            ot = getattr(o, \'type\', None)\n            ot_n = (str(getattr(ot, \'name\', \'\') or \'\').upper() or\n                    str(ot).split(\'.\')[-1].upper() or \'UNK\')\n            rows.append({\n                \'option_index\': float(i),\n                \'num_options\': float(len(select.option)),\n                \'min_count\': float(getattr(select, \'minCount\', 1) or 1),\n                \'max_count\': float(getattr(select, \'maxCount\', 1) or 1),\n                \'card_id\': float(getattr(o, \'id\', 0) or 0),\n                \'target_card_id\': float(getattr(o, \'targetId\', 0) or 0),\n                \'attack_id\': float(getattr(o, \'attackId\', 0) or 0),\n                \'number_value\': float(getattr(o, \'number\', 0) or 0),\n                \'in_play_index\': float(getattr(o, \'inPlayIndex\', 0) or 0),\n                \'remain_damage_counter\': float(getattr(o, \'remainDamageCounter\', 0) or 0),\n                \'remain_energy_cost\': float(getattr(o, \'remainEnergyCost\', 0) or 0),\n                \'turn\': float(obs.current.turn),\n                \'turn_action_count\': 0.0,\n                \'my_active_id\': float(getattr(my_active, \'id\', 0) or 0),\n                \'my_active_hp\': float(getattr(my_active, \'hp\', 0) or 0),\n                \'my_active_energy_count\': float(len(getattr(my_active, \'energies\', []) or [])),\n                \'op_active_id\': float(getattr(op_active, \'id\', 0) or 0),\n                \'op_active_hp\': op_hp,\n                \'op_active_energy_count\': float(len(getattr(op_active, \'energies\', []) or [])),\n                \'my_bench_count\': float(len(my_bench)),\n                \'op_bench_count\': float(len(op_bench)),\n                \'my_alakazam_count\': float(sum(1 for _id in my_ids if _id == 743)),\n                \'my_kadabra_count\': float(sum(1 for _id in my_ids if _id == 742)),\n                \'my_abra_count\': float(sum(1 for _id in my_ids if _id == 741)),\n                \'my_dudunsparce_count\': float(sum(1 for _id in my_ids if _id == 66)),\n                \'my_hand_count\': my_hand_n,\n                \'op_hand_count\': float(len(op_ps.hand or [])),\n                \'my_deck_count\': my_deck_n,\n                \'op_deck_count\': float(getattr(op_ps, \'deckCount\', 0)),\n                \'my_prizes_left\': float(len([p for p in (my_ps.prize or []) if p is not None])),\n                \'op_prizes_left\': float(len([p for p in (op_ps.prize or []) if p is not None])),\n                \'stadium_id\': float(getattr(getattr(obs.current, \'stadium\', None), \'id\', 0) or 0),\n                \'powerful_hand_damage_est\': powerful_hand,\n                \'powerful_hand_can_ko_active\': float(powerful_hand >= op_hp),\n                \'deckout_risk_feature\': float(my_deck_n <= 4),\n                \'context_name\': \'UNKNOWN_0\',\n                \'option_type\': ot_n,\n                \'area\': str(getattr(o, \'area\', 0) or 0),\n                \'in_play_area\': str(getattr(o, \'inPlayArea\', 0) or 0),\n                \'option_signature\': sig,\n            })\n        prep = _U0_LGBM[\'prep\']\n        feature_names = prep[\'feature_names\']\n        cat_maps = prep.get(\'cat_maps\', {})\n        cat_set = set(prep.get(\'cat_features\', []))\n        X = _np.zeros((len(rows), len(feature_names)), dtype=\'float64\')\n        for ci, col in enumerate(feature_names):\n            if col in cat_maps:\n                m = cat_maps[col]\n                for ri, r in enumerate(rows):\n                    X[ri, ci] = float(m.get(str(r.get(col, \'UNK\') or \'UNK\'), 0))\n            elif col in cat_set:\n                for ri, r in enumerate(rows):\n                    X[ri, ci] = float(r.get(col, 0) or 0)\n            else:\n                for ri, r in enumerate(rows):\n                    X[ri, ci] = float(r.get(col, 0) or 0)\n        return _U0_LGBM[\'model\'].predict(X).tolist()\n    except Exception:\n        return None\n'

_LGBM_PATCH_TARGET = '    unknown0_policy_selected = _unknown0_policy_select('
_LGBM_PATCH_REPL = (
    '    if _U0_LGBM is not None and _unknown0_policy_is_context(context):\n'
    '        _lgbm_scores = _u0_lgbm_scores(obs, select, my_index, 1 - my_index)\n'
    '        if _lgbm_scores and len(_lgbm_scores) == len(select.option):\n'
    '            _best_i = max(range(len(_lgbm_scores)), key=lambda _i: _lgbm_scores[_i])\n'
    '            _sel = safe_unique_action([_best_i], len(select.option), min_count, max_count)\n'
    '            if len(_sel) >= min_count:\n'
    '                return _sel\n'
    '    unknown0_policy_selected = _unknown0_policy_select('
)

def make_unknown0_lgbm_submission_agent_source(policy_table_source):
    source = policy_table_source
    marker = 'def agent(obs_dict: dict) -> list[int]:'
    if marker not in source:
        return source
    idx = source.find(marker)
    source = source[:idx] + _LGBM_INJ_CODE + '\n\n' + source[idx:]
    if _LGBM_PATCH_TARGET in source:
        source = source.replace(_LGBM_PATCH_TARGET, _LGBM_PATCH_REPL)
    return source

print('make_unknown0_lgbm_submission_agent_source: defined')


Extra numeric features available: ['supporter_played', 'stadium_played', 'energy_attached', 'my_discard_count', 'op_discard_count']
Winner-weighted training: 165685/297473 winner rows (weight=4.0x)


/tmp/ipykernel_11626/2561798104.py:693: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[c] = df[c].fillna(0).astype('float32')
/tmp/ipykernel_11626/2561798104.py:693: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[c] = df[c].fillna(0).astype('float32')
/tmp/ipykernel_11626/2561798104.py:693: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)

  LambdaRank groups: train=31287 decisions, valid=5521 decisions
[50]	train's ndcg@1: 0.57206	train's ndcg@3: 0.594982	valid's ndcg@1: 0.812534	valid's ndcg@3: 0.845919
[100]	train's ndcg@1: 0.58771	train's ndcg@3: 0.605993	valid's ndcg@1: 0.817062	valid's ndcg@3: 0.848299
[150]	train's ndcg@1: 0.597489	train's ndcg@3: 0.612534	valid's ndcg@1: 0.820866	valid's ndcg@3: 0.850528
[200]	train's ndcg@1: 0.604005	train's ndcg@3: 0.617808	valid's ndcg@1: 0.820504	valid's ndcg@3: 0.850629
unknown0 lgbm: best_iter=161
[early_game_filter] kept 23825/43752 rows (step_frac<=2/3)
unknown0 lgbm report: {'status': 'ok', 'model': 'lightgbm_lambdarank_winner_wt4', 'rows': 350000, 'train_rows': 297473, 'valid_rows': 52527, 'train_decisions': 31287, 'valid_decisions': 5521, 'best_iteration': 161, 'num_leaves': 63, 'row_auc': 0.8635473130896986, 'row_logloss': None, 'unknown0_lgbm_decision_metrics': {'decisions': 5521, 'top1': 0.521463502988589, 'top3': 0.7759463865241804, 'avg_chosen_rank': 2.58721246151

In [18]:
# -----------------------------
# Local / confirm / meta evaluation helpers
# -----------------------------

HOP_CONTROL_FALLBACK_DECK = (
    [878] * 4 + [879] * 4 + [311] * 3 + [304] * 2 + [1092] * 1 +
    [1115] * 4 + [1134] * 4 + [1122] * 4 + [1097] * 1 + [1152] * 2 +
    [1197] * 2 + [1171] * 4 + [1219] * 4 + [1227] * 4 + [1182] * 2 +
    [1225] * 3 + [1255] * 4 + [19] * 4 + [11] * 4
)
LUCARIO_EXP14_FALLBACK_DECK = (
    [673] * 2 + [674] * 2 + [675] * 2 + [676] * 3 + [677] * 3 + [678] * 4 +
    [1102] * 4 + [1123] * 2 + [1141] * 4 + [1142] * 4 + [1152] * 4 +
    [1159] * 1 + [1182] * 2 + [1192] * 4 + [1227] * 4 + [1252] * 2 + [6] * 13
)
assert len(HOP_CONTROL_FALLBACK_DECK) == 60
assert len(LUCARIO_EXP14_FALLBACK_DECK) == 60


def find_cg_dir_for_import() -> Path | None:
    candidates = [
        Path('../agent_submission/cg'),
        Path('../../input/raw/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg'),
    ]
    for c in candidates:
        if c.exists() and (c / 'api.py').exists():
            return c
    for pat in ['/kaggle/input/**/sample_submission/cg', '/kaggle/input/**/cg-lib/cg', '/kaggle/input/**/cg']:
        for p in glob.glob(pat, recursive=True):
            pp = Path(p)
            if pp.exists() and (pp / 'api.py').exists():
                return pp
    return None


def ensure_cg_importable() -> dict:
    status = {'available': False, 'cg_dir': None, 'cg_parent': None, 'added_to_sys_path': False, 'error': ''}
    try:
        import cg.api  # noqa: F401
        import cg.game  # noqa: F401
        status['available'] = True
        status['error'] = ''
        return status
    except Exception as first_exc:
        status['error'] = repr(first_exc)
    cg_dir = find_cg_dir_for_import()
    if cg_dir is None:
        status['error'] = status['error'] or 'cg_dir_not_found'
        return status
    cg_parent = str(cg_dir.parent)
    status['cg_dir'] = str(cg_dir)
    status['cg_parent'] = cg_parent
    if cg_parent not in sys.path:
        sys.path.insert(0, cg_parent)
        status['added_to_sys_path'] = True
    try:
        import cg.api  # noqa: F401
        import cg.game  # noqa: F401
        status['available'] = True
        status['error'] = ''
    except Exception as exc:
        status['error'] = repr(exc)
    return status


CG_IMPORT_STATUS = ensure_cg_importable()
print('cg import status:', CG_IMPORT_STATUS)


def cg_available() -> bool:
    global CG_IMPORT_STATUS
    if not CG_IMPORT_STATUS.get('available'):
        CG_IMPORT_STATUS = ensure_cg_importable()
    return bool(CG_IMPORT_STATUS.get('available'))


def make_random_agent(deck: list[int]):
    def random_agent(obs_dict):
        from cg.api import to_observation_class
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            return deck
        max_count = int(obs.select.maxCount)
        min_count = int(obs.select.minCount)
        idxs = list(range(len(obs.select.option)))
        if max_count <= 0:
            return []
        k = max(min_count, min(max_count, len(idxs)))
        if k >= len(idxs):
            return idxs[:k]
        return random.sample(idxs, k)
    return random_agent


def make_first_valid_agent(deck: list[int]):
    def first_valid_agent(obs_dict):
        from cg.api import to_observation_class
        obs = to_observation_class(obs_dict)
        if obs.select is None:
            return deck
        min_count = max(0, int(obs.select.minCount))
        max_count = min(len(obs.select.option), max(0, int(obs.select.maxCount)))
        k = min(max_count, max(min_count, 0))
        return list(range(k))
    return first_valid_agent


def play_game(agent0, deck0, agent1, deck1, max_steps: int = LOCAL_EVAL_MAX_STEPS):
    from cg.api import to_observation_class
    from cg.game import battle_select, battle_start
    try:
        obs, start_data = battle_start(deck0, deck1)
        if obs is None:
            return {'result': None, 'steps': 0, 'error': f'battle_start_failed:{getattr(start_data, "errorPlayer", None)}:{getattr(start_data, "errorType", None)}', 'illegal_action': False}
        for step in range(max_steps + 1):
            obc = to_observation_class(obs)
            if obc.current.result >= 0:
                return {'result': int(obc.current.result), 'steps': step, 'error': '', 'illegal_action': False}
            active_index = int(obc.current.yourIndex)
            action = agent0(obs) if active_index == 0 else agent1(obs)
            if not isinstance(action, list) or any(not isinstance(x, int) for x in action):
                return {'result': None, 'steps': step, 'error': f'illegal_action_type:{action!r}', 'illegal_action': True}
            # If select is not None, validate option-index action shape before asking engine.
            if obc.select is not None:
                min_count = int(obc.select.minCount)
                max_count = int(obc.select.maxCount)
                nopt = len(obc.select.option)
                if len(action) < min_count or len(action) > max_count or len(set(action)) != len(action) or any((x < 0 or x >= nopt) for x in action):
                    return {'result': None, 'steps': step, 'error': f'illegal_action_value:{action!r}:min={min_count}:max={max_count}:nopt={nopt}', 'illegal_action': True}
            obs = battle_select(action)
        return {'result': None, 'steps': max_steps, 'error': 'max_steps_exceeded', 'illegal_action': False}
    except Exception as exc:
        return {'result': None, 'steps': None, 'error': repr(exc), 'illegal_action': False}


def import_agent_from_source(main_source: str, label: str):
    path = WORKING_DIR / f'tmp_v05b_{label}_{int(time.time()*1000)}_{random.randint(0,10**9)}.py'
    write_text(path, main_source)
    return import_module_from_path(path.stem, path)


def summarize_eval_rows(rows: list[dict], suite: str, opponent: str | None = None) -> dict:
    n = len(rows)
    wins = sum(int(r.get('won', False)) for r in rows)
    errors = sum(int(bool(r.get('error'))) for r in rows)
    illegal = sum(int(bool(r.get('illegal_action'))) for r in rows)
    steps = [r.get('steps') for r in rows if isinstance(r.get('steps'), int)]
    return {
        'suite': suite,
        'opponent': opponent,
        'games': int(n),
        'wins': int(wins),
        'win_rate': wins / max(1, n),
        'errors': int(errors),
        'illegal_action_count': int(illegal),
        'avg_steps': float(sum(steps) / len(steps)) if steps else None,
        'rows_preview': rows[:5],
    }


def evaluate_against_agent(module, deck: list[int], opponent_agent, opponent_deck: list[int], n_games: int, suite: str, opponent_name: str) -> tuple[list[dict], dict]:
    rows = []
    for i in range(int(n_games)):
        if i % 2 == 0:
            res = play_game(module.agent, deck, opponent_agent, opponent_deck)
            won = res.get('result') == 0
            seat = 0
        else:
            res = play_game(opponent_agent, opponent_deck, module.agent, deck)
            won = res.get('result') == 1
            seat = 1
        rows.append({'suite': suite, 'opponent': opponent_name, 'game': i, 'seat': seat, 'won': bool(won), **res})
    return rows, summarize_eval_rows(rows, suite=suite, opponent=opponent_name)


def run_local_eval_suite(main_source: str, deck: list[int]) -> dict:
    status = ensure_cg_importable()
    if not status.get('available'):
        return {'status': 'skipped', 'reason': 'cg_not_available', 'cg_import_status': status}
    module = import_agent_from_source(main_source, 'main_agent')
    all_rows = []
    summaries = {}

    # Smoke: random opponent, same deck. This is for crash/illegal-action detection.
    rows, summary = evaluate_against_agent(module, deck, make_random_agent(deck), deck, SMOKE_EVAL_GAMES, 'smoke', 'random_same_deck')
    all_rows.extend(rows)
    summaries['smoke'] = summary

    # Confirm: first-valid and random mixed. Still not a strength test; it is a stability test.
    if RUN_CONFIRM_EVAL:
        confirm_rows = []
        for opp_name, opp_agent in [('random_same_deck', make_random_agent(deck)), ('first_valid_same_deck', make_first_valid_agent(deck))]:
            rows, summary = evaluate_against_agent(module, deck, opp_agent, deck, max(1, CONFIRM_EVAL_GAMES // 2), 'confirm', opp_name)
            confirm_rows.extend(rows)
            summaries[f'confirm_{opp_name}'] = summary
        all_rows.extend(confirm_rows)
        summaries['confirm'] = summarize_eval_rows(confirm_rows, suite='confirm', opponent='mixed_basic')

    # Meta sanity: deck/engine stability against top-meta-style decklists, not faithful top agents.
    if RUN_META_EVAL:
        meta_specs = [
            ('random_hop_control_deck', make_random_agent(list(HOP_CONTROL_FALLBACK_DECK)), list(HOP_CONTROL_FALLBACK_DECK)),
            ('random_alakazam_mirror_deck', make_random_agent(deck), deck),
            ('first_valid_alakazam_mirror_deck', make_first_valid_agent(deck), deck),
            ('random_lucario_exp14_deck', make_random_agent(list(LUCARIO_EXP14_FALLBACK_DECK)), list(LUCARIO_EXP14_FALLBACK_DECK)),
        ]
        meta_rows = []
        for opp_name, opp_agent, opp_deck in meta_specs:
            rows, summary = evaluate_against_agent(module, deck, opp_agent, opp_deck, META_EVAL_GAMES_PER_OPPONENT, 'meta', opp_name)
            meta_rows.extend(rows)
            summaries[f'meta_{opp_name}'] = summary
        all_rows.extend(meta_rows)
        summaries['meta'] = summarize_eval_rows(meta_rows, suite='meta', opponent='mixed_meta_deck_sanity')

    ARTIFACT_PATHS['local_eval_rows_json'] = write_json(OUTPUT_DIR / 'local_eval_rows.json', all_rows)
    eval_rows_df = pd.DataFrame(all_rows)
    ARTIFACT_PATHS['local_eval_rows'] = safe_save_table(eval_rows_df, OUTPUT_DIR / 'local_eval_rows_table')
    return {'status': 'ok', 'cg_import_status': status, 'summaries': summaries, 'total_rows': len(all_rows)}


cg import status: {'available': True, 'cg_dir': '/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg', 'cg_parent': '/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission', 'added_to_sys_path': True, 'error': ''}


In [19]:
# -----------------------------
# Build submission variants with UNKNOWN_0 policy hit/fallback instrumentation
# -----------------------------

def write_deck_csv(deck: list[int], path: str | Path = 'deck.csv'):
    path = Path(path)
    if len(deck) != 60:
        raise ValueError(f'deck must be 60 cards, got {len(deck)}')
    write_text(path, '\n'.join(str(int(x)) for x in deck) + '\n')


def find_cg_dir() -> Path | None:
    candidates = [
        Path('../agent_submission/cg'),
        Path('../../input/raw/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg'),
        Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg'),
    ]
    for c in candidates:
        if c.exists() and (c / 'api.py').exists():
            return c
    for pat in ['/kaggle/input/**/sample_submission/cg', '/kaggle/input/**/cg-lib/cg', '/kaggle/input/**/cg']:
        for p in glob.glob(pat, recursive=True):
            pp = Path(p)
            if pp.exists() and (pp / 'api.py').exists():
                return pp
    return None


def exclude_python_cache(tarinfo: tarfile.TarInfo) -> tarfile.TarInfo | None:
    name = tarinfo.name
    if '__pycache__' in name or name.endswith(('.pyc', '.pyo')):
        return None
    return tarinfo


def build_submission_archive(main_source: str, deck: list[int], archive_path: str | Path = 'submission.tar.gz', extra_files=None) -> dict:
    assert_no_import_time_deck_io(main_source, 'FINAL_MAIN_SOURCE')
    compile(main_source, 'main.py', 'exec')
    # torch allowed in v05d16: UNKNOWN_0 MLP uses torch
    if len(deck) != 60 or not all(isinstance(int(x), int) for x in deck):
        raise ValueError('selected deck must be 60 integer card IDs')
    write_text('main.py', main_source)
    write_deck_csv(deck, 'deck.csv')
    cg_dir = find_cg_dir()
    if cg_dir is None:
        raise FileNotFoundError('Could not find cg directory in Kaggle input. Attach the competition sample submission or dataset containing cg/.')
    archive_path = Path(archive_path)
    if archive_path.exists():
        archive_path.unlink()
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add('main.py', arcname='main.py')
        tar.add('deck.csv', arcname='deck.csv')
        tar.add(str(cg_dir), arcname='cg', filter=exclude_python_cache)
        for _ef_src, _ef_arc in (extra_files or []):
            if Path(str(_ef_src)).exists():
                tar.add(str(_ef_src), arcname=_ef_arc)
    with tarfile.open(archive_path, 'r:gz') as tar:
        names = sorted(tar.getnames())
    required = {'main.py', 'deck.csv', 'cg/api.py', 'cg/game.py'}
    missing = sorted(required - set(names))
    if missing:
        raise RuntimeError(f'submission archive missing: {missing}')
    if any('__pycache__' in n or n.endswith(('.pyc', '.pyo')) for n in names):
        raise RuntimeError('submission archive contains Python cache files')
    return {'archive': str(archive_path), 'files': len(names), 'cg_dir': str(cg_dir), 'required_present': sorted(required), 'preview': names[:25]}


def hard_submission_safety_checks(main_source: str, deck: list[int], submission_summary: dict, eval_summary: dict | None = None):
    if len(deck) != 60:
        raise AssertionError(f'selected deck length must be 60, got {len(deck)}')
    compile(main_source, 'main.py', 'exec')
    assert_no_import_time_deck_io(main_source, 'FINAL_MAIN_SOURCE')
    # torch allowed in v05d16: UNKNOWN_0 MLP uses torch
    if not submission_summary or submission_summary.get('status') == 'error':
        raise AssertionError(f'submission build failed: {submission_summary}')
    required = {'main.py', 'deck.csv', 'cg/api.py', 'cg/game.py'}
    present = set(submission_summary.get('required_present', []))
    if not required.issubset(present):
        raise AssertionError(f'submission archive missing required files: {sorted(required - present)}')
    if eval_summary and eval_summary.get('status') == 'ok':
        smoke = eval_summary.get('summaries', {}).get('smoke', {})
        if int(smoke.get('illegal_action_count', 0) or 0) > 0:
            raise AssertionError(f'smoke eval found illegal actions: {smoke}')
    return True


def extract_unknown0_policy_stats(module) -> dict:
    stats = dict(getattr(module, 'UNKNOWN0_POLICY_STATS', {}) or {})
    selected = dict(getattr(module, 'UNKNOWN0_POLICY_SELECTED_OPTION_TYPE_COUNTS', {}) or {})
    hit_keys = dict(getattr(module, 'UNKNOWN0_POLICY_HIT_KEY_COUNTS', {}) or {})
    eligible = max(1, int(stats.get('eligible', 0) or 0))
    return {
        'stats': stats,
        'selected_option_type_counts': selected,
        'hit_key_counts_top20': sorted(hit_keys.items(), key=lambda kv: kv[1], reverse=True)[:20],
        'hit_rate_over_eligible': float((stats.get('selected', 0) or 0) / eligible),
        'miss_rate_over_eligible': float((stats.get('miss', 0) or 0) / eligible),
    }


def run_variant_smoke_eval(main_source: str, deck: list[int], variant: str, games: int | None = None) -> dict:
    if not RUN_LOCAL_EVAL:
        return {'status': 'skipped', 'reason': 'RUN_LOCAL_EVAL_FALSE', 'variant': variant}
    status = ensure_cg_importable() if 'ensure_cg_importable' in globals() else {'available': False, 'error': 'ensure_cg_importable_not_defined'}
    if not status.get('available'):
        return {'status': 'skipped', 'reason': 'cg_not_available', 'variant': variant, 'cg_import_status': status}
    try:
        module = import_agent_from_source(main_source, f'main_agent_{variant}')
        g = int(games if games is not None else max(4, min(12, SMOKE_EVAL_GAMES)))
        rows, summary = evaluate_against_agent(module, deck, make_random_agent(deck), deck, g, 'variant_smoke', variant)
        summary['variant'] = variant
        summary['unknown0_policy_runtime_stats'] = extract_unknown0_policy_stats(module)
        return {'status': 'ok', 'summary': summary, 'rows_preview': rows[:5]}
    except Exception as exc:
        return {'status': 'error', 'variant': variant, 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}


RULE_ONLY_MAIN_SOURCE = make_alakazam_agent_source(SELECTED_ALAKAZAM_DECK)
compile(RULE_ONLY_MAIN_SOURCE, 'rule_only_main.py', 'exec')
assert_no_import_time_deck_io(RULE_ONLY_MAIN_SOURCE, 'RULE_ONLY_MAIN_SOURCE')

UNKNOWN0_POLICY_MAIN_SOURCE = RULE_ONLY_MAIN_SOURCE
UNKNOWN0_POLICY_SOURCE_STATUS = {'status': 'not_built', 'reason': 'empty_or_disabled'}
if BUILD_UNKNOWN0_POLICY_SUBMISSION and 'make_unknown0_policy_submission_agent_source' in globals():
    try:
        UNKNOWN0_POLICY_MAIN_SOURCE = make_unknown0_policy_submission_agent_source(
            RULE_ONLY_MAIN_SOURCE,
            UNKNOWN0_POLICY_TABLE if 'UNKNOWN0_POLICY_TABLE' in globals() else {},
            enabled=bool(UNKNOWN0_POLICY_TABLE) if 'UNKNOWN0_POLICY_TABLE' in globals() else False,
        )
        UNKNOWN0_POLICY_SOURCE_STATUS = {
            'status': 'ok' if (UNKNOWN0_POLICY_TABLE if 'UNKNOWN0_POLICY_TABLE' in globals() else {}) else 'empty_policy_table_rule_equivalent',
            'policy_table_entries': int(len(UNKNOWN0_POLICY_TABLE)) if 'UNKNOWN0_POLICY_TABLE' in globals() else 0,
            'torch_in_submission': 'import torch' in UNKNOWN0_POLICY_MAIN_SOURCE,
            'runtime_instrumentation': 'UNKNOWN0_POLICY_STATS' in UNKNOWN0_POLICY_MAIN_SOURCE,
        }
    except Exception as exc:
        UNKNOWN0_POLICY_MAIN_SOURCE = RULE_ONLY_MAIN_SOURCE
        UNKNOWN0_POLICY_SOURCE_STATUS = {'status': 'error', 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
        log_error('unknown0_policy_source_build', error=repr(exc), traceback=traceback.format_exc(limit=8))

# v05d16: Build MLP direct submission source
UNKNOWN0_MLP_DIRECT_MAIN_SOURCE = UNKNOWN0_POLICY_MAIN_SOURCE
UNKNOWN0_MLP_DIRECT_STATUS = {'status': 'not_built'}
_u0_lgbm_txt = MODEL_DIR / f'{RUN_PREFIX}-unknown0_lgbm_reranker.txt'
_u0_lgbm_prep = MODEL_DIR / f'{RUN_PREFIX}-unknown0_lgbm_prep.json'
if 'make_unknown0_lgbm_submission_agent_source' in globals() and _u0_lgbm_txt.exists() and _u0_lgbm_prep.exists():
    try:
        UNKNOWN0_MLP_DIRECT_MAIN_SOURCE = make_unknown0_lgbm_submission_agent_source(
            UNKNOWN0_POLICY_MAIN_SOURCE
        )
        UNKNOWN0_MLP_DIRECT_STATUS = {'status': 'ok', 'model': str(_u0_lgbm_txt)}
        print(f'LGBM direct source built, len={len(UNKNOWN0_MLP_DIRECT_MAIN_SOURCE)}')
        assert '_u0_load_lgbm' in UNKNOWN0_MLP_DIRECT_MAIN_SOURCE, 'LGBM code not injected'
        assert '_u0_lgbm_scores' in UNKNOWN0_MLP_DIRECT_MAIN_SOURCE, 'LGBM score function missing'
    except Exception as _exc:
        UNKNOWN0_MLP_DIRECT_STATUS = {'status': 'error', 'error': repr(_exc)}
        print(f'LGBM direct source build failed: {_exc}')
        UNKNOWN0_MLP_DIRECT_MAIN_SOURCE = UNKNOWN0_POLICY_MAIN_SOURCE
else:
    UNKNOWN0_MLP_DIRECT_STATUS = {'status': 'skipped',
        'reason': f'model_missing={not _u0_lgbm_txt.exists()}'}
    print(f'LGBM direct: skipped ({UNKNOWN0_MLP_DIRECT_STATUS["reason"]})')

SUBMISSION_VARIANT_SOURCES = {
    'rule_only': RULE_ONLY_MAIN_SOURCE,
    'unknown0_policy_table': UNKNOWN0_POLICY_MAIN_SOURCE,
    'unknown0_mlp_direct': UNKNOWN0_MLP_DIRECT_MAIN_SOURCE,
}
SUBMISSION_VARIANT_ARCHIVES = {
    'rule_only': f'{RUN_PREFIX}-submission-rule-only.tar.gz',
    'unknown0_policy_table': f'{RUN_PREFIX}-submission-unknown0-policy-table.tar.gz',
    'unknown0_mlp_direct': f'{RUN_PREFIX}-submission-unknown0-mlp-direct.tar.gz',
}
SUBMISSION_VARIANT_EXTRA_FILES = {
    'rule_only': [],
    'unknown0_policy_table': [],
    'unknown0_mlp_direct': ([(str(_u0_lgbm_txt), 'unknown0_lgbm.txt'),
                              (str(_u0_lgbm_prep), 'unknown0_lgbm_prep.json')]
                             if _u0_lgbm_txt.exists() and _u0_lgbm_prep.exists() else []),
}
FINAL_SUBMISSION_ARCHIVE = f'{RUN_PREFIX}-submission.tar.gz'

SUBMISSION_VARIANT_SUMMARIES = {}
if BUILD_SUBMISSION:
    for variant, source in SUBMISSION_VARIANT_SOURCES.items():
        if variant == 'unknown0_policy_table' and not BUILD_UNKNOWN0_POLICY_SUBMISSION:
            SUBMISSION_VARIANT_SUMMARIES[variant] = {'status': 'skipped', 'reason': 'BUILD_UNKNOWN0_POLICY_SUBMISSION_FALSE'}
            continue
        try:
            _ef = (SUBMISSION_VARIANT_EXTRA_FILES.get(variant, [])
                   if 'SUBMISSION_VARIANT_EXTRA_FILES' in globals() else [])
            summary = build_submission_archive(source, SELECTED_ALAKAZAM_DECK,
                                               SUBMISSION_VARIANT_ARCHIVES[variant],
                                               extra_files=_ef)
            summary['status'] = 'ok'
            summary['variant'] = variant
            summary['torch_in_submission'] = 'import torch' in source
            summary['runtime_instrumentation'] = 'UNKNOWN0_POLICY_STATS' in source
            SUBMISSION_VARIANT_SUMMARIES[variant] = summary
            ARTIFACT_PATHS[f'submission_{variant}'] = str(SUBMISSION_VARIANT_ARCHIVES[variant])
        except Exception as exc:
            SUBMISSION_VARIANT_SUMMARIES[variant] = {'status': 'error', 'variant': variant, 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
            log_error(f'submission_build_{variant}', error=repr(exc), traceback=traceback.format_exc(limit=8))

SUBMISSION_VARIANT_EVAL_SUMMARIES = {}
if RUN_LOCAL_EVAL:
    for variant, source in SUBMISSION_VARIANT_SOURCES.items():
        if SUBMISSION_VARIANT_SUMMARIES.get(variant, {}).get('status') == 'ok':
            SUBMISSION_VARIANT_EVAL_SUMMARIES[variant] = run_variant_smoke_eval(source, SELECTED_ALAKAZAM_DECK, variant)

FINAL_SUBMISSION_VARIANT_REQUESTED = FINAL_SUBMISSION_VARIANT if FINAL_SUBMISSION_VARIANT in SUBMISSION_VARIANT_SOURCES else 'rule_only'
if FINAL_SUBMISSION_VARIANT_REQUESTED != FINAL_SUBMISSION_VARIANT:
    log_error('final_submission_variant_unknown', requested=FINAL_SUBMISSION_VARIANT, fallback='rule_only')
if SUBMISSION_VARIANT_SUMMARIES.get(FINAL_SUBMISSION_VARIANT_REQUESTED, {}).get('status') != 'ok':
    log_error('final_submission_variant_unavailable', requested=FINAL_SUBMISSION_VARIANT_REQUESTED, fallback='rule_only')
    FINAL_SUBMISSION_VARIANT_EFFECTIVE = 'rule_only'
else:
    FINAL_SUBMISSION_VARIANT_EFFECTIVE = FINAL_SUBMISSION_VARIANT_REQUESTED

FINAL_MAIN_SOURCE = SUBMISSION_VARIANT_SOURCES[FINAL_SUBMISSION_VARIANT_EFFECTIVE]
compile(FINAL_MAIN_SOURCE, 'main.py', 'exec')
assert_no_import_time_deck_io(FINAL_MAIN_SOURCE, 'FINAL_MAIN_SOURCE')
# torch check removed in v05d16: UNKNOWN_0 MLP uses torch

LOCAL_EVAL_SUMMARY = {}
if RUN_LOCAL_EVAL:
    try:
        LOCAL_EVAL_SUMMARY = run_local_eval_suite(FINAL_MAIN_SOURCE, SELECTED_ALAKAZAM_DECK)
        LOCAL_EVAL_SUMMARY['final_variant'] = FINAL_SUBMISSION_VARIANT_EFFECTIVE
        # Full suite imports a fresh module internally. Keep explicit variant smoke stats for policy hit diagnostics.
        LOCAL_EVAL_SUMMARY['selected_variant_smoke_eval_with_policy_stats'] = SUBMISSION_VARIANT_EVAL_SUMMARIES.get(FINAL_SUBMISSION_VARIANT_EFFECTIVE, {})
    except Exception as exc:
        LOCAL_EVAL_SUMMARY = {'status': 'error', 'final_variant': FINAL_SUBMISSION_VARIANT_EFFECTIVE, 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
        log_error('local_eval_suite', error=repr(exc), traceback=traceback.format_exc(limit=8))
print('local eval summary:', json.dumps(LOCAL_EVAL_SUMMARY, ensure_ascii=False, indent=2)[:5000])

SUBMISSION_SUMMARY = {}
if BUILD_SUBMISSION:
    try:
        SUBMISSION_SUMMARY = build_submission_archive(FINAL_MAIN_SOURCE, SELECTED_ALAKAZAM_DECK, FINAL_SUBMISSION_ARCHIVE)
        SUBMISSION_SUMMARY['status'] = 'ok'
        SUBMISSION_SUMMARY['variant'] = FINAL_SUBMISSION_VARIANT_EFFECTIVE
        SUBMISSION_SUMMARY['runtime_instrumentation'] = 'UNKNOWN0_POLICY_STATS' in FINAL_MAIN_SOURCE
        ARTIFACT_PATHS['submission_final'] = FINAL_SUBMISSION_ARCHIVE
    except Exception as exc:
        SUBMISSION_SUMMARY = {'status': 'error', 'variant': FINAL_SUBMISSION_VARIANT_EFFECTIVE, 'error': repr(exc), 'traceback': traceback.format_exc(limit=8)}
        log_error('submission_build_final', error=repr(exc), traceback=traceback.format_exc(limit=8))
print('submission variant summaries:', json.dumps(SUBMISSION_VARIANT_SUMMARIES, ensure_ascii=False, indent=2)[:4000])
print('submission variant eval summaries:', json.dumps(SUBMISSION_VARIANT_EVAL_SUMMARIES, ensure_ascii=False, indent=2)[:4000])
print('selected final submission summary:', SUBMISSION_SUMMARY)

if BUILD_SUBMISSION:
    hard_submission_safety_checks(FINAL_MAIN_SOURCE, SELECTED_ALAKAZAM_DECK, SUBMISSION_SUMMARY, LOCAL_EVAL_SUMMARY)


LGBM direct source built, len=48952
local eval summary: {
  "status": "ok",
  "cg_import_status": {
    "available": true,
    "cg_dir": null,
    "cg_parent": null,
    "added_to_sys_path": false,
    "error": ""
  },
  "summaries": {
    "smoke": {
      "suite": "smoke",
      "opponent": "random_same_deck",
      "games": 16,
      "wins": 15,
      "win_rate": 0.9375,
      "errors": 0,
      "illegal_action_count": 0,
      "avg_steps": 85.9375,
      "rows_preview": [
        {
          "suite": "smoke",
          "opponent": "random_same_deck",
          "game": 0,
          "seat": 0,
          "won": true,
          "result": 0,
          "steps": 51,
          "error": "",
          "illegal_action": false
        },
        {
          "suite": "smoke",
          "opponent": "random_same_deck",
          "game": 1,
          "seat": 1,
          "won": false,
          "result": 0,
          "steps": 15,
          "error": "",
          "illegal_action": false
        },
 

In [20]:
# -----------------------------
# Final summary
# -----------------------------


# --- v0-05d9: phase-stratified winner_margin (early/mid/late game) ---
MC_PHASE_WINNER_MARGIN = {'status': 'not_run'}
try:
    _eval_df = UNKNOWN0_POLICY_DECISION_EVAL_DF.copy() if 'UNKNOWN0_POLICY_DECISION_EVAL_DF' in globals() and not UNKNOWN0_POLICY_DECISION_EVAL_DF.empty else pd.DataFrame()
    _dr_mc = DECISION_ROWS_DF[['decision_id','mc_step_weight','step']].drop_duplicates('decision_id') if ('DECISION_ROWS_DF' in globals() and 'mc_step_weight' in DECISION_ROWS_DF.columns) else pd.DataFrame()
    _ep_steps = EPISODE_SPLIT_DF[['episode_id','n_steps']].drop_duplicates('episode_id') if ('EPISODE_SPLIT_DF' in globals() and 'n_steps' in EPISODE_SPLIT_DF.columns) else pd.DataFrame()

    if not _eval_df.empty and not _dr_mc.empty:
        _eval_df = _eval_df.merge(_dr_mc, on='decision_id', how='left')
        _eval_df['episode_id'] = _eval_df['decision_id'].str.split(':').str[0]
        if not _ep_steps.empty:
            _eval_df = _eval_df.merge(_ep_steps, on='episode_id', how='left')
            _eval_df['game_phase'] = 'early'
            _eval_df.loc[_eval_df['step'] > _eval_df['n_steps'] / 3, 'game_phase'] = 'mid'
            _eval_df.loc[_eval_df['step'] > _eval_df['n_steps'] * 2 / 3, 'game_phase'] = 'late'
        else:
            _eval_df['game_phase'] = 'unknown'

        _ho_all = _eval_df[(_eval_df['split'].astype(str) == 'holdout') & (_eval_df['policy_hit'] == True)]
        _phases = {}
        for _phase in ['early', 'mid', 'late', 'all']:
            _ph = _ho_all if _phase == 'all' else _ho_all[_ho_all.get('game_phase', pd.Series(dtype=str)) == _phase]
            if _ph.empty:
                continue
            _w = _ph[_ph['won'] == 1.0]; _l = _ph[_ph['won'] == 0.0]
            _wn, _ln = len(_w), len(_l)
            _wt1 = float(_w['policy_correct'].mean()) if _wn > 0 else None
            _lt1 = float(_l['policy_correct'].mean()) if _ln > 0 else None
            _margin = round(_wt1 - _lt1, 6) if (_wt1 and _lt1) else None
            _phases[_phase] = {'n_winner': _wn, 'n_loser': _ln, 'winner_top1': _wt1, 'loser_top1': _lt1, 'winner_margin': _margin, 'low_confidence': _wn < 100 or _ln < 100}

        MC_PHASE_WINNER_MARGIN = {'status': 'ok', 'gamma': MC_DISCOUNT_GAMMA, 'use_mc_weights': USE_MC_RETURN_WEIGHTS, 'baseline_v05d7': 0.081, 'phases': _phases}
        print('\n=== phase-stratified winner_margin (UNKNOWN_0 holdout hits) ===')
        print(f'  gamma={MC_DISCOUNT_GAMMA}, use_mc_weights={USE_MC_RETURN_WEIGHTS}')
        for _ph, _r in _phases.items():
            _flag = ' [low_conf]' if _r.get('low_confidence') else ''
            _wm = f"{_r['winner_margin']:+.4f}" if _r.get('winner_margin') is not None else '  N/A '
            print(f"  {_ph:8s}  margin={_wm}  n=({_r['n_winner']},{_r['n_loser']}){_flag}")
        print(f'  baseline (v05d7, no MC): +0.0810')
    else:
        MC_PHASE_WINNER_MARGIN = {'status': 'skipped', 'reason': 'missing data'}
except Exception as _mc_exc:
    MC_PHASE_WINNER_MARGIN = {'status': 'error', 'error': repr(_mc_exc)}
    import traceback; traceback.print_exc()

# --- v0-05d7: winner_margin for UNKNOWN_0 policy hits (holdout) ---
UNKNOWN0_WINNER_MARGIN_REPORT = {'status': 'not_run'}
try:
    _eval_df = UNKNOWN0_POLICY_DECISION_EVAL_DF if 'UNKNOWN0_POLICY_DECISION_EVAL_DF' in globals() else pd.DataFrame()
    if not _eval_df.empty and 'won' in _eval_df.columns and 'policy_hit' in _eval_df.columns:
        _ho = _eval_df[(_eval_df['split'].astype(str) == 'holdout') & (_eval_df['policy_hit'] == True)].copy()
        _won = _ho[_ho['won'] == 1.0]
        _lost = _ho[_ho['won'] == 0.0]
        _w_n = int(len(_won))
        _l_n = int(len(_lost))
        _low_conf = _w_n < 200 or _l_n < 200
        _w_top1 = float(_won['policy_correct'].mean()) if _w_n > 0 else None
        _l_top1 = float(_lost['policy_correct'].mean()) if _l_n > 0 else None
        _margin = (_w_top1 - _l_top1) if (_w_top1 is not None and _l_top1 is not None) else None
        UNKNOWN0_WINNER_MARGIN_REPORT = {
            'status': 'ok',
            'scope': 'UNKNOWN_0 holdout policy_hit=True',
            'winner_decisions': _w_n,
            'loser_decisions': _l_n,
            'winner_top1': _w_top1,
            'loser_top1': _l_top1,
            'winner_margin': _margin,
            'low_confidence': _low_conf,
        }
        print(f"winner_margin (UNKNOWN_0 holdout hits): {_margin:.4f}" if _margin is not None else "winner_margin: N/A")
        print(f"  winner_top1={_w_top1}, loser_top1={_l_top1}, n_winner={_w_n}, n_loser={_l_n}")
        if _low_conf:
            print("  WARNING: low_confidence (n < 200 in winner or loser slice)")
    else:
        UNKNOWN0_WINNER_MARGIN_REPORT = {'status': 'skipped', 'reason': 'missing cols or empty df'}
except Exception as _wm_exc:
    UNKNOWN0_WINNER_MARGIN_REPORT = {'status': 'error', 'error': repr(_wm_exc)}
    print(f"winner_margin computation error: {_wm_exc}")

V05_RUN_SUMMARY = {
    'experiment': EXPERIMENT_NAME,
    'run_mode': RUN_MODE,
    'train_mode': TRAIN_MODE,
    'submission_policy_mode': SUBMISSION_POLICY_MODE,
    'use_neural_for_submission': USE_NEURAL_FOR_SUBMISSION,
    'use_mlp_reranker_for_submission': USE_MLP_RERANKER_FOR_SUBMISSION,
    'mlp_submission_contexts': list(MLP_SUBMISSION_CONTEXTS),
    'mlp_alpha': float(MLP_ALPHA),
    'auto_select_submission_deck': AUTO_SELECT_SUBMISSION_DECK,
    'submission_deck_hash_override': SUBMISSION_DECK_HASH_OVERRIDE,
    'selected_deck_source': SELECTED_ALAKAZAM_DECK_SOURCE,
    'selected_deck_hash': SELECTED_ALAKAZAM_DECK_HASH,
    'candidate_deck_rows': int(len(ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF)) if 'ALAKAZAM_CANDIDATE_DECK_COMPARISON_DF' in globals() else 0,
    'candidate_deck_review_rows': int(len(CANDIDATE_DECK_REVIEW_DF)) if 'CANDIDATE_DECK_REVIEW_DF' in globals() else 0,
    'unknown_context_rows': int(len(UNKNOWN_CONTEXT_REPORT_DF)) if 'UNKNOWN_CONTEXT_REPORT_DF' in globals() else 0,
    'unknown_context_deep_dive_rows': int(len(UNKNOWN_CONTEXT_DEEP_DIVE_DF)) if 'UNKNOWN_CONTEXT_DEEP_DIVE_DF' in globals() else 0,
    'top200_rows': int(len(TOP200_DF)),
    'episode_files': int(len(EPISODE_FILES)),
    'non_episode_json_skipped': int(len(NON_EPISODE_JSON_ROWS)) if 'NON_EPISODE_JSON_ROWS' in globals() else 0,
    'episode_index_rows': int(len(EPISODE_INDEX_DF)),
    'decklist_rows': int(len(DECKLISTS_DF)),
    'decision_rows': int(len(DECISION_ROWS_DF)),
    'option_rows': int(len(OPTION_ROWS_DF)),
    'episode_split_rows': int(len(EPISODE_SPLIT_DF)) if 'EPISODE_SPLIT_DF' in globals() else 0,
    'split_summary_rows': int(len(SPLIT_SUMMARY_DF)) if 'SPLIT_SUMMARY_DF' in globals() else 0,
    'stress_slice_summary_rows': int(len(STRESS_SLICE_SUMMARY_DF)) if 'STRESS_SLICE_SUMMARY_DF' in globals() else 0,
    'episode_split_counts': EPISODE_SPLIT_DF['split'].value_counts().sort_index().to_dict() if 'EPISODE_SPLIT_DF' in globals() and not EPISODE_SPLIT_DF.empty and 'split' in EPISODE_SPLIT_DF.columns else {},
    'alakazam_option_model_rows': int(len(ALAKAZAM_OPTION_MODEL_DF)),
    'bc_contexts': sorted(ALAKAZAM_BC_MODELS.keys()),
    'weighted_bc_contexts': sorted(ALAKAZAM_WEIGHTED_BC_MODELS.keys()),
    'value_report': ALAKAZAM_VALUE_REPORT,
    'mlp_reranker_report': MLP_RERANKER_REPORT if 'MLP_RERANKER_REPORT' in globals() else {'status': 'not_defined'},
    'mlp_context_report_rows': int(len(MLP_CONTEXT_REPORT_DF)) if 'MLP_CONTEXT_REPORT_DF' in globals() else 0,
    'mlp_disagreement_examples_rows': int(len(MLP_DISAGREEMENT_EXAMPLES_DF)) if 'MLP_DISAGREEMENT_EXAMPLES_DF' in globals() else 0,
    'unknown0_action_report_rows': int(len(UNKNOWN0_ACTION_REPORT_DF)) if 'UNKNOWN0_ACTION_REPORT_DF' in globals() else 0,
    'mlp_submission_safety': MLP_SUBMISSION_SAFETY if 'MLP_SUBMISSION_SAFETY' in globals() else {},
    'unknown0_mlp_report': UNKNOWN0_MLP_REPORT if 'UNKNOWN0_MLP_REPORT' in globals() else {'status': 'not_defined'},
    'unknown0_mlp_signature_report_rows': int(len(UNKNOWN0_MLP_SIGNATURE_REPORT_DF)) if 'UNKNOWN0_MLP_SIGNATURE_REPORT_DF' in globals() else 0,
    'unknown0_mlp_calibration_rows': int(len(UNKNOWN0_MLP_CALIBRATION_DF)) if 'UNKNOWN0_MLP_CALIBRATION_DF' in globals() else 0,
    'unknown0_rule_proxy_assist_report_rows': int(len(UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF)) if 'UNKNOWN0_RULE_PROXY_ASSIST_REPORT_DF' in globals() else 0,
    'unknown0_policy_decision_eval_rows': int(len(UNKNOWN0_POLICY_DECISION_EVAL_DF)) if 'UNKNOWN0_POLICY_DECISION_EVAL_DF' in globals() else 0,
    'unknown0_policy_split_eval_rows': int(len(UNKNOWN0_POLICY_SPLIT_EVAL_DF)) if 'UNKNOWN0_POLICY_SPLIT_EVAL_DF' in globals() else 0,
    'unknown0_policy_stress_eval_rows': int(len(UNKNOWN0_POLICY_STRESS_EVAL_DF)) if 'UNKNOWN0_POLICY_STRESS_EVAL_DF' in globals() else 0,
    'unknown0_policy_holdout_eval': UNKNOWN0_POLICY_SPLIT_EVAL_DF[UNKNOWN0_POLICY_SPLIT_EVAL_DF['split'].astype(str) == 'holdout'].to_dict('records')[0] if 'UNKNOWN0_POLICY_SPLIT_EVAL_DF' in globals() and not UNKNOWN0_POLICY_SPLIT_EVAL_DF.empty and (UNKNOWN0_POLICY_SPLIT_EVAL_DF['split'].astype(str) == 'holdout').any() else {},
    'unknown0_policy_summary': UNKNOWN0_POLICY_SUMMARY if 'UNKNOWN0_POLICY_SUMMARY' in globals() else {},
    'unknown0_policy_fit_split_summary': UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY if 'UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY' in globals() else {},
    'unknown0_policy_fit_signature_report_rows': int(len(UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF)) if 'UNKNOWN0_POLICY_FIT_SIGNATURE_REPORT_DF' in globals() else 0,
    'unknown0_policy_table_entries': int(len(UNKNOWN0_POLICY_TABLE)) if 'UNKNOWN0_POLICY_TABLE' in globals() else 0,
    'unknown0_policy_min_table_acc': float(UNKNOWN0_POLICY_MIN_TABLE_ACC),
    'unknown0_policy_source_status': UNKNOWN0_POLICY_SOURCE_STATUS if 'UNKNOWN0_POLICY_SOURCE_STATUS' in globals() else {},
    'unknown0_direct_mlp_packaging_status': UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS if 'UNKNOWN0_DIRECT_MLP_PACKAGING_STATUS' in globals() else {},
    'final_submission_variant_requested': FINAL_SUBMISSION_VARIANT if 'FINAL_SUBMISSION_VARIANT' in globals() else 'unknown0_policy_table',
    'final_submission_variant_effective': FINAL_SUBMISSION_VARIANT_EFFECTIVE if 'FINAL_SUBMISSION_VARIANT_EFFECTIVE' in globals() else 'rule_only',
    'submission_variant_summaries': SUBMISSION_VARIANT_SUMMARIES if 'SUBMISSION_VARIANT_SUMMARIES' in globals() else {},
    'submission_variant_eval_summaries': SUBMISSION_VARIANT_EVAL_SUMMARIES if 'SUBMISSION_VARIANT_EVAL_SUMMARIES' in globals() else {},
    'eval_games': {
        'smoke': SMOKE_EVAL_GAMES,
        'confirm': CONFIRM_EVAL_GAMES,
        'meta_per_opponent': META_EVAL_GAMES_PER_OPPONENT,
    },
    'cg_import_status': CG_IMPORT_STATUS if 'CG_IMPORT_STATUS' in globals() else {},
    'local_eval_summary': LOCAL_EVAL_SUMMARY,
    'submission_summary': SUBMISSION_SUMMARY,
    'error_count': len(V05_ERROR_ROWS),
    'artifacts': ARTIFACT_PATHS,
    'next_review_points': [
        'Review candidate_deck_review before setting SUBMISSION_DECK_HASH_OVERRIDE.',
        'Inspect unknown_context_deep_dive before assigning manual context names.',
        'Treat meta eval opponents as stability tests unless real top-agent sources are added.',
        'Check unknown0_policy_runtime_stats: selected/hit rate should be > 0 before trusting the policy-table submission.',
        'If unknown0_policy_table has illegal actions or zero hit rate, fall back to rule_only.',
        'Keep torch out of main.py; UNKNOWN_0 assist uses a distilled policy table with rule fallback.',
    ],
}
V05_RUN_SUMMARY['unknown0_winner_margin_report'] = UNKNOWN0_WINNER_MARGIN_REPORT
V05_RUN_SUMMARY['mc_phase_winner_margin'] = MC_PHASE_WINNER_MARGIN
write_json(OUTPUT_DIR / 'mc_phase_winner_margin.json', MC_PHASE_WINNER_MARGIN)
write_json(OUTPUT_DIR / 'v05_run_summary.json', V05_RUN_SUMMARY)
write_json(OUTPUT_DIR / 'v05_errors.json', V05_ERROR_ROWS)
print(json.dumps(V05_RUN_SUMMARY, ensure_ascii=False, indent=2)[:8000])
if V05_ERROR_ROWS:
    print('v05 error preview:')
    display(pd.DataFrame(V05_ERROR_ROWS).head(50)) if 'display' in globals() else print(pd.DataFrame(V05_ERROR_ROWS).head(50))
else:
    print('no v05 errors')

assert len(SELECTED_ALAKAZAM_DECK) == 60
assert_no_import_time_deck_io(FINAL_MAIN_SOURCE, 'FINAL_MAIN_SOURCE')
print('v0-05d1 UNKNOWN_0 policy hit-check notebook finished')


# --- v0-05d10: live hit_rate vs holdout hit_rate comparison ---
LIVE_HITRATE_REPORT = {'status': 'not_run'}
try:
    _sves = SUBMISSION_VARIANT_EVAL_SUMMARIES if 'SUBMISSION_VARIANT_EVAL_SUMMARIES' in globals() else {}
    _pt_eval = _sves.get('unknown0_policy_table', {})
    _stats = _pt_eval.get('summary', {}).get('unknown0_policy_runtime_stats', {})
    _live_hr = _stats.get('hit_rate_over_eligible', None)
    _holdout_hr = None
    if 'UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY' in globals() and isinstance(UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY, dict):
        _ho_eval = UNKNOWN0_POLICY_FIT_SPLIT_SUMMARY.get('by_split', {})
        _ho = _ho_eval.get('holdout', {})
        _holdout_hr = _ho.get('hit_rate', None)
    LIVE_HITRATE_REPORT = {
        'status': 'ok',
        'live_hit_rate': float(_live_hr) if _live_hr is not None else None,
        'holdout_hit_rate': float(_holdout_hr) if _holdout_hr is not None else None,
        'use_abstract_sig': bool('USE_ABSTRACT_OPTION_SIGNATURE' in globals() and USE_ABSTRACT_OPTION_SIGNATURE),
        'note': 'live=12 games vs rule_only smoke; holdout=top-200 episode split',
    }
    print('live_hit_rate:', LIVE_HITRATE_REPORT.get('live_hit_rate'))
    print('holdout_hit_rate:', LIVE_HITRATE_REPORT.get('holdout_hit_rate'))
except Exception as _e:
    LIVE_HITRATE_REPORT = {'status': 'error', 'error': str(_e)}

write_json(OUTPUT_DIR / 'live_hitrate_report.json', LIVE_HITRATE_REPORT)




=== phase-stratified winner_margin (UNKNOWN_0 holdout hits) ===
  gamma=0.98, use_mc_weights=True
  baseline (v05d7, no MC): +0.0810
winner_margin: N/A
  winner_top1=None, loser_top1=None, n_winner=0, n_loser=0
{
  "experiment": "v0_05d25_lgbm_winner_wt4",
  "run_mode": "full",
  "train_mode": true,
  "submission_policy_mode": "alakazam_rule_base",
  "use_neural_for_submission": false,
  "use_mlp_reranker_for_submission": false,
  "mlp_submission_contexts": [],
  "mlp_alpha": 1.0,
  "auto_select_submission_deck": false,
  "submission_deck_hash_override": "",
  "selected_deck_source": "default_kadoraba_fallback",
  "selected_deck_hash": "e702674c1864",
  "candidate_deck_rows": 13,
  "candidate_deck_review_rows": 13,
  "unknown_context_rows": 12,
  "unknown_context_deep_dive_rows": 41,
  "top200_rows": 200,
  "episode_files": 5516,
  "non_episode_json_skipped": 0,
  "episode_index_rows": 5516,
  "decklist_rows": 11032,
  "decision_rows": 513148,
  "option_rows": 8395122,
  "episode_spli

'/kaggle/working/pokemon-20260625-v0-05d25/pokemon-20260625-v0-05d25-live_hitrate_report.json'